In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import matplotlib.patches as mpatches
from matplotlib.cm import get_cmap
import matplotlib
import pybedtools
import os
from pybedtools import BedTool
from pG4utils.utils import parse_fasta
from tqdm import tqdm 
from pG4utils.utils import parse_fasta
from pG4utils.vcftools import TrinucleotideModel
import sys
import pyranges as pr

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
dataset_path = Path("/scratch/10904/nikolchanchan/data")

# # pG4s # 
G4HUNTER = dataset_path / "pG4s_extractions" / "g4hunter" / "chm13v2_g4hunter.txt.gz"
REGEX = dataset_path / "pG4s_extractions" / "quadparser" / "chm13v2_regex_motifs.txt"

EG4 = Path("/work/10904/nikolchanchan/vista/g4_revisions/g4_t2t_revisions/notebooks") / "eG4.txt"
EG4_df = pd.read_csv(EG4, sep="\t").drop_duplicates(subset=["seqID", "start", "end"]).reset_index(drop=True)
EG4_df

In [ ]:
datasets = [
            ("eG4", EG4_df), 
            ("G4Hunter", pd.read_table(G4HUNTER)), 
            ("Quadparser", pd.read_table(REGEX))
            ]

In [ ]:
FIELDS = ["#CHROM", "start", "end", "mutation", "mut_type", "REF", "ALT", "AC_samples", "AN_samples"]
mut_df = pd.read_table("/work/10904/nikolchanchan/vista/g4_t2t_revisions/pG4utils/processed_VCF/hprc_processed_annotated_all.tsv.gz",
                       usecols=FIELDS)
mut_df = mut_df[(mut_df["AC_samples"] > 0)].reset_index(drop=True)
mut_df

In [ ]:
mutations = mut_df["mut_type"].unique()
mut_df_collection = {mut: mut_df[mut_df["mut_type"] == mut].drop_duplicates(subset=["#CHROM", "start"]).reset_index(drop=True) for mut in mutations}
mut_df_collection

In [ ]:
from tqdm import tqdm 
from pG4utils.utils import parse_fasta

FASTA = dataset_path / "fasta" / "hs1.fa.gz"
FASTA = dataset_path / "hs1.fa.gz"
assert FASTA.is_file()
# # # #
seq_sizes = pd.read_table(dataset_path / "genome.txt", 
                          header=None)
seq_sizes = dict(zip(seq_sizes[0], seq_sizes[1]))
chrom_sequences = dict()
for seqID, seq in tqdm(parse_fasta(FASTA), total=24):
    if seqID == "chrY":
        continue
    chrom_sequences[seqID] = seq.upper()

In [ ]:
import numpy as np

WINDOW_SIZE = 20
mut_expanded = dict()
mutations = ["del", "ins", "smalldel", "smallins", "snp", "mnp"]

for mutation in tqdm(mutations):
    w = WINDOW_SIZE
    if mutation == "del":
        for pos, mut_name in zip(["start", "end"],
                                  ["del_breakpoint_5prime", "del_breakpoint_3prime"]):
            mut_expanded[mut_name] = mut_df_collection[mutation].copy().rename(columns={"seqID": "#CHROM"})
            if pos == "end":
                mut_expanded[mut_name][pos] = mut_expanded[mut_name][pos] - 1
            mut_expanded[mut_name].loc[:, "chrom_size"]     = mut_expanded[mut_name]["#CHROM"].apply(lambda x: seq_sizes[x])
            mut_expanded[mut_name].loc[:, "expanded_start"] = np.maximum(mut_expanded[mut_name][pos] - w, 0)
            mut_expanded[mut_name].loc[:, "expanded_end"]   = np.minimum(mut_expanded[mut_name][pos] + w + 1, mut_expanded[mut_name]["chrom_size"])
            mut_expanded[mut_name].loc[:, "mut_sequence"]   = mut_expanded[mut_name].apply(
                    lambda row: chrom_sequences[row["#CHROM"]][row["expanded_start"]:row["expanded_end"]], axis=1)
            mut_expanded[mut_name].loc[:, "GC_content"] = mut_expanded[mut_name]["mut_sequence"].str.count("[GC]")
            mut_expanded[mut_name].loc[:, "length"]     = mut_expanded[mut_name]["mut_sequence"].apply(len)
            mut_expanded[mut_name].drop(columns=["mut_sequence"], inplace=True)
        print("DONE! (del breakpoints)")
    else:
        mut_expanded[mutation] = mut_df_collection[mutation].copy().rename(columns={"seqID": "#CHROM"})
        anchor = ((mut_expanded[mutation]["start"] + mut_expanded[mutation]["end"]) // 2
                  if (mutation == "smalldel" or mutation == "mnp")
                  else mut_expanded[mutation]["start"])
        mut_expanded[mutation].loc[:, "chrom_size"]     = mut_expanded[mutation]["#CHROM"].apply(lambda x: seq_sizes[x])
        mut_expanded[mutation].loc[:, "expanded_start"] = np.maximum(anchor - w, 0)
        mut_expanded[mutation].loc[:, "expanded_end"]   = np.minimum(anchor + w + 1, mut_expanded[mutation]["chrom_size"])
        mut_expanded[mutation].loc[:, "mut_sequence"]   = mut_expanded[mutation].apply(
                    lambda row: chrom_sequences[row["#CHROM"]][row["expanded_start"]:row["expanded_end"]], axis=1)
        mut_expanded[mutation].loc[:, "GC_content"] = mut_expanded[mutation]["mut_sequence"].str.count("[GC]")
        mut_expanded[mutation].loc[:, "length"]     = mut_expanded[mutation]["mut_sequence"].apply(len)
        mut_expanded[mutation].drop(columns=["mut_sequence"], inplace=True)
        print(f"DONE! ({mutation})")

In [ ]:
gw_densities = dict()
genome_size = sum(seq_sizes.values())

for ds_name, df in datasets:
    g4_pr = pr.from_dict({
        "Chromosome": df["seqID"],
        "Start":      df["start"],
        "End":        df["end"],
    })
    g4_merged = g4_pr.merge()

    total_g4_bases = (g4_merged.End - g4_merged.Start).sum()
    genome_size    = sum(seq_sizes.values())

    gw_density = total_g4_bases * 1e6 / genome_size
    print(f"Genome-wide G4 density: {gw_density:.4f} bp per Mb")

    gw_densities[ds_name] = total_g4_bases * 1e6 / genome_size
gw_densities

In [ ]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import joblib 

def pval_to_stars(p):
    if p < 0.0001: return "****"
    elif p < 0.001:  return "***"
    elif p < 0.01:  return "**"
    elif p < 0.05:  return "*"
    else:  return ""

def load_gc_models(models_dir):
    models_dir = Path(models_dir)
    data = {}
    for pkl in models_dir.glob("*_model.pkl"):
        group = pkl.stem.replace("_model", "")
        resid_path = models_dir / f"{group}_residuals.npy"
        data[group] = {
            "model":     joblib.load(pkl),
            "residuals": np.load(resid_path) if resid_path.exists() else None,
        }
    return data
    
models_dir = Path("/work/10904/nikolchanchan/vista/figures_g4_t2t_NEW/PRMD9_fig/figure/gc_models")
models  = load_gc_models(models_dir)
mutations = ["smallins", "smalldel", "ins", "del_breakpoint_5prime", "del_breakpoint_3prime", "snp", "mnp"]


In [ ]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import joblib 

def pval_to_stars(p):
    if p < 0.0001: return "****"
    elif p < 0.001:  return "***"
    elif p < 0.01:  return "**"
    elif p < 0.05:  return "*"
    else:  return ""

def load_gc_models(models_dir):
    models_dir = Path(models_dir)
    data = {}
    for pkl in models_dir.glob("*_model.pkl"):
        group = pkl.stem.replace("_model", "")
        resid_path = models_dir / f"{group}_residuals.npy"
        data[group] = {
            "model":     joblib.load(pkl),
            "residuals": np.load(resid_path) if resid_path.exists() else None,
        }
    return data
    
models_dir = Path("/work/10904/nikolchanchan/vista/figures_g4_t2t_NEW/PRMD9_fig/figure/gc_models")
models  = load_gc_models(models_dir)
mutations = ["smallins", "smalldel", "ins", "del_breakpoint_5prime", "del_breakpoint_3prime", "snp", "mnp"]

def empirical_pvalue(residuals: np.ndarray, observation: float) -> float:
    n = len(residuals)
    # p = (abs(residuals) >= abs(observation)).sum() / n
    p = (residuals >= observation).sum() / n
    return p

def compute_enrichment_merged(mut_expanded, ds_name, g4_bed):
    records = []
    for mutation in tqdm(mutations, desc=ds_name):
        mut_b = BedTool.from_dataframe(
            mut_expanded[mutation].drop_duplicates(subset=["#CHROM", "start"])[["#CHROM", "expanded_start", "expanded_end"]]
        ).sort().merge().sort()

        merged_df = mut_b.to_dataframe(names=["seqID", "start", "end"])
        merged_df["GC_content"] = merged_df.apply(
            lambda row: chrom_sequences[row["seqID"]][row["start"]:row["end"]].count("G") +
                        chrom_sequences[row["seqID"]][row["start"]:row["end"]].count("C"),
            axis=1
        )

        mut_b_gc = BedTool.from_dataframe(merged_df[["seqID", "start", "end", "GC_content"]]).sort()

        cov_cols = ["seqID", "start", "end", "GC_content", "total_hits", "overlapping_bases", "all_bases", "coverage"]
        g4_cov   = pd.read_table(mut_b_gc.coverage(g4_bed).fn, header=None, names=cov_cols)
        g4_density = g4_cov["overlapping_bases"].sum() * 1e6 / g4_cov["all_bases"].sum()

        records.append({
            "mutation":        mutation,
            "g4_density":      g4_density,
            "all_bases":       g4_cov["all_bases"].sum(),
            "GC_content":      g4_cov["GC_content"].sum(),
            "fold_enrichment": g4_density / gw_densities[ds_name],
        })

    return pd.DataFrame(records)

all_results_merged = []
g4hunter_bed = BedTool.from_dataframe(g4_df[["seqID", "start", "end"]]).sort()
regex_bed    = BedTool.from_dataframe(regex_df[["seqID", "start", "end"]]).sort()
g4hunter_filtered_bed = BedTool.from_dataframe(g4_filtered_df[["seqID", "start", "end"]]).sort()

datasets = [
    ("Quadparser", regex_bed),
    ("G4Hunter",   g4hunter_bed),
    ("G4Hunter_filtered", g4hunter_filtered_bed)
]
for ds_name, g4_bed in datasets:
    df = compute_enrichment_merged(mut_expanded, ds_name=ds_name, g4_bed=g4_bed)
    df["dataset"] = ds_name
    all_results_merged.append(df)

results_merged_df = pd.concat(all_results_merged, ignore_index=True)
results_merged_df["GC_prop"] = results_merged_df["GC_content"] / results_merged_df["all_bases"]
for ds_name, _ in datasets:
    mask = results_merged_df["dataset"] == ds_name
    results_merged_df.loc[mask, "predicted_enrichment"] = models[ds_name]["model"].predict(results_merged_df.loc[mask, "GC_prop"].to_numpy().reshape(-1, 1))
    results_merged_df.loc[mask, "residuals"] = results_merged_df.loc[mask, "fold_enrichment"] - results_merged_df.loc[mask, "predicted_enrichment"]
    results_merged_df.loc[mask, "pvalue"] = results_merged_df.loc[mask, "residuals"].apply(
            lambda obs: empirical_pvalue(models[ds_name]["residuals"], obs)
        )    
results_merged_df.loc[:, "fe_adj"] = results_merged_df["fold_enrichment"] / results_merged_df["predicted_enrichment"]
multipletests_results = multipletests(results_merged_df["pvalue"], method="fdr_bh")
results_merged_df["qvalue"] = multipletests_results[1]
results_merged_df

In [ ]:
def empirical_pvalue(residuals: np.ndarray, observation: float) -> float:
    n = len(residuals)
    p = (abs(residuals) >= abs(observation)).sum() / n
    p = (residuals >= observation).sum() / n
    return p

results_merged_df["GC_prop"] = results_merged_df["GC_content"] / results_merged_df["all_bases"]
for ds_name, _ in datasets:
    mask = results_merged_df["dataset"] == ds_name
    if ds_name == "G4Hunter_filtered":
        ds_name_model = "G4Hunter"
    else:
        ds_name_model = ds_name
    results_merged_df.loc[mask, "predicted_enrichment"] = models[ds_name_model]["model"].predict(results_merged_df.loc[mask, "GC_prop"].to_numpy().reshape(-1, 1))
    results_merged_df.loc[mask, "residuals"] = results_merged_df.loc[mask, "fold_enrichment"] - results_merged_df.loc[mask, "predicted_enrichment"]
    results_merged_df.loc[mask, "pvalue"] = results_merged_df.loc[mask, "residuals"].apply(
            lambda obs: empirical_pvalue(models[ds_name_model]["residuals"], obs)
        )    
results_merged_df.loc[:, "fe_adj"] = results_merged_df["fold_enrichment"] / results_merged_df["predicted_enrichment"]
multipletests_results = multipletests(results_merged_df["pvalue"], method="fdr_bh")
results_merged_df["qvalue"] = multipletests_results[1]
results_merged_df

In [ ]:
BAR_W = 0.35
mutation_labels = {
    "snp":                   "Substitution",
    "mnp":                   "Multi-nucleotide",
    "smalldel":              "Small Deletion",
    "smallins":              "Small Insertion",
    "ins":                   "Large Insertion",
    "del":                   "Large Deletion",
    "del_breakpoint_5prime": "5' Breakpoint",
    "del_breakpoint_3prime": "3' Breakpoint",
}

def pval_to_stars(p):
    if p < 0.001: return "***"
    elif p < 0.01:  return "**"
    elif p < 0.05:  return "*"
    else:           return "ns"

results_merged_df["stars"] = results_merged_df["qvalue"].apply(pval_to_stars)

for ds_name in ["G4Hunter", "Quadparser"]:
    ds_df  = results_merged_df[results_merged_df["dataset"] == ds_name]
    lookup = {r["mutation"]: r for r in ds_df.to_dict("records")}

    mut_order = (
        ds_df.sort_values("fold_enrichment", ascending=False)["mutation"]
        .drop_duplicates()
        .tolist()
    )

    x          = np.arange(len(mut_order))
    fe_raw     = np.array([lookup.get(m, {}).get("fold_enrichment", np.nan) for m in mut_order])
    fe_adj     = np.array([lookup.get(m, {}).get("fe_adj",          np.nan) for m in mut_order])
    residuals  = np.array([lookup.get(m, {}).get("residuals",       np.nan) for m in mut_order])
    stars_list = [lookup.get(m, {}).get("stars", "ns") for m in mut_order]

    vmax       = np.nanmax(np.abs(residuals))
    norm       = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    cmap       = cm.PuOr
    bar_colors = [cmap(norm(r)) if not np.isnan(r) else "gray" for r in residuals]

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor("white")

    ax.bar(x - BAR_W / 2, fe_raw, width=BAR_W, color=bar_colors,
           edgecolor="black", linewidth=1.5, alpha=0.85, zorder=3, label="Fold Enrichment")

    ax.bar(x + BAR_W / 2, fe_adj, width=BAR_W, color=bar_colors,
           edgecolor="black", ls="--", linewidth=2.5, alpha=0.85, zorder=3,
           label="GC-Adj. Fold Enrichment")

    ax.axhline(1.0, ls="--", color="gray", lw=3.0, zorder=0)

    ymax = np.nanmax(np.concatenate([fe_raw, fe_adj])) if not np.all(np.isnan(fe_adj)) else 2.0
    for xi, (yi, star) in enumerate(zip(fe_adj, stars_list)):
        if not np.isnan(yi) and star != "ns":
            ax.text(xi + BAR_W / 2 + 0.03, yi + ymax * 0.02, star,
                    ha="center", va="bottom", fontsize=14, fontweight="bold", rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels([mutation_labels.get(m, m) for m in mut_order],
                       rotation=45, ha="right", fontsize=18)
    ax.set_ylabel("Fold Enrichment", fontsize=18)
    ax.tick_params(axis="y", labelsize=18)
    ax.set_ylim(bottom=0, top=ymax * 1.35)
    ax.grid(axis="y", lw=0.4, alpha=0.6)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=14, frameon=False)

    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.06, pad=0.02)
    cbar.set_label("Residual", fontsize=20)
    cbar.ax.tick_params(labelsize=14)

    ax.text(
        0.5, 1.02, ds_name,
        transform=ax.transAxes,
        ha="center", va="bottom", fontsize=22, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="lightgray", edgecolor="black", linewidth=1.5),
    )

    fname = ds_name.lower().replace(" ", "_")
    fig.savefig(f"{target_fig}/mutation_enrichment_g4_{fname}.pdf",
                transparent=True, bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
BAR_W = 0.35
mutation_labels = {
    "snp":                   "Substitution",
    "mnp":                   "Multi-nucleotide",
    "smalldel":              "Small Deletion",
    "smallins":              "Small Insertion",
    "ins":                   "Large Insertion",
    "del":                   "Large Deletion",
    "del_breakpoint_5prime": "5' Breakpoint",
    "del_breakpoint_3prime": "3' Breakpoint",
}

def pval_to_stars(p):
    if p < 0.001: return "***"
    elif p < 0.01:  return "**"
    elif p < 0.05:  return "*"
    else:           return "ns"

results_merged_df["stars"] = results_merged_df["qvalue"].apply(pval_to_stars)

for ds_name in ["G4Hunter", "Quadparser"]:
    ds_df  = results_merged_df[results_merged_df["dataset"] == ds_name]
    lookup = {r["mutation"]: r for r in ds_df.to_dict("records")}

    mut_order = (
        ds_df.sort_values("fold_enrichment", ascending=False)["mutation"]
        .drop_duplicates()
        .tolist()
    )

    x          = np.arange(len(mut_order))
    fe_raw     = np.array([lookup.get(m, {}).get("fold_enrichment", np.nan) for m in mut_order])
    fe_adj     = np.array([lookup.get(m, {}).get("fe_adj",          np.nan) for m in mut_order])
    residuals  = np.array([lookup.get(m, {}).get("residuals",       np.nan) for m in mut_order])
    stars_list = [lookup.get(m, {}).get("stars", "ns") for m in mut_order]

    vmax       = np.nanmax(np.abs(residuals))
    norm       = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    cmap       = cm.PuOr
    bar_colors = [cmap(norm(r)) if not np.isnan(r) else "gray" for r in residuals]

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor("white")

    ax.bar(x - BAR_W / 2, fe_raw, width=BAR_W, color=bar_colors,
           edgecolor="black", linewidth=1.5, alpha=0.85, zorder=3, label="Fold Enrichment")

    ax.bar(x + BAR_W / 2, fe_adj, width=BAR_W, color=bar_colors,
           edgecolor="black", ls="--", linewidth=2.5, alpha=0.85, zorder=3,
           label="GC-Adj. Fold Enrichment")

    ax.axhline(1.0, ls="--", color="gray", lw=3.0, zorder=0)

    ymax = np.nanmax(np.concatenate([fe_raw, fe_adj])) if not np.all(np.isnan(fe_adj)) else 2.0
    for xi, (yi, star) in enumerate(zip(fe_adj, stars_list)):
        if not np.isnan(yi) and star != "ns":
            ax.text(xi + BAR_W / 2 + 0.03, yi + ymax * 0.02, star,
                    ha="center", va="bottom", fontsize=14, fontweight="bold", rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels([mutation_labels.get(m, m) for m in mut_order],
                       rotation=45, ha="right", fontsize=18)
    ax.set_ylabel("Fold Enrichment", fontsize=18)
    ax.tick_params(axis="y", labelsize=18)
    ax.set_ylim(bottom=0, top=ymax * 1.35)
    ax.grid(axis="y", lw=0.4, alpha=0.6)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=14, frameon=False)

    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.06, pad=0.02)
    cbar.set_label("Residual", fontsize=20)
    cbar.ax.tick_params(labelsize=14)

    ax.text(
        0.5, 1.02, ds_name,
        transform=ax.transAxes,
        ha="center", va="bottom", fontsize=22, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="lightgray", edgecolor="black", linewidth=1.5),
    )

    fname = ds_name.lower().replace(" ", "_")
    fig.savefig(f"{target_fig}/mutation_enrichment_g4_{fname}.pdf",
                transparent=True, bbox_inches="tight")
    fig.savefig(f"{target_fig}/mutation_enrichment_g4_{fname}.png",
            transparent=True, dpi=300,bbox_inches="tight")
    plt.show()
    plt.close()


In [ ]:
import pyranges as pr
from pybedtools import BedTool
from statsmodels.stats.multitest import multipletests
import numpy as np

mutations = ["smallins", "ins", "del_breakpoint_5prime", "del_breakpoint_3prime",
             "mnp", "smalldel", "snp"]

g4_bed    = BedTool.from_dataframe(g4_df).sort()
regex_bed = BedTool.from_dataframe(regex_df).sort()

datasets_with_df = [
    ("G4Hunter",   g4_bed),
    ("Quadparser", regex_bed),
]
regions_df = pd.read_table(Path(os.getenv('WORK')).joinpath("compartments_coords.tsv.gz"))
mut_dfs  = {}
mut_beds = {}
regions_bed = BedTool.from_dataframe(
    regions_df[["seqID", "start", "end", "group"]].rename(columns={"seqID": "chrom"})
).sort()

for mutation in mutations:
    # expanded window as the BED interval so f=1.0 checks containment of the window
    intersected = pd.read_table(
        BedTool.from_dataframe(
            mut_expanded[mutation][["#CHROM", "expanded_start", "expanded_end", "start", "end", "GC_content"]]
            .drop_duplicates(subset=["#CHROM", "start", "end"])
            .rename(columns={"#CHROM": "seqID"})
        ).sort().intersect(regions_bed, f=1.0, wo=True).fn,
        header=None,
        names=["seqID", "expanded_start", "expanded_end", "start", "end", "GC_content",
               "chrom", "region_start", "region_end", "group", "overlap"]
    )[["seqID", "expanded_start", "expanded_end", "start", "end", "GC_content",
       "chrom", "region_start", "region_end", "group", "overlap"]]

    merged_records = []
    for group_name, group_df in intersected.groupby("group"):
        merged_bed = (
            BedTool.from_dataframe(
                group_df[["seqID", "expanded_start", "expanded_end"]]
                .rename(columns={"expanded_start": "start", "expanded_end": "end"})
            ).sort().merge()
        )
        merged_df = merged_bed.to_dataframe(names=["seqID", "start", "end"])
        merged_df["GC_content"] = merged_df.apply(
            lambda row: (
                chrom_sequences[row["seqID"]][row["start"]:row["end"]].count("G") +
                chrom_sequences[row["seqID"]][row["start"]:row["end"]].count("C")
            ),
            axis=1,
        )
        merged_df["group"] = group_name
        merged_records.append(merged_df)

    merged_all = pd.concat(merged_records, ignore_index=True)
    mut_dfs[mutation]  = merged_all
    mut_beds[mutation] = BedTool.from_dataframe(
        merged_all[["seqID", "start", "end", "GC_content", "group"]]
    ).sort()

    print(f"DONE merging {mutation} across {intersected['group'].nunique()} compartments")

cov_names = ["seqID", "start", "end", "GC_content", "group",
             "total_hits", "overlapping_bp", "all_bases", "coverage"]

data = []
for ds_name, df_bed in tqdm(datasets_with_df):
    for mutation in mut_beds:
        temp_df = (
            pd.read_table(mut_beds[mutation].coverage(df_bed).fn, header=None, names=cov_names)
            .assign(at_least_one_g4 = lambda d: (d["overlapping_bp"] > 0).astype(int))
            .groupby("group", as_index=False)
            .agg(GC_content=("GC_content", "sum"),
                 all_bases=("all_bases", "sum"),
                 at_least_one_g4=("at_least_one_g4", "sum"),
                 overlapping_bp=("overlapping_bp", "sum"),
                 total_regions=("group", "count")
                 )
            .assign(
                g4_density      = lambda d: d["overlapping_bp"] * 1e6 / d["all_bases"],
                fold_enrichment = lambda d: d["overlapping_bp"] * 1e6 / d["all_bases"] / gw_densities[ds_name],
                GC_prop         = lambda d: d["GC_content"] / d["all_bases"],
                g4_presence = lambda d: 1e2 * d["at_least_one_g4"] / d["total_regions"]
            )
        )
        temp_df["dataset"]  = ds_name
        temp_df["mutation"] = mutation
        data.append(temp_df)

data = pd.concat(data, ignore_index=True)
data


In [ ]:
def empirical_pvalue(residuals: np.ndarray, observation: float) -> float:
    n = len(residuals)
    p = (abs(residuals) >= abs(observation)).sum() / n
    return p

data["GC_prop"] = data["GC_content"] / data["all_bases"]
datasets = ["G4Hunter", "Quadparser"]
for ds_name in datasets:
    mask = data["dataset"] == ds_name
    data.loc[mask, "predicted_enrichment"] = models[ds_name]["model"].predict(data.loc[mask, "GC_prop"].to_numpy().reshape(-1, 1))
    data.loc[mask, "residuals"] = data.loc[mask, "fold_enrichment"] - data.loc[mask, "predicted_enrichment"]
    data.loc[mask, "pvalue"] = data.loc[mask, "residuals"].apply(
            lambda obs: empirical_pvalue(models[ds_name]["residuals"], obs)
        )    
data.loc[:, "fe_adj"] = data["fold_enrichment"] / data["predicted_enrichment"]
multipletests_results = multipletests(data["pvalue"], method="fdr_bh")
data["qvalue"] = multipletests_results[1]
data

In [ ]:
from pybedtools import BedTool
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

def calculate_total_bases(bed):
    df = bed.merge().to_dataframe(names=["seqID", "start", "end"])
    if df.empty:
        return 0
    return (df["end"] - df["start"]).sum()

def conditional_enrichment(H_bed, G_bed, R_bed):
    """
    enrichment = P(mutation | G4, compartment) / P(mutation | compartment)

    H = mutation windows
    G = G4 regions  
    R = compartment (the new 'world')
    """
    R_bed = R_bed.sort().merge()
    G_bed = G_bed.sort().merge()
    H_bed = H_bed.sort().merge()

    R_bases = calculate_total_bases(R_bed)
    if R_bases == 0:
        return None

    G_and_R       = G_bed.intersect(R_bed).sort().merge()
    G_and_R_bases = calculate_total_bases(G_and_R)
    if G_and_R_bases == 0:
        return None

    H_and_R       = H_bed.intersect(R_bed).sort().merge()
    H_and_R_bases = calculate_total_bases(H_and_R)
    if H_and_R_bases == 0:
        return None

    H_and_G_and_R       = H_bed.intersect(G_and_R).sort().merge()
    H_and_G_and_R_bases = calculate_total_bases(H_and_G_and_R)

    prob_H_given_GR = H_and_G_and_R_bases / G_and_R_bases
    prob_H_given_R  = H_and_R_bases        / R_bases
    enrichment      = prob_H_given_GR / prob_H_given_R

    # Fisher: independence of mutation and G4 within R (base-level 2x2 table)
    a = H_and_G_and_R_bases                                              # mutated G4
    b = G_and_R_bases - H_and_G_and_R_bases                             # G4 not mutated
    c = H_and_R_bases - H_and_G_and_R_bases                             # mutated non-G4
    d = R_bases - G_and_R_bases - H_and_R_bases + H_and_G_and_R_bases   # neither
    odds, pval = fisher_exact([[a, b], [c, d]], alternative="greater")

    return {
        "R_bases":             R_bases,
        "G_and_R_bases":       G_and_R_bases,
        "H_and_R_bases":       H_and_R_bases,
        "H_and_G_and_R_bases": H_and_G_and_R_bases,
        "prob_H_given_GR":     prob_H_given_GR,
        "prob_H_given_R":      prob_H_given_R,
        "enrichment":          enrichment,
        "odds":                odds,
        "pvalue":              pval,
    }

# build per-compartment BedTools once
compartment_beds = {
    group_name: BedTool.from_dataframe(
        group_df[["seqID", "start", "end"]].rename(columns={"seqID": "chrom"})
    ).sort()
    for group_name, group_df in regions_df.groupby("group")
}

datasets_g4 = {"G4Hunter": g4_bed, "Quadparser": regex_bed}

cond_results = []
for ds_name, G_bed in datasets_g4.items():
    for mutation in tqdm(mutations, desc=ds_name):
        for group_name, R_bed in compartment_beds.items():
            group_df = mut_dfs[mutation][mut_dfs[mutation]["group"] == group_name]
            if group_df.empty:
                continue
            H_bed  = BedTool.from_dataframe(group_df[["seqID", "start", "end"]]).sort()
            result = conditional_enrichment(H_bed, G_bed, R_bed)
            if result is None:
                continue
            result.update({"dataset": ds_name, "mutation": mutation, "group": group_name})
            cond_results.append(result)

cond_df = pd.DataFrame(cond_results)

_, cond_df["qvalue"], _, _ = multipletests(cond_df["pvalue"], method="fdr_bh")
cond_df["mutation_label"] = cond_df["mutation"].map(mutation_labels).fillna(cond_df["mutation"])
cond_df


In [ ]:
cond_df.to_csv(f"{target_fig}/g4_enrichment_by_compartment.tsv.gz", sep="\t", index=False)
data.to_csv(f"{target_fig}/g4_enrichment_by_compartment_gc_adjusted.tsv.gz", sep="\t", index=False)


In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist
from matplotlib.colors import TwoSlopeNorm
from matplotlib.gridspec import GridSpec
import matplotlib.cm as cm
from matplotlib.patches import Patch

mutation_labels = {
    "snp":                   "Substitution",
    "mnp":                   "Multi-nucleotide",
    "smalldel":              "Small Deletion",
    "smallins":              "Small Insertion",
    "ins":                   "Large Insertion",
    "del_breakpoint_5prime": "5' Breakpoint",
    "del_breakpoint_3prime": "3' Breakpoint",
}

P_G4 = {"G4Hunter": 0.01810, "Quadparser": 0.00336}
COV_THRESHOLDS = {
    "G4Hunter":   22_000,
    "Quadparser": 119_000,
}

VMAX, VMIN = 5, -3
FS         = 18

fig = plt.figure(figsize=(14, 16))
gs  = GridSpec(3, 2, figure=fig, hspace=0.3, wspace=0.6,
               height_ratios=[15, 1.2, 0.4])

axes    = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])]
cbar_ax = fig.add_subplot(gs[2, :])

for ax, ds_name in zip(axes, ["G4Hunter", "Quadparser"]):
    cov_threshold = COV_THRESHOLDS[ds_name]
    sub = data[data["dataset"] == ds_name].copy()

    sub["arrow"] = ""
    sig = sub["qvalue"] < 0.05
    sub.loc[sig & (sub["residuals"] > 0), "arrow"] = "↑"
    sub.loc[sig & (sub["residuals"] < 0), "arrow"] = "↓"
    sub["low_cov"] = sub["all_bases"] < cov_threshold

    sufficient_groups = (
        sub.groupby("group")["all_bases"].max()
        .loc[lambda s: s >= cov_threshold].index
    )
    sub = sub[sub["group"].isin(sufficient_groups)]

    sig_groups = sub.loc[sig, "group"].unique()
    sub = sub[sub["group"].isin(sig_groups)]

    heatmap_vals = sub.pivot(index="group", columns="mutation", values="fold_enrichment")
    arrows       = sub.pivot(index="group", columns="mutation", values="arrow")
    low_cov      = sub.pivot(index="group", columns="mutation", values="low_cov")

    for df in [heatmap_vals, arrows, low_cov]:
        df.columns = [mutation_labels.get(c, c) for c in df.columns]

    data_log = np.log2(heatmap_vals.clip(lower=1e-10))

    row_idx = leaves_list(linkage(pdist(data_log.fillna(0).values),   method="ward"))
    col_idx = leaves_list(linkage(pdist(data_log.fillna(0).values.T), method="ward"))
    data_log = data_log.iloc[row_idx, col_idx]
    arrows   = arrows.iloc[row_idx, col_idx]
    low_cov  = low_cov.iloc[row_idx, col_idx]

    sns.heatmap(
        data_log, ax=ax,
        cmap="PuOr", center=0, vmin=VMIN, vmax=VMAX,
        linewidths=0.5, linecolor="black",
        cbar=False, xticklabels=True, yticklabels=True,
    )

    for i in range(data_log.shape[0]):
        for j in range(data_log.shape[1]):
            is_low = low_cov.iloc[i, j]
            arrow  = arrows.iloc[i, j]
            if arrow and not is_low:
                color = "#e075ce" if arrow == "↑" else "blue"
                ax.text(j + 0.5, i + 0.5, arrow,
                        ha="center", va="center", fontsize=FS + 4,
                        color=color, fontweight="bold", zorder=3)
            if is_low:
                ax.add_patch(plt.Rectangle(
                    (j + 0.05, i + 0.05), 0.9, 0.9,
                    fill=False, linestyle="--", edgecolor="black",
                    linewidth=1.5, zorder=2,
                ))

    ax.tick_params(axis="y", rotation=0,  labelsize=FS + 1)
    ax.tick_params(axis="x", rotation=45, labelsize=FS + 6)
    plt.setp(ax.get_xticklabels(), ha="right")
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.set_title(ds_name, fontsize=FS + 12, fontweight="bold", y=1.02,
                 bbox=dict(boxstyle="round,pad=0.55", facecolor="#f0f0f0",
                           edgecolor="#333333", lw=1.5))

fig.legend(
    handles=[Patch(facecolor="none", edgecolor="black", linestyle="--",
                   label="Insufficient Coverage (CV > 5%)")],
    loc="lower center", bbox_to_anchor=(0.5, 0.12),
    fontsize=FS - 2, frameon=True,
)

norm = TwoSlopeNorm(vmin=VMIN, vcenter=0, vmax=VMAX)
sm   = cm.ScalarMappable(cmap="PuOr", norm=norm)
cb   = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cb.set_label("log₂(Fold Enrichment)", fontsize=FS + 8)
cb.ax.tick_params(labelsize=FS + 5)

fig.savefig(target_fig / "mutation_g4_region_enrichment.pdf",
            transparent=True, bbox_inches="tight")
fig.savefig(target_fig / "mutation_g4_region_enrichment.png",
            dpi=300, bbox_inches="tight")
plt.show()


## Enrichment of rare vs commonm

In [ ]:
from scipy.stats import fisher_exact, wilcoxon
from statsmodels.stats.multitest import multipletests
import numpy as np

target = os.getenv("SCRATCH")
target = Path(target).joinpath("g4_t2t_revisions_data", "mut_analysis")
target_fig = target.joinpath("mut_figures"); target_fig.mkdir(exist_ok=True, parents=True)

# 
RARE_THRESHOLD = 0.05
mut_df.loc[:, "AF_samples"] = mut_df["AC_samples"] / mut_df["AN_samples"]
target = Path(target).joinpath("g4_t2t_revisions_data", "mut_analysis")
target_fig = target.joinpath("mut_figures"); target_fig.mkdir(exist_ok=True, parents=True)
common_df_filtered = mut_df[(mut_df["AF_samples"] > RARE_THRESHOLD) & (mut_df["AN_samples"] >= 73)].reset_index(drop=True)
rare_df_filtered = mut_df[(mut_df["AF_samples"] <= RARE_THRESHOLD) & (mut_df["AN_samples"] >= 73)].reset_index(drop=True)
FIELDS = ["#CHROM", "start", "end", "mutation", "mut_type", "REF", "ALT", "AC_samples", "AN_samples"]
common_df_filtered = common_df_filtered[FIELDS]
rare_df_filtered = rare_df_filtered[FIELDS]

mutations = ["smallins", "ins", "del",  "mnp", "smalldel", "snp"]
RARITY_COLLECTIONS = {
    "low_frequency":   rare_df_filtered,
    "common": common_df_filtered,
    "all":   mut_df,
}

In [ ]:
from scipy.stats import fisher_exact
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test
import numpy as np

def greedy_independent_pairs(cluster_map):
    """
    Within each overlap cluster, greedily select non-overlapping G4s
    by sorting by end position (activity selection algorithm).
    Returns pair_idx array of selected pairs.
    """
    def _select(group):
        group = group.sort_values("g4_end")
        selected, last_end = [], -1
        for _, row in group.iterrows():
            if row["g4_start"] >= last_end:
                selected.append(row["pair_idx"])
                last_end = row["g4_end"]
        return selected

    all_selected = []
    for _, group in cluster_map.groupby("cluster_id"):
        all_selected.extend(_select(group))
    return np.array(all_selected)

def compute_enrichment(mut_df, label, g4_bed, ctrl_bed, ds_name, matched_df):
    g4_merged   = g4_bed.sort().merge().sort()
    ctrl_merged = ctrl_bed.sort().merge().sort()

    records = []
    mutations = ["del", "ins", "smallins", "mnp", "smalldel", "snp"]
    for mutation in tqdm(mutations, desc=label):
        mut_b = BedTool.from_dataframe(
            mut_df[mut_df["mut_type"] == mutation]
            .drop_duplicates(subset=["#CHROM", "start"])
            [["#CHROM", "start", "end"]]
        ).sort().merge()

        cov_cols = ["seqID", "start", "end", "total_hits", "overlapping_bases", "all_bases", "coverage"]

        g4_cov_m   = pd.read_table(g4_merged.coverage(mut_b).fn,   header=None, names=cov_cols)
        ctrl_cov_m = pd.read_table(ctrl_merged.coverage(mut_b).fn, header=None, names=cov_cols)
        g4_cov     = pd.read_table(g4_bed.coverage(mut_b).fn,      header=None, names=cov_cols)
        ctrl_cov   = pd.read_table(ctrl_bed.coverage(mut_b).fn,    header=None, names=cov_cols)

        matched_new_df = (
            matched_df[["seqID", "start", "end", "control_seqID", "control_start", "control_end"]]
            .merge(
                g4_cov[["seqID", "start", "end", "total_hits", "coverage"]],
                how="inner", on=["seqID", "start", "end"],
            )
            .merge(
                ctrl_cov[["seqID", "start", "end", "total_hits", "coverage"]],
                how="inner",
                left_on=["control_seqID", "control_start", "control_end"],
                right_on=["seqID", "start", "end"],
                suffixes=("_g4", "_ctrl"),
            )
            .reset_index(drop=True)
        )
        assert matched_df.shape[0] == matched_new_df.shape[0], (
            f"{matched_df.shape[0]} vs {matched_new_df.shape[0]} for {mutation} in {label} of {ds_name}"
        )

        # McNemar on non-overlapping subset (one G4 per overlap cluster)
        mc_pairs = matched_new_df
        g4_hit   = mc_pairs["total_hits_g4"]   > 0
        ctrl_hit = mc_pairs["total_hits_ctrl"] > 0
        a = int(( g4_hit &  ctrl_hit).sum())
        b = int(( g4_hit & ~ctrl_hit).sum())   # G4 hit, ctrl not
        c = int((~g4_hit &  ctrl_hit).sum())   # ctrl hit, G4 not
        d = int((~g4_hit & ~ctrl_hit).sum())
        mn         = mcnemar_test([[a, b], [c, d]], exact=False)
        pval_mn    = mn.pvalue
        paired_or  = b / c if c > 0 else np.nan

        # Fisher on merged intervals
        g4_yes   = int((g4_cov_m["total_hits"]  > 0).sum()); g4_no   = len(g4_cov_m)   - g4_yes
        ctrl_yes = int((ctrl_cov_m["total_hits"] > 0).sum()); ctrl_no = len(ctrl_cov_m) - ctrl_yes
        odds, pval_fisher = fisher_exact([[g4_yes, g4_no], [ctrl_yes, ctrl_no]], alternative="greater")

        # Fisher on unmerged motifs
        mg4_yes   = int((g4_cov["total_hits"]  > 0).sum()); mg4_no   = len(g4_cov)   - mg4_yes
        mctrl_yes = int((ctrl_cov["total_hits"] > 0).sum()); mctrl_no = len(ctrl_cov) - mctrl_yes
        modds, mpval_fisher = fisher_exact([[mg4_yes, mg4_no], [mctrl_yes, mctrl_no]], alternative="greater")

        # density on merged intervals
        g4_bases     = g4_cov_m["all_bases"].sum()
        ctrl_bases   = ctrl_cov_m["all_bases"].sum()
        g4_density   = g4_cov_m["overlapping_bases"].sum()   * 1e6 / g4_bases
        ctrl_density = ctrl_cov_m["overlapping_bases"].sum() * 1e6 / ctrl_bases

        records.append({
            "mutation":            mutation,
            "frequency_class":     label,
            "g4_density":          g4_density,
            "ctrl_density":        ctrl_density,
            "rel_fold_enrichment": g4_density / ctrl_density if ctrl_density > 0 else np.nan,
            "log2_fe":             np.log2(g4_density / ctrl_density) if ctrl_density > 0 and g4_density > 0 else np.nan,
            # McNemar (paired, independent subset)
            "paired_or":           paired_or,
            "pval_mcnemar":        pval_mn,
            "mcnemar_b":           b,
            "mcnemar_c":           c,
            # Fisher (merged)
            "odds":                odds,
            "pval_fisher":         pval_fisher,
            # Fisher (unmerged motifs)
            "modds":               modds,
            "mpval_fisher":        mpval_fisher,
        })

    return pd.DataFrame(records)

In [ ]:
datatypes = ["masked"]
ds_names = ["G4Hunter"]
matched_dfs = dict()
matched_meth_dfs = dict()

CELLS = ["HG002", "CHM13"]
STATES = ["Hypomethylated", "Methylated", "Hypermethylated"]
# matched_controls_masked_G4Hunter_global.masked.both.filtered.__final__.tsv.gz
# matched_controls_unmasked_G4Hunter_global.masked.both.filtered.__final__.tsv.gz
# METH PATH
# /scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_controls/matched_controls_masked_G4Hunter_HG002_with_meth.__final__.tsv.gz
for datatype in datatypes:
    for ds_name in ds_names:
        # /scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_controls_GHunter_unmasked_global.masked.both.filtered.__final__.tsv.gz
        matched_dfs[ds_name, datatype] = pd.read_table(f"/scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_controls_{datatype}_{ds_name}_global.masked.both.filtered.__final__.tsv.gz")\
                    .sort_values(by=["seqID", "start"]).reset_index(drop=True)

        for tissue in CELLS:
            matched_meth_dfs[tissue, ds_name, datatype] = pd.read_table(f"/scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_controls/matched_controls_{datatype}_{ds_name}_{tissue}_with_meth.__final__.tsv.gz")

matched_dfs["G4Hunter", "masked"], len(matched_meth_dfs)

In [ ]:
datatypes = ["masked"]
ds_names = ["G4Hunter"]
matched_dfs = dict()
matched_meth_dfs = dict()

CELLS = ["HG002", "CHM13"]
STATES = ["Hypomethylated", "Methylated", "Hypermethylated"]
# matched_controls_masked_G4Hunter_global.masked.both.filtered.__final__.tsv.gz
# matched_controls_unmasked_G4Hunter_global.masked.both.filtered.__final__.tsv.gz
# METH PATH
# /scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_controls/matched_controls_masked_G4Hunter_HG002_with_meth.__final__.tsv.gz
for datatype in datatypes:
    for ds_name in ds_names:
        # /scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_controls_GHunter_unmasked_global.masked.both.filtered.__final__.tsv.gz
        matched_dfs[ds_name, datatype] = pd.read_table(f"/scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_controls_{datatype}_{ds_name}_global.masked.both.filtered.__final__.tsv.gz")\
                    .sort_values(by=["seqID", "start"]).reset_index(drop=True)

        for tissue in CELLS:
            matched_meth_dfs[tissue, ds_name, datatype] = pd.read_table(f"/scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_controls/matched_controls_{datatype}_{ds_name}_{tissue}_with_meth.__final__.tsv.gz")

matched_dfs["G4Hunter", "masked"], len(matched_meth_dfs)

In [ ]:
matched_dfs[ds_name, datatype]

In [ ]:
datatypes = ["masked"]
ds_names = ["G4Hunter"]
matched_dedup_dfs = dict()

CELLS = ["HG002", "CHM13"]
STATES = ["Hypomethylated", "Methylated", "Hypermethylated"]
def deduplicate_g4s(matched_df):
    df = matched_df.reset_index(drop=True)
    df_sorted = df.sort_values(["seqID", "end"])
    selected, last_end = [], {}
    for _, row in df_sorted.iterrows():
        chrom = row["seqID"]
        if row["start"] >= last_end.get(chrom, -1):
            selected.append(row.name)
            last_end[chrom] = row["end"]
    return matched_df.iloc[selected].reset_index(drop=True)

for datatype in datatypes:
    for ds_name in ds_names:
        matched_dedup_dfs[ds_name, datatype] = deduplicate_g4s(matched_dfs[ds_name, datatype])

        for tissue in CELLS:
            matched_meth_dfs[tissue, ds_name, datatype] = deduplicate_g4s(matched_meth_dfs[tissue, ds_name, datatype])

matched_dfs["G4Hunter", "masked"], len(matched_meth_dfs)


In [ ]:
import re
import itertools
# # # #
def decompose_g4(g4_df):
    loops_df       = []
    gruns_df       = []
    transitions_df = []
    for row in g4_df.itertuples(index=False):
        seqID         = row.seqID
        start         = row.start
        end           = row.end
        sequence      = row.sequence.upper()
        grun_char     = sequence[0]

        gruns = list(re.finditer(r"%s{3,}" % re.escape(grun_char), sequence))
        if len(gruns) < 2:
            continue

        gruns_df.append({
            "seqID":            seqID,
            "start":            start,
            "end":              start + gruns[0].span()[1],
            "type":             "grun",
            "motif_start":      start,
            "motif_end":        end,
            "sequence":         sequence[:gruns[0].span()[1]],
        })

        for prev_grun, cur_grun in zip(gruns, gruns[1:]):
            g_start_prev, g_end_prev = prev_grun.span()
            g_start_cur,  g_end_cur  = cur_grun.span()

            loops_df.append({
                "seqID":            seqID,
                "start":            start + g_end_prev,
                "end":              start + g_start_cur,
                "motif_start":      start,
                "motif_end":        end,
                "type":             "loop",
                "sequence":         sequence[g_end_prev:g_start_cur],
            })

            run_seq = sequence[g_start_cur:g_end_cur]
            gruns_df.append({
                "seqID":            seqID,
                "start":            start + g_start_cur,
                "end":              start + g_end_cur,
                "motif_start":      start,
                "motif_end":        end,
                "type":             "grun",
                "sequence":         run_seq,
            })

            for offset in [g_end_prev, g_start_cur]:
                transitions_df.append({
                    "seqID":            seqID,
                    "start":            start + offset - 1,
                    "end":              start + offset + 1,
                    "motif_start":      start,
                    "motif_end":        end,
                    "type":             "transition",
                    "sequence":         sequence[offset - 1: offset + 1],
                })

    gruns_df       = pd.DataFrame(gruns_df)
    loops_df       = pd.DataFrame(loops_df)
    transitions_df = pd.DataFrame(transitions_df)

    return {
        "G-runs": gruns_df,
        "loops": loops_df,
        "transitions": transitions_df,
    }

decomposed = dict()
for ds_name in ds_names:
    for masked in ["masked", "unmasked"]:
        decomposed[ds_name, masked] = decompose_g4(matched_dfs[ds_name, masked])
        
decomposed["G4Hunter", "masked"]["G-runs"]

In [ ]:
temp_df = matched_dfs["G4Hunter", "masked"].copy()
temp_df.loc[:, "canonical"] = temp_df["sequence"].apply(lambda seq: ''.join({'a': 't', 't': 'a', 'c': 'g', 'g': 'c'}.get(base, base) for base in reversed(seq)))
temp_df.loc[:, "g-runs_total"] = temp_df["canonical"].apply(lambda seq: len(re.findall(r"g{3,}", seq)))
temp_df = temp_df[temp_df["g-runs_total"] == 4].reset_index(drop=True)
temp_df

In [ ]:
datasets_exploded = []
for (ds_name, mask), matched_df in matched_dfs.items():
    dedup_df = deduplicate_g4s(matched_df)
    print(f"[{ds_name} | {mask}] {len(matched_df)} → {len(dedup_df)} G4s kept ({len(dedup_df)/len(matched_df):.1%})")
    g4_bed = BedTool.from_dataframe(dedup_df[["seqID", "start", "end"]]).sort()
    ctrl_bed = BedTool.from_dataframe(
        dedup_df[["control_seqID", "control_start", "control_end"]].rename(columns={
            "control_seqID": "seqID", "control_start": "start", "control_end": "end",
        })
    ).sort()
    datasets_exploded.append((ds_name, mask, g4_bed, ctrl_bed, dedup_df))

# METH
datasets_meth_exploded = []
for (tissue, ds_name, mask), matched_df in matched_meth_dfs.items():
    for LEVEL in METH_LEVELS:
        temp_df = matched_df[matched_df["methylation_level"] == LEVEL]
        dedup_df = deduplicate_g4s(temp_df)
        print(f"[{tissue} | {ds_name} | {mask} | {LEVEL}] {len(temp_df)} → {len(dedup_df)} G4s kept ({len(dedup_df)/len(temp_df):.1%})")
        g4_bed = BedTool.from_dataframe(dedup_df[["seqID", "start", "end"]]).sort()
        ctrl_bed = BedTool.from_dataframe(
            dedup_df[["control_seqID", "control_start", "control_end"]].rename(columns={
                "control_seqID": "seqID", "control_start": "start", "control_end": "end",
            })
        ).sort()
        datasets_meth_exploded.append((LEVEL, tissue, ds_name, mask, g4_bed, ctrl_bed, dedup_df))


In [ ]:
all_results = []
for (ds_name, mask, g4_bed, ctrl_bed, df_temp) in datasets_exploded:
    for rarity, mut_dict in RARITY_COLLECTIONS.items():
        df = compute_enrichment(mut_dict, 
                                label=rarity, 
                                g4_bed=g4_bed, 
                                ctrl_bed=ctrl_bed, 
                                ds_name=ds_name, 
                                matched_df=df_temp)
        df["dataset"] = ds_name
        df["masking"] = mask
        all_results.append(df)

all_results = pd.concat(all_results, ignore_index=True)
all_results


In [ ]:
all_meth_results = []
for (LEVEL, tissue, ds_name, mask, g4_bed, ctrl_bed, df_temp) in datasets_meth_exploded:
    for rarity, mut_dict in RARITY_COLLECTIONS.items():
        df = compute_enrichment(mut_dict, 
                                label=rarity, 
                                g4_bed=g4_bed, 
                                ctrl_bed=ctrl_bed, 
                                ds_name=ds_name, 
                                matched_df=df_temp)
        df["dataset"] = ds_name
        df["masking"] = mask
        df["tissue"] = tissue
        df["methylation_level"] = LEVEL
        all_meth_results.append(df)

all_meth_results = pd.concat(all_meth_results, ignore_index=True)
all_meth_results


In [ ]:
for pval_col, qval_col in [
    ("pval_fisher",   "qval_fisher"),
    ("pval_mcnemar",       "qval_mcnemar"),
    ("mpval_fisher",  "qval_prop_bp"),

]:
    _, qvals, _, _ = multipletests(all_results[pval_col], method="fdr_bh")
    all_results[qval_col] = qvals

    _, qvals_meth, _, _ = multipletests(all_meth_results[pval_col], method="fdr_bh")
    all_meth_results[qval_col] = qvals_meth

In [ ]:
all_results.to_csv(target / "mutation_g4_enrichment_results_MASKED.tsv.gz", sep="\t", index=False, compression="gzip")
all_meth_results.to_csv(target / "mutation_g4_enrichment_results_MASKED_with_meth.tsv.gz", sep="\t", index=False, compression="gzip")

In [ ]:
all_results = []
for (ds_name, mask, g4_bed, ctrl_bed, df_temp) in datasets_exploded:
    for rarity, mut_dict in RARITY_COLLECTIONS.items():
        df = compute_enrichment(mut_dict, 
                                label=rarity, 
                                g4_bed=g4_bed, 
                                ctrl_bed=ctrl_bed, 
                                ds_name=ds_name, 
                                matched_df=df_temp)
        df["dataset"] = ds_name
        df["masking"] = mask
        all_results.append(df)

all_results = pd.concat(all_results, ignore_index=True)
all_results


In [ ]:
mutation_labels = {
    "del": "Large Deletion",
    "ins": "Large Insertion",
    "smallins": "Small Insertion",
    "mnp": "Multi-nucleotide",
    "smalldel": "Small Deletion",
    "snp": "Substitution",
}
def p_to_star(p):
    if p < 1e-4:  return "****"
    if p < 1e-3:  return "***"
    if p < 1e-2:  return "**"
    if p < 0.05:  return "*"
    return "ns"

rv = all_results[all_results["frequency_class"] == "all"].copy()
rv["mutation_label"] = rv["mutation"].map(mutation_labels).fillna(rv["mutation"])
g4h = rv
g4h["g4_density_kb"]   = g4h["g4_density"]   / 1e3
g4h["ctrl_density_kb"] = g4h["ctrl_density"] / 1e3

g4h["mutation_label"] = g4h["mutation"].map(mutation_labels).fillna(g4h["mutation"])
mut_order = (
    g4h.groupby("mutation_label")["paired_or"]
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
FS = 18

for mask in ["masked"]:
    sub = g4h # [g4h["masking"] == mask].copy()

    long = (
        sub[["mutation_label", "g4_density_kb", "ctrl_density_kb", "paired_or", "qval_mcnemar"]]
        .melt(
            id_vars=["mutation_label", "paired_or", "qval_mcnemar"],
            value_vars=["g4_density_kb", "ctrl_density_kb"],
            var_name="group", value_name="density",
        )
    )
    long["group"] = long["group"].map({"g4_density_kb": "G4", "ctrl_density_kb": "Control"})

    fig, ax = plt.subplots(figsize=(9, 4))
    sns.barplot(
        data=long, x="mutation_label", y="density",
        hue="group", order=mut_order, hue_order=["G4", "Control"],
        palette={"G4": "#ba8de0", "Control": "#d3d3d3"},
        edgecolor="black", lw=1.2, ax=ax, errorbar=None,
    )

    # dotted border on Control bars only
    for patch in ax.containers[1]:
        patch.set_linestyle((0, (3, 3)))

    ax.set_xlabel("", fontsize=FS)
    ax.set_ylabel("Variant Density (per kb)", fontsize=FS)
    ax.tick_params(axis="x", rotation=35, labelsize=FS - 2)
    ax.tick_params(axis="y", labelsize=FS - 2)
    plt.setp(ax.get_xticklabels(), ha="right")
    ax.grid(axis="y", lw=0.4, alpha=0.6)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(title="", fontsize=FS - 3, frameon=False)
    ax.set_yscale("log")

    g4_bars   = ax.containers[0]
    ctrl_bars = ax.containers[1]

    for i, mut in enumerate(mut_order):
        row = sub[sub["mutation_label"] == mut]
        if row.empty:
            continue
        or_val = row["paired_or"].values[0]
        star   = p_to_star(row["qval_mcnemar"].values[0])
        star   = "" if star == "ns" else star
        label  = f"OR={or_val:.1f}{star}"

        x1 = g4_bars[i].get_x()   + g4_bars[i].get_width()   / 2
        x2 = ctrl_bars[i].get_x() + ctrl_bars[i].get_width() / 2
        y  = max(g4_bars[i].get_height(), ctrl_bars[i].get_height())
        y_base = y * 1.25
        y_line = y * 1.40

        ax.plot([x1, x1, x2, x2], [y_base, y_line, y_line, y_base],
                lw=1.2, color="black", clip_on=False)
        ax.text((x1 + x2) / 2, y_line, label,
                ha="center", va="bottom", fontsize=FS - 4)

    fig.savefig(target_fig / f"g4_density_barplot_{mask}.pdf", transparent=True, bbox_inches="tight")
    plt.show()


In [ ]:
target_fig / f"g4_density_barplot_{mask}.pdf"

In [ ]:
all_results_meth_temp = all_meth_results[all_meth_results["tissue"] == tissue]
all_results_meth_temp.query("mutation == 'snp'")

In [ ]:
from matplotlib.colors import TwoSlopeNorm

tissue = "HG002"
FS = 18
METH_LEVELS = ["Hypomethylated", "Methylated", "Hypermethylated"]
MUT_LABELS = {
    "snp": "Substitution",
    "mnp": "Multi-nucleotide",
    "smalldel": "Small Deletion",
    "smallins": "Small Insertion",
    "ins": "Large Insertion",
    "del": "Large Deletion",
}
all_results_meth_temp = all_meth_results[all_meth_results["tissue"] == tissue]
vmax = np.nanmax(all_results_meth_temp["paired_or"].values)
vmin = np.nanmin(all_results_meth_temp["paired_or"].values)
norm = TwoSlopeNorm(vmin=min(vmin, 0), vcenter=1.0, vmax=max(vmax, 2.0))
MUTATIONS = ["snp", "mnp", "smalldel", "smallins", "ins", "del"]
all_results_meth_temp = all_results_meth_temp[all_results_meth_temp["dataset"] == "G4Hunter"]
fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={"hspace": 0.08})
for ax, rarity in zip(axes, ["low_frequency", "common"]):
    mat = np.array([
        [all_results_meth_temp.query("frequency_class==@rarity and mutation==@m and methylation_level==@ml")["paired_or"].values[0]
         for m in MUTATIONS]
        for ml in METH_LEVELS
    ], dtype=float)

    is_bottom = (rarity == "common")
    df_mat = pd.DataFrame(mat, index=METH_LEVELS, columns=[MUT_LABELS[m] for m in MUTATIONS])
    sns.heatmap(
        df_mat, ax=ax,
        cmap="PuOr", 
        norm=norm,
        linewidths=0.5, linecolor="white",
        xticklabels=is_bottom, yticklabels=True,
        cbar=False, annot=True, fmt=".2f",
        annot_kws={"size": FS},
    )

    ax.set_ylabel(rarity.capitalize().replace("Rare", "Low Frequency").replace("Low_frequency", "Low Frequency"), fontsize=FS + 2)
    ax.yaxis.label.set_bbox(dict(boxstyle="round,pad=0.45", facecolor="#f0f0f0", edgecolor="#333333", lw=1.5))
    ax.yaxis.set_label_coords(-0.4, 0.5)
    ax.tick_params(axis="y", labelsize=FS)
    if is_bottom:
        ax.tick_params(axis="x", labelsize=FS, rotation=45)
        plt.setp(ax.get_xticklabels(), ha="right")

cbar = fig.colorbar(axes[-1].collections[0], ax=axes, fraction=0.02, pad=0.02)
cbar.set_label("Matched Odds Ratio", fontsize=FS + 6)
cbar.ax.tick_params(labelsize=FS + 4)

target = Path("/work/10904/nikolchanchan/vista/figures_g4_t2t_NEW")
fig.savefig(f"{target}/mut_enrichment_meth_heatmap_{tissue}.paired_or.pdf", bbox_inches="tight", transparent=True)
fig.savefig(f"{target}/mut_enrichment_meth_heatmap_{tissue}.paired_or.png", bbox_inches="tight", dpi=300, transparent=True)
plt.show()

In [ ]:
f"{target}/mut_enrichment_meth_heatmap_{tissue}.paired_or.png"

#### Methylation Analysis?

In [ ]:

methylation_HG002_df = pd.read_table(Path(os.getenv("SCRATCH")).joinpath("g4_t2t_revisions_data").joinpath("chm13v2.0_hg002_CpG_ont_guppy6.1.2.bedgraph.collapsed.gz"))
# methylation_HG002_bed = BedTool.from_dataframe(methylation_HG002_df).sort()

methylation_CHM13v2_df = pd.read_table("/scratch/10904/nikolchanchan/g4_t2t_revisions_data/chm13v2.0_CHM13_CpG_ont_guppy3.6.0_nanopolish0.13.2.bedgraph",
                                       header=None,
                                       names=["Chromosome", "Start", "End", "methylation_level"]
                                       )
methylation_CHM13v2_df.head()

In [ ]:
meth_matched = pd.read_table("/scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_rank_HG002_G4Hunter_filtered_with_meth.cpg_strict.no_rank.filtered.tsv.gz")
meth_matched

In [ ]:
mut_df["AF_samples"] = mut_df["AC_samples"] / mut_df["AN_samples"]
rare_df = mut_df[(mut_df["AF_samples"] < RARE_THRESHOLD) & (mut_df["AN_samples"] >= 73)].drop_duplicates(subset=["#CHROM", "start"]).reset_index(drop=True)
common_df = mut_df[(mut_df["AF_samples"] >= RARE_THRESHOLD) & (mut_df["AN_samples"] >= 73)].drop_duplicates(subset=["#CHROM", "start"]).reset_index(drop=True)


In [ ]:
RARITY_SNVs = dict()
RARITY_COLLECTIONS = {
    "low_frequency":   rare_df_filtered,
    "common": common_df_filtered,
    "all":   mut_df,
}
for rarity, df in RARITY_COLLECTIONS.items():
    temp_df = df[df["mut_type"] == "snp"].copy().rename(columns={"#CHROM": "seqID"}).drop_duplicates(subset=["seqID", "start"]).reset_index(drop=True)
    RARITY_SNVs[rarity] = temp_df

In [ ]:
STATES = ["Hypomethylated", "Methylated", "Hypermethylated"]
for rarity, df in RARITY_SNVs.items():
    for STATE in STATES:
        subset = meth_matched[meth_matched["methylation_state"] == STATE]
        g4_df = subset[["seqID", "start", "end"]]
        ctrl_df = subset[["control_seqID", "control_start", "control_end"]].rename(columns={
            "control_seqID": "seqID",
            "control_start": "start",
            "control_end":   "end",
        })
        print(f"{STATE}: {len(subset)} CpGs, {subset['overlaps_g4'].mean():.2%} overlap G4s")

In [ ]:
# ── Cell 1 · CpG PyRanges with bedgraph classification ────────────────────
import pyranges as pr
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib
import matplotlib.patches as mpatches
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test
from statsmodels.stats.multitest import multipletests
from pathlib import Path

matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42
FIGS = Path("/scratch/10904/nikolchanchan/figures_g4_t2t"); FIGS.mkdir(exist_ok=True)

STATES      = ["Hypomethylated", "Methylated", "Hypermethylated"]
STATE_SHORT = {"Hypomethylated": "Hypo", "Methylated": "Meth", "Hypermethylated": "Hyper"}

def _classify(v):
    return "Hypomethylated" if v < 0.2 else ("Methylated" if v < 0.8 else "Hypermethylated")

hg002_cpg_pr = pr.PyRanges(
    methylation_HG002_df
    .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
    .assign(meth_state=lambda d: d["methylation_level"].apply(_classify))
    [["Chromosome", "Start", "End", "meth_state"]]
    .drop_duplicates(["Chromosome", "Start", "End"])
    .reset_index(drop=True)
)

chm13_cpg_pr = pr.PyRanges(
    methylation_CHM13v2_df
    .assign(meth_state=lambda d: d["methylation_level"].apply(_classify))
    [["Chromosome", "Start", "End", "meth_state"]]
    .drop_duplicates(["Chromosome", "Start", "End"])
    .reset_index(drop=True)
)

TISSUE_CPG = {"HG002": hg002_cpg_pr, "CHM13": chm13_cpg_pr}


In [ ]:
meth_matched = pd.read_table("/scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_rank_HG002_G4Hunter_filtered_with_meth.cpg_strict.no_rank.filtered.tsv.gz")
meth_matched_masked = pd.read_table("/scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_rank_HG002_G4Hunter_filtered_with_meth.cpg_strict.no_rank.filtered.__final__.tsv.gz")
CHM13_meth_matched_mask = pd.read_table("/scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_rank_CHM13_G4Hunter_filtered_with_meth.cpg_strict.no_rank.filtered.__final__.tsv.gz")


meth_df = {
          "HG002": meth_matched_masked, 
           # "HG002_unmasked": meth_matched,
           "CHM13": CHM13_meth_matched_mask
           }


In [ ]:
g4_df = pd.read_table(G4HUNTER)
g4_pr = pr.PyRanges(
    g4_df[["seqID", "start", "end"]]
    .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
    .drop_duplicates(["Chromosome", "Start", "End"])
    .reset_index(drop=True)
).merge()

# matched
matched_g4_pr = pr.PyRanges(
    meth_matched[["seqID", "start", "end"]]
    .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
    .drop_duplicates(["Chromosome", "Start", "End"])
    .reset_index(drop=True)
)

# matched ctrl
matched_ctrl_pr = pr.PyRanges(
    meth_matched[["control_seqID", "control_start", "control_end"]]
    .rename(columns={
        "control_seqID": "Chromosome",
        "control_start": "Start",
        "control_end":   "End",
    })
    .drop_duplicates(["Chromosome", "Start", "End"])
    .reset_index(drop=True)
)


In [ ]:
LEVELS = ["Hypomethylated", "Methylated", "Hypermethylated"]

mut_cpg_dict = dict()
for tissue, meth_pr in TISSUE_CPG.items():
    for frequency, mut_df in RARITY_SNVs.items():
        mut_pr = pr.PyRanges(
            mut_df.rename(columns={"seqID":"Chromosome","start":"Start","end":"End"})
            [["Chromosome","Start","End"]].drop_duplicates()
        )

        cpgs_with_g4s = (
            meth_pr.join(g4_pr, how=None).df[["Chromosome","Start","End"]].drop_duplicates()
            .assign(overlaps_g4=True)
        )

        # inner join → positions that have a mutation
        mutated_pos = (
            meth_pr.join(mut_pr, how=None).df
            [["Chromosome","Start","End"]].drop_duplicates()
            .assign(is_mutated=True)
        )
        
        # merge back onto full CpG set to get the is_mutated flag
        hg002_cpg_mutated = (
            meth_pr.df
            .merge(mutated_pos, on=["Chromosome","Start","End"], how="left")
            .assign(is_mutated=lambda d: d["is_mutated"].fillna(False))
            .merge(cpgs_with_g4s, on=["Chromosome","Start","End"], how="left")
            .assign(overlaps_g4=lambda d: d["overlaps_g4"].fillna(False))
        )
        mut_cpg_dict[tissue, frequency] = hg002_cpg_mutated


In [ ]:
from scipy.stats import binomtest
records = []
for tissue, frequency in mut_cpg_dict.keys():
    temp_df = mut_cpg_dict[tissue, frequency]
    background = (
        temp_df.groupby("meth_state")["is_mutated"]
        .agg(["sum","count"])
        .assign(rate=lambda d: d["sum"] / d["count"])
    )

    for state in STATES:
        g4 = temp_df[
            temp_df["overlaps_g4"] & (temp_df["meth_state"] == state)
        ]["is_mutated"]

        bg_rate = background.loc[state, "rate"]
        k = int(g4.sum())
        n = len(g4)

        result = binomtest(k, n, p=bg_rate, alternative="two-sided")
        records.append(dict(
            tissue=tissue,
            meth_state=state, 
            frequency=frequency,
            g4_rate=k/n, 
            background_rate=bg_rate,
            fold_enrichment=(k/n) / bg_rate,
            k=k, 
            n=n, 
            pval=result.pvalue,
        ))

res = pd.DataFrame(records)
_, res["qval"], _, _ = multipletests(res["pval"], method="fdr_bh")
res["stars"] = res["qval"].apply(sig_stars)
res

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test
from statsmodels.stats.multitest import multipletests

def yield_records(paired_rates):
    records = []
    for state in STATES:
        sub = paired_rates[paired_rates["meth_state"] == state].dropna(
            subset=["g4_n_mutated", "ctrl_n_mutated", "g4_mut_rate", "ctrl_mut_rate"]
        )

        g4_hit   = sub["g4_n_mutated"]   > 0
        ctrl_hit = sub["ctrl_n_mutated"] > 0

        a = int(( g4_hit &  ctrl_hit).sum())
        b = int(( g4_hit & ~ctrl_hit).sum())
        c = int((~g4_hit &  ctrl_hit).sum())
        d = int((~g4_hit & ~ctrl_hit).sum())

        mn = mcnemar_test([[a, b], [c, d]], exact=False)

        # Cliff's delta (paired): (n_g4>ctrl - n_g4<ctrl) / n_pairs
        diff    = sub["g4_mut_rate"] - sub["ctrl_mut_rate"]
        n       = len(diff)
        cliffs  = ((diff > 0).sum() - (diff < 0).sum()) / n if n > 0 else np.nan

        records.append(dict(
            meth_state=state, n_pairs=len(sub),
            a=a, b=b, c=c, d=d,
            paired_or=b/c if c > 0 else np.nan,
            log2_or=np.log2(b/c) if c > 0 and b > 0 else np.nan,
            pval=mn.pvalue,
            cliffs_delta=cliffs,
        ))
    return records

def cpg_rate_per_region(cpg_pr, region_pr, mut_pos_df):
    """Mutation rate of fully-contained CpGs, grouped per region interval."""
    joined = (
        cpg_pr.join(region_pr, how=None).df
        .query("Start >= Start_b and End <= End_b")
        .rename(columns={"Start_b": "region_start", "End_b": "region_end"})
        [["Chromosome", "Start", "End", "meth_state", "region_start", "region_end"]]
        .drop_duplicates(["Chromosome", "Start", "End", "region_start", "region_end"])
    )
    # flag mutated CpGs
    joined = (joined
              .merge(mut_pos_df, on=["Chromosome","Start","End"], how="left")
              .assign(is_mutated=lambda d: d["is_mutated"].fillna(False)))

    # rate per region interval
    return (joined
            .groupby(["Chromosome","region_start","region_end","meth_state"])
            .agg(n_cpg=("is_mutated","count"),
                 n_mutated=("is_mutated","sum"))
            .assign(mut_rate=lambda d: d["n_mutated"] / d["n_cpg"])
            .reset_index())

records = []
print(meth_matched.shape)
meth_matched_temp = meth_matched[meth_matched["cpg"] == meth_matched["control_cpg"]]
print(meth_matched_temp.shape)
for tissue, meth_pr in TISSUE_CPG.items():
    for frequency, mut_df in RARITY_SNVs.items():
        mut_pr = pr.PyRanges(
            mut_df.rename(columns={"seqID":"Chromosome","start":"Start","end":"End"})
            [["Chromosome","Start","End"]].drop_duplicates()
        )
        mutated_pos = (
            hg002_cpg_pr.join(mut_pr, how=None).df
            [["Chromosome","Start","End"]].drop_duplicates()
            .assign(is_mutated=True)
        )

        g4_rates   = cpg_rate_per_region(hg002_cpg_pr, matched_g4_pr,   mutated_pos)
        ctrl_rates = cpg_rate_per_region(hg002_cpg_pr, matched_ctrl_pr, mutated_pos)

        paired_rates = (
            meth_matched
            .merge(
                g4_rates.rename(columns={
                    "Chromosome":"seqID", "region_start":"start", "region_end":"end",
                    "mut_rate":"g4_mut_rate", "n_cpg":"g4_n_cpg", "n_mutated":"g4_n_mutated"
                }),
                on=["seqID","start","end"], how="left"
            )
            .merge(
                ctrl_rates.rename(columns={
                    "Chromosome":"control_seqID", "region_start":"control_start", "region_end":"control_end",
                    "mut_rate":"ctrl_mut_rate", "n_cpg":"ctrl_n_cpg", "n_mutated":"ctrl_n_mutated"
                }),
                on=["control_seqID","control_start","control_end","meth_state"],  # join on meth_state too
                how="left"
            )
        )
        data = yield_records(paired_rates)
        for entry in data:
            entry["frequency"] = frequency
            entry["tissue"] = tissue
        records.extend(data)

mc_res = pd.DataFrame(records)
_, mc_res["qval"], _, _ = multipletests(mc_res["pval"], method="fdr_bh")
mc_res["stars"] = mc_res["qval"].apply(sig_stars)
mc_res

In [ ]:
# control regions that have NO CpG from the HG002 bedgraph
ctrl_with_cpg = hg002_cpg_pr.join(matched_ctrl_pr, how=None).df[["Start_b","End_b","Chromosome"]]\
                .drop_duplicates().rename(columns={"Start_b":"Start","End_b":"End"})

ctrl_all = matched_ctrl_pr.df[["Chromosome","Start","End"]].drop_duplicates()

no_cpg = ctrl_all.merge(ctrl_with_cpg, on=["Chromosome","Start","End"], how="left", indicator=True)\
                 .query("_merge == 'left_only'").drop(columns="_merge")

print(f"Ctrl regions with no HG002 CpG: {len(no_cpg)} / {len(ctrl_all)}")
print(f"Fraction: {len(no_cpg)/len(ctrl_all):.2%}")



In [ ]:
paired_rates.shape, ctrl_rates.shape, g4_rates.shape

In [ ]:
ctrl_rates[ctrl_rates["mut_rate"].isna()]

In [ ]:
paired_rates.dropna(subset=["g4_mut_rate", "ctrl_mut_rate"])

In [ ]:
CpG_ctrl_mut.groupby(["meth_state", "is_mutated"]).size().unstack(fill_value=0)

In [ ]:
CpG_g4_mut.groupby(["meth_state", "is_mutated"]).size().unstack(fill_value=0)

In [ ]:
# meth_matched_df = pd.read_table("/scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_rank_HG002_G4Hunter_filtered_with_meth.cpg_strict.no_rank.filtered.tsv.gz")
g4_pr = pr.PyRanges(
    meth_matched_df[["seqID", "start", "end", "methylation_level"]].drop_duplicates()
    .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
)
ctrl_pr = pr.PyRanges(
    meth_matched_df[["control_seqID", "control_start", "control_end", "methylation_level"]].drop_duplicates()
    .rename(columns={
        "control_seqID": "Chromosome",
        "control_start": "Start",
        "control_end":   "End",
    })
)
CpG_g4_mut, CpG_ctrl_mut = CpG_mut_data["all", "HG002"]
joined_cpg = pr.PyRanges(CpG_g4_mut).join(g4_pr, how=None).df.query("Start >= Start_b and End <= End_b")\
                .groupby(["Chromosome", "Start_b", "End_b", "meth_state"], as_index=False)\
                .agg({"is_mutated": "mean"})
joined_cpg_ctrl = pr.PyRanges(CpG_ctrl_mut).join(ctrl_pr, how=None).df.query("Start >= Start_b and End <= End_b")\
                .groupby(["Chromosome", "Start_b", "End_b", "meth_state"], as_index=False)\
                .agg({"is_mutated": "mean"})

matched_joined_cpg = joined_cpg.merge(
                meth_matched_df,
                left_on=["Chromosome", "Start_b", "End_b", "meth_state"],
                right_on=["seqID", "start", "end", "methylation_level"]
            )\
            .merge(
                joined_cpg_ctrl,
                left_on=["control_seqID", "control_start", "control_end", "control_methylation_level"],
                right_on=["Chromosome", "Start_b", "End_b", "meth_state"],
                suffixes=("_g4", "_ctrl")
            )

joined_cpg

In [ ]:
matched_joined_cpg = joined_cpg.merge(
                meth_matched_df,
                left_on=["Chromosome", "Start_b", "End_b", "meth_state"],
                right_on=["seqID", "start", "end", "methylation_level"],
            )\
            .merge(
                joined_cpg_ctrl,
                left_on=["control_seqID", "control_start", "control_end", "control_methylation_level"],
                right_on=["Chromosome", "Start_b", "End_b", "meth_state"],
                suffixes=("_g4", "_ctrl")
            )

matched_joined_cpg

In [ ]:
from scipy.stats import wilcoxon
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test
from statsmodels.stats.multitest import multipletests

STATES = ["Hypomethylated", "Methylated", "Hypermethylated"]

records = []
for state in STATES:
    sub = matched_joined_cpg[matched_joined_cpg["meth_state_g4"] == state].dropna(
        subset=["is_mutated_g4", "is_mutated_ctrl"]
    )
    if len(sub) < 10:
        continue

    g4   = sub["is_mutated_g4"].astype(float)
    ctrl = sub["is_mutated_ctrl"].astype(float)

    # Wilcoxon signed-rank on paired differences
    diff = g4 - ctrl
    if (diff == 0).all():
        wx_stat, wx_p = np.nan, np.nan
    else:
        wx_stat, wx_p = wilcoxon(g4, ctrl, alternative="two-sided")

    # McNemar — binarise
    g4_hit   = g4   > 0
    ctrl_hit = ctrl > 0
    a = int(( g4_hit &  ctrl_hit).sum())
    b = int(( g4_hit & ~ctrl_hit).sum())
    c = int((~g4_hit &  ctrl_hit).sum())
    d = int((~g4_hit & ~ctrl_hit).sum())
    mn      = mcnemar_test([[a, b], [c, d]], exact=False)
    paired_or = b / c if c > 0 else np.nan

    records.append(dict(
        meth_state=state, n=len(sub),
        g4_mean=g4.mean(), ctrl_mean=ctrl.mean(),
        wilcoxon_stat=wx_stat, pval_wilcoxon=wx_p,
        a=a, b=b, c=c, d=d,
        paired_or=paired_or,
        log2_or=np.log2(paired_or) if paired_or and paired_or > 0 else np.nan,
        pval_mcnemar=mn.pvalue,
    ))

res = pd.DataFrame(records)

# FDR
for col in ["pval_wilcoxon", "pval_mcnemar"]:
    mask = res[col].notna()
    if mask.any():
        _, q, _, _ = multipletests(res.loc[mask, col], method="fdr_bh")
        res.loc[mask, col.replace("pval", "qval")] = q

res["stars_wx"] = res["qval_wilcoxon"].apply(sig_stars)
res["stars_mn"] = res["qval_mcnemar"].apply(sig_stars)
res


In [ ]:
# ── Cell 4 · barplot ───────────────────────────────────────────────────────
def sig_stars(q):
    if q < 0.001: return "***"
    if q < 0.01:  return "**"
    if q < 0.05:  return "*"
    return ""

TISSUE_COLORS = {"HG002": "#2166ac", "CHM13": "#d6604d"}
TISSUES       = ["HG002", "CHM13"]
STATE_LABELS  = {"Hypomethylated": "Hypo", "Methylated": "Meth", "Hypermethylated": "Hyper"}
x             = np.arange(len(STATES))
bar_w         = 0.35

rarity_keys = list(RARITY_SNVs.keys())
fig, axes = plt.subplots(1, len(rarity_keys),
                         figsize=(5.5 * len(rarity_keys), 5.5),
                         sharey=True)
if len(rarity_keys) == 1:
    axes = [axes]

for ax, rarity in zip(axes, rarity_keys):
    sub = results_cpg[results_cpg["rarity"] == rarity]
    for i, tissue in enumerate(TISSUES):
        ts     = sub[sub["tissue"] == tissue].set_index("meth_state")
        vals   = [ts.loc[s, "log2_fe"]     if s in ts.index else np.nan for s in STATES]
        qvals  = [ts.loc[s, "qval_fisher"] if s in ts.index else 1      for s in STATES]
        offset = (i - 0.5) * bar_w
        ax.bar(x + offset, vals, width=bar_w, color=TISSUE_COLORS[tissue],
               label=tissue, edgecolor="white", linewidth=0.5, alpha=0.88)
        for xi, (v, q) in enumerate(zip(vals, qvals)):
            if not np.isnan(v) and q < 0.05:
                ax.text(xi + offset, v + (0.03 if v >= 0 else -0.1),
                        sig_stars(q), ha="center", va="bottom", fontsize=11)

    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([STATE_LABELS[s] for s in STATES], fontsize=14)
    ax.set_xlabel("G4 methylation state", fontsize=15)
    ax.set_title(rarity, fontsize=14)
    ax.grid(axis="y", lw=0.4, alpha=0.6)
    ax.tick_params(axis="y", labelsize=13)

axes[0].set_ylabel("log$_2$ FE (mutated CpGs: G4 vs control)", fontsize=14)
legend_patches = [mpatches.Patch(color=TISSUE_COLORS[t], label=t) for t in TISSUES]
fig.legend(handles=legend_patches, loc="upper right",
           bbox_to_anchor=(1.0, 1.02), fontsize=12, title="Sample")
fig.tight_layout()
fig.savefig(FIGS / "cpg_mutation_enrichment_by_meth_state.pdf",
            bbox_inches="tight", transparent=True)
plt.show()


In [ ]:
meth_matched_masked = pd.read_table("/scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_rank_HG002_G4Hunter_filtered_with_meth.cpg_strict.no_rank.filtered.__final__.tsv.gz")
CHM13_meth_matched_mask = pd.read_table("/scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_rank_CHM13_G4Hunter_filtered_with_meth.cpg_strict.no_rank.filtered.__final__.tsv.gz")
CHM13_meth_matched_mask

meth_df = {"HG002": meth_matched_masked, "CHM13": CHM13_meth_matched_mask}

In [ ]:
mut_df["AF_samples"] = mut_df["AC_samples"] / mut_df["AN_samples"]
RARE_THRESHOLD = 0.05
rare_df_filtered = mut_df[(mut_df["AF_samples"] < RARE_THRESHOLD) & (mut_df["AN_samples"] >= 73)].drop_duplicates(subset=["#CHROM", "start"]).reset_index(drop=True)
common_df_filtered = mut_df[(mut_df["AF_samples"] >= RARE_THRESHOLD) & (mut_df["AN_samples"] >= 73)].drop_duplicates(subset=["#CHROM", "start"]).reset_index(drop=True)
RARITY_COLLECTIONS = {"all": mut_df, "rare": rare_df_filtered, "common": common_df_filtered}
METH_LEVELS = ["Hypomethylated", "Methylated", "Hypermethylated"]
all_results_meth = []

datasets_meth = []
for tissue in meth_df:
    for LEVEL in METH_LEVELS:
        base_df = meth_df[tissue][meth_df[tissue]["methylation_level"] == LEVEL]
        g4_bed = BedTool.from_dataframe(base_df[["seqID", "start", "end"]]).sort()
        ctrl_bed = BedTool.from_dataframe(
            base_df[["control_seqID", "control_start", "control_end"]].rename(columns={
                "control_seqID": "seqID",
                "control_start": "start",
            "control_end":   "end",
        })
        ).sort()
        datasets_meth.append((LEVEL, tissue, g4_bed, ctrl_bed, base_df))
ds_name = "G4Hunter"
mutations = mut_df["mut_type"].unique().tolist()
for (LEVEL, tissue, g4_bed, ctrl_bed, df_temp) in datasets_meth:
    for rarity, mut_dict in RARITY_COLLECTIONS.items():
        df = compute_enrichment(mut_dict, 
                                label=rarity, 
                                g4_bed=g4_bed, 
                                ctrl_bed=ctrl_bed, 
                                ds_name=ds_name, 
                                matched_df=df_temp)
        df["dataset"] = LEVEL
        df["tissue"] = tissue
        all_results_meth.append(df)

all_results_meth = pd.concat(all_results_meth, ignore_index=True)
all_results_meth


In [ ]:
all_results_meth.to_csv(target / "mutation_g4_enrichment_results_by_methylation.tsv.gz", sep="\t", index=False, compression="gzip")

In [ ]:
target / "mutation_g4_enrichment_results_by_methylation.tsv.gz"

In [ ]:
for pval_col, qval_col in [
    ("pval_fisher",   "qval_fisher"),
   #  ("pval_wx",       "qval_wx"),
    ("mpval_fisher",  "mqval_fisher"),
    ("pval_mcnemar", "qval_mcnemar"),

]:
    _, qvals, _, _ = multipletests(all_results_meth[pval_col], method="fdr_bh")
    all_results_meth[qval_col] = qvals

all_results_meth

In [ ]:
df = (
    all_results_meth[all_results_meth["frequency_class"] == "common"]
    .query("mutation == 'snp'")
    [["dataset", "g4_density", "ctrl_density","odds", "rel_fold_enrichment", "modds", "qval_mcnemar", "paired_or", "qval_fisher", "tissue"]]
)
with pd.option_context("display.float_format", "{:.4f}".format):
    print(df.to_string())


In [ ]:
df = (
    all_results_meth[all_results_meth["frequency_class"] == "rare"]
    .query("mutation == 'snp'")
    [["dataset", "g4_density", "ctrl_density","odds", "rel_fold_enrichment", "modds", "qval_mcnemar", "paired_or", "qval_fisher", "tissue"]]
)
with pd.option_context("display.float_format", "{:.4f}".format):
    print(df.to_string())


In [ ]:
from statsmodels.stats.proportion import proportions_ztest

rows = []
for sample in SAMPLES:
    for db in ["G4Hunter", "Quadparser", "eG4"]:
        sub = locus_df[
            (locus_df["sample"] == sample) &
            (locus_df["database"] == db)
        ]
        # genome-wide background fraction
        p_genome = (sub["meth_cat"] == "Hypomethylated").mean()

        for rank in RANKS:
            r = sub[sub["rank"] == rank]
            n_hypo  = (r["meth_cat"] == "Hypomethylated").sum()
            n_total = len(r)

            stat, p = proportions_ztest(count=n_hypo, nobs=n_total, value=p_genome)
            rows.append({
                "sample":   sample,
                "database": db,
                "rank":     rank,
                "n_total":  n_total,
                "n_hypo":   n_hypo,
                "pct_hypo": 100 * n_hypo / n_total,
                "p_genome": round(p_genome, 4),
                "z":        round(stat, 3),
                "p":        p,
            })

prop_df = pd.DataFrame(rows)
prop_df["sig"] = prop_df["p"].apply(
    lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
)
prop_df["enriched"] = prop_df.apply(
    lambda r: "enriched" if r["pct_hypo"] / 100 > r["p_genome"] else "depleted", axis=1
)
print(prop_df.to_string(index=False))



In [ ]:
from matplotlib.colors import TwoSlopeNorm

tissue = "HG002"
FS = 18
all_results_meth_temp = all_results_meth[all_results_meth["tissue"] == tissue]
vmax = np.nanmax(all_results_meth_temp["paired_or"].values)
vmin = np.nanmin(all_results_meth_temp["paired_or"].values)
norm = TwoSlopeNorm(vmin=min(vmin, 0), vcenter=1.0, vmax=max(vmax, 2.0))
MUTATIONS = ["snp", "mnp", "smalldel", "smallins", "ins", "del"]

fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={"hspace": 0.08})
for ax, rarity in zip(axes, ["rare", "common"]):
    mat = np.array([
        [all_results_meth_temp.query("dataset==@ml and frequency_class==@rarity and mutation==@m")["paired_or"].values[0]
         for m in MUTATIONS]
        for ml in METH_LEVELS
    ], dtype=float)

    is_bottom = (rarity == "common")
    df_mat = pd.DataFrame(mat, index=METH_LEVELS, columns=[MUT_LABELS[m] for m in MUTATIONS])
    sns.heatmap(
        df_mat, ax=ax,
        cmap="PuOr", 
        norm=norm,
        linewidths=0.5, linecolor="white",
        xticklabels=is_bottom, yticklabels=True,
        cbar=False, annot=True, fmt=".2f",
        annot_kws={"size": FS},
    )

    ax.set_ylabel(rarity.capitalize().replace("Rare", "Low Frequency"), fontsize=FS + 2)
    ax.yaxis.label.set_bbox(dict(boxstyle="round,pad=0.45", facecolor="#f0f0f0", edgecolor="#333333", lw=1.5))
    ax.yaxis.set_label_coords(-0.4, 0.5)
    ax.tick_params(axis="y", labelsize=FS)
    if is_bottom:
        ax.tick_params(axis="x", labelsize=FS, rotation=45)
        plt.setp(ax.get_xticklabels(), ha="right")

cbar = fig.colorbar(axes[-1].collections[0], ax=axes, fraction=0.02, pad=0.02)
cbar.set_label("Matched Odds Ratio", fontsize=FS + 6)
cbar.ax.tick_params(labelsize=FS + 4)

target = Path("/work/10904/nikolchanchan/vista/figures_g4_t2t_NEW")
fig.savefig(f"{target}/mut_enrichment_meth_heatmap_{tissue}.paired_or.pdf", bbox_inches="tight", transparent=True)
fig.savefig(f"{target}/mut_enrichment_meth_heatmap_{tissue}.paired_or.png", bbox_inches="tight", dpi=300, transparent=True)
plt.show()

In [ ]:
from matplotlib.colors import TwoSlopeNorm

tissue = "HG002"
FS = 18
all_results_meth_temp = all_results_meth[all_results_meth["tissue"] == tissue]
vmax = np.nanmax(all_results_meth_temp["paired_or"].values)
vmin = np.nanmin(all_results_meth_temp["paired_or"].values)
norm = TwoSlopeNorm(vmin=min(vmin, 0), vcenter=1.0, vmax=max(vmax, 2.0))
MUTATIONS = ["snp", "mnp", "smalldel", "smallins", "ins", "del"]

fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={"hspace": 0.08})
for ax, rarity in zip(axes, ["rare", "common"]):
    mat = np.array([
        [all_results_meth_temp.query("dataset==@ml and frequency_class==@rarity and mutation==@m")["paired_or"].values[0]
         for m in MUTATIONS]
        for ml in METH_LEVELS
    ], dtype=float)

    is_bottom = (rarity == "common")
    df_mat = pd.DataFrame(mat, index=METH_LEVELS, columns=[MUT_LABELS[m] for m in MUTATIONS])
    sns.heatmap(
        df_mat, ax=ax,
        cmap="PuOr", 
        norm=norm,
        linewidths=0.5, linecolor="white",
        xticklabels=is_bottom, yticklabels=True,
        cbar=False, annot=True, fmt=".2f",
        annot_kws={"size": FS},
    )

    ax.set_ylabel(rarity.capitalize().replace("Rare", "Low Frequency"), fontsize=FS + 2)
    ax.yaxis.label.set_bbox(dict(boxstyle="round,pad=0.45", facecolor="#f0f0f0", edgecolor="#333333", lw=1.5))
    ax.yaxis.set_label_coords(-0.4, 0.5)
    ax.tick_params(axis="y", labelsize=FS)
    if is_bottom:
        ax.tick_params(axis="x", labelsize=FS, rotation=45)
        plt.setp(ax.get_xticklabels(), ha="right")

cbar = fig.colorbar(axes[-1].collections[0], ax=axes, fraction=0.02, pad=0.02)
cbar.set_label("Matched Odds Ratio", fontsize=FS + 6)
cbar.ax.tick_params(labelsize=FS + 4)

fig.savefig(f"{target_fig}/mut_enrichment_meth_heatmap_{tissue}.paired_or.pdf", bbox_inches="tight", transparent=True)
fig.savefig(f"{target_fig}/mut_enrichment_meth_heatmap_{tissue}.paired_or.png", bbox_inches="tight", dpi=300, transparent=True)
plt.show()

In [ ]:
meth_matched = pd.read_table("/scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_rank_HG002_G4Hunter_filtered_with_meth.cpg_strict.no_rank.filtered.tsv.gz")
meth_matched.loc[:, "sequence"] = meth_matched["sequence"].str.upper()

# unmasked
meth_matched_unmasked = pd.read_table("/scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_rank_HG002_G4Hunter_with_meth.cpg_strict.no_rank.tsv.gz")
meth_matched_unmasked = meth_matched_unmasked[meth_matched_unmasked["gc_content"] == meth_matched_unmasked["control_gc_content"]].reset_index(drop=True)
meth_matched_unmasked.loc[:, "sequence"] = meth_matched_unmasked["sequence"].str.upper()
meth_matched_unmasked

print(meth_matched.shape, meth_matched_unmasked.shape)
# datasets
meth_db = {
           "masked": meth_matched, 
           "unmasked": meth_matched_unmasked
    }

In [ ]:
FIELDS = ["#CHROM", "start", "end"]
SNV_df = {
          "Rare": BedTool.from_dataframe(rare_df_filtered[rare_df_filtered["mut_type"] == "snp"][FIELDS].drop_duplicates()).sort(),
          "Common": BedTool.from_dataframe(common_df_filtered[common_df_filtered["mut_type"] == "snp"][FIELDS].drop_duplicates()).sort()
          }
SNV_df

genome_size = sum(len(seq) for seq in chrom_sequences.values())

def count_bp(bed):
    df = bed.to_dataframe(names=["chrom", "start", "end"])
    return int((df["end"] - df["start"]).sum())

gw_density = {
    rarity: count_bp(bed) / genome_size
    for rarity, bed in SNV_df.items()
}
gw_density

In [ ]:
def p_to_star(p):
    if p < 1e-4:  return "****"
    if p < 1e-3:  return "***"
    if p < 1e-2:  return "**"
    if p < 0.05:  return "*"
    return "ns"

_, fdr_vals_fisher, _, _ = multipletests(burden_meth_df["p_fisher"], method="fdr_bh")
burden_meth_df.loc[:, "pval_fdr_fisher"] = fdr_vals_fisher
fdr_stars_fisher = [p_to_star(p) for p in fdr_vals_fisher]
burden_meth_df.loc[:, "stars_fdr_fisher"] = fdr_stars_fisher

# #
burden_meth_df.loc[:, "rate"] = burden_meth_df["observed_bp"] / burden_meth_df["expected_g4"]
burden_meth_df.loc[:, "ctrl_rate"] = burden_meth_df["observed_bp_ctrl"] / burden_meth_df["expected_ctrl_gw"]
plot_meth_df = pd.concat([
    all_results_meth[["rarity", "meth_level", "variant_density_g4", "stars_fdr_fisher"]]
        .rename(columns={"variant_density_g4": "variant_density",
                                "stars_fdr_fisher":   "stars_fdr"})
                .assign(region="G4"),
    all_results_meth[["rarity", "meth_level", "variant_density_ctrl", "stars_fdr_fisher"]]
        .rename(columns={"variant_density_ctrl": "variant_density",
                                "stars_fdr_fisher":   "stars_fdr"})
        .assign(region="Control"),
], ignore_index=True)

plot_meth_df["variant_density"] = plot_meth_df["variant_density"] * 1000
burden_meth_df


In [ ]:
all_results_meth

In [ ]:
all_results_meth.loc[:, "stars_fdr_fisher"] = all_results_meth["qval_fisher"].apply(p_to_star)
all_results_meth.loc[:, "stars_fdr_wx"] = all_results_meth["qval_wx"].apply(p_to_star)
all_results_meth.loc[:, "stars_fdr_mfisher"] = all_results_meth["mqval_fisher"].apply(p_to_star)

In [ ]:
plot_meth_df = pd.concat([
    all_results_meth[["tissue", "mutation", "frequency_class", "dataset", "g4_density", "stars_fdr_fisher"]]
        .rename(columns={"g4_density": "variant_density",
                                "stars_fdr_fisher":   "stars_fdr"})
                .assign(region="G4"),
    all_results_meth[["tissue", "mutation","frequency_class", "dataset", "ctrl_density", "stars_fdr_fisher"]]
        .rename(columns={"ctrl_density": "variant_density",
                                "stars_fdr_fisher":   "stars_fdr"})
        .assign(region="Control"),
], ignore_index=True).query("frequency_class != 'all' & mutation == 'snp'")

plot_meth_df["variant_density"] = plot_meth_df["variant_density"] * 1000
plot_meth_df = plot_meth_df.rename(columns={"dataset": "meth_level"})
plot_meth_df

In [ ]:
# all_meth_results["stars_fdr"] = all_meth_results["qval_mcnemar"].apply(p_to_star)

plot_meth_df = pd.concat([
    all_meth_results[["tissue", "mutation", "frequency_class", "methylation_level", "g4_density", "qval_mcnemar"]]
        .rename(columns={"g4_density": "variant_density",
                                "qval_mcnemar":   "stars_fdr"})
                .assign(region="G4"),
    all_meth_results[["tissue", "mutation","frequency_class", "methylation_level", "ctrl_density", "qval_mcnemar"]]
        .rename(columns={"ctrl_density": "variant_density",
                                "qval_mcnemar":   "stars_fdr"})
        .assign(region="Control"),
], ignore_index=True).query("frequency_class != 'all' & mutation == 'snp'")

plot_meth_df["variant_density"] = plot_meth_df["variant_density"] * 1000
plot_meth_df = plot_meth_df.rename(columns={"methylation_level": "meth_level"})
plot_meth_df

In [ ]:
all_meth_results["stars_fdr"] = all_meth_results["qval_mcnemar"].apply(p_to_star)

In [ ]:
REGION_PALETTE = {"G4": "#9A22F7", "Control": "#757575"}
REGION_ORDER   = ["G4", "Control"]
RARITY_ORDER   = ["low_frequency", "common"]
METH_ORDER     = ["Hypomethylated", "Methylated", "Hypermethylated"]
TISSUES        = ["HG002", "CHM13"]
bar_width      = 0.8 / len(REGION_ORDER)
global_ymax    = plot_meth_df["variant_density"].max()
# plot_meth_df["stars_fdr"] = plot_meth_df["stars_fdr"].apply(p_to_star)

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey="row",
                         gridspec_kw={"hspace": 0.2, "wspace": 0.08})

for row_i, tissue in enumerate(TISSUES):
    for col_i, rarity in enumerate(RARITY_ORDER):
        ax  = axes[row_i, col_i]
        sub = plot_meth_df[(plot_meth_df["frequency_class"] == rarity) & (plot_meth_df["tissue"] == tissue)]
        # McNemar data for this tissue/rarity/snp
        mc_sub = all_meth_results[
            (all_meth_results["tissue"] == tissue) &
            (all_meth_results["frequency_class"] == rarity) &
            (all_meth_results["mutation"] == "snp")
        ]

        sns.barplot(
            data=sub, x="meth_level", y="variant_density",
            hue="region", order=METH_ORDER, hue_order=REGION_ORDER,
            palette=REGION_PALETTE, edgecolor="black", lw=1.5,
            ax=ax, errorbar=None,
        )
        for patch in ax.containers[1]:
            patch.set_linestyle((0, (3, 3)))

        ax.set_xlabel("", fontsize=16)
        ax.tick_params(labelsize=17)
        ax.tick_params(axis="x", labelsize=20, labelrotation=15)
        ax.grid(axis="y", lw=0.4, alpha=0.6)
        ax.set_axisbelow(True)
        ax.set_ylim(0, global_ymax * 1.2)
        if row_i == 1:
            ax.legend(title="", fontsize=18, bbox_to_anchor=(0.5, -0.2))
        else:
            ax.legend(handles=[], frameon=False)
        ax.spines[["top", "right"]].set_visible(False)

        if row_i == 0:
            ax.set_xticklabels([])

        if row_i == 0:
            ax.text(0.5, 1.06,
                    rarity.replace("low_frequency", "Low Frequency").capitalize(),
                    transform=ax.transAxes,
                    ha="center", va="bottom", fontsize=22, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.4", facecolor="#f0f0f0",
                              edgecolor="#333333", lw=1.5))

        if col_i == 0:
            ax.set_ylabel("Variant Density (SNVs / kb)", fontsize=16)
            ax.text(-0.24, 0.5, tissue,
                    transform=ax.transAxes,
                    ha="center", va="center", fontsize=22, fontweight="bold",
                    rotation=90,
                    bbox=dict(boxstyle="round,pad=0.45", facecolor="#f0f0f0",
                              edgecolor="#333333", lw=1.5))
        else:
            ax.set_ylabel("")

        # significance brackets with McNemar stars + OR label
        g4_bars   = ax.containers[0]
        ctrl_bars = ax.containers[1]
        for i, meth in enumerate(METH_ORDER):
            mc_row = mc_sub[mc_sub["methylation_level"] == meth]
            if mc_row.empty:
                continue
            star   = mc_row["stars_fdr"].values[0]
            paired_or = mc_row["paired_or"].values[0]
            x_g4  = g4_bars[i].get_x()   + g4_bars[i].get_width()   / 2
            x_ctl = ctrl_bars[i].get_x() + ctrl_bars[i].get_width() / 2
            y_top = global_ymax * 1.08
            ax.plot([x_g4, x_g4, x_ctl, x_ctl],
                    [y_top - global_ymax * 0.02, y_top, y_top, y_top - global_ymax * 0.02],
                    color="black", lw=1)
            ax.text((x_g4 + x_ctl) / 2, y_top + global_ymax * 0.01,
                    f"{star}\nOR={paired_or:.2f}",
                    ha="center", va="bottom", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(target_fig, "burden_meth_g4_control.png"),
            dpi=300, transparent=True, bbox_inches="tight")
plt.savefig(os.path.join(target_fig, "burden_meth_g4_control.pdf"),
            transparent=True, bbox_inches="tight")
plt.show()


In [ ]:
os.path.join(target_fig, "burden_meth_g4_control.png")

In [ ]:
target_fig

In [ ]:
import re
import itertools
# # # #
loops_meth_df       = []
gruns_meth_df       = []
transitions_meth_df = []
for row in meth_matched.itertuples(index=False):
    seqID         = row.seqID
    start         = row.start
    end           = row.end
    sequence      = row.sequence.upper()
    meth_level    = row.methylation_level
    grun_char     = sequence[0]

    gruns = list(re.finditer(r"%s{3,}" % re.escape(grun_char), sequence))
    if len(gruns) < 2:
        continue

    gruns_meth_df.append({
        "seqID":            seqID,
        "start":            start,
        "end":              start + gruns[0].span()[1],
        "type":             "grun",
        "motif_start":      start,
        "motif_end":        end,
        "sequence":         sequence[:gruns[0].span()[1]],
        "methylation_level": meth_level,
    })

    for prev_grun, cur_grun in zip(gruns, gruns[1:]):
        g_start_prev, g_end_prev = prev_grun.span()
        g_start_cur,  g_end_cur  = cur_grun.span()

        loops_meth_df.append({
            "seqID":            seqID,
            "start":            start + g_end_prev,
            "end":              start + g_start_cur,
            "motif_start":      start,
            "motif_end":        end,
            "type":             "loop",
            "sequence":         sequence[g_end_prev:g_start_cur],
            "methylation_level": meth_level,
        })

        run_seq = sequence[g_start_cur:g_end_cur]
        gruns_meth_df.append({
            "seqID":            seqID,
            "start":            start + g_start_cur,
            "end":              start + g_end_cur,
            "motif_start":      start,
            "motif_end":        end,
            "type":             "grun",
            "sequence":         run_seq,
            "methylation_level": meth_level,
        })

        for offset in [g_end_prev, g_start_cur]:
            transitions_meth_df.append({
                "seqID":            seqID,
                "start":            start + offset - 1,
                "end":              start + offset + 1,
                "motif_start":      start,
                "motif_end":        end,
                "type":             "transition",
                "sequence":         sequence[offset - 1: offset + 1],
                "methylation_level": meth_level,
            })

gruns_meth_df       = pd.DataFrame(gruns_meth_df)
loops_meth_df       = pd.DataFrame(loops_meth_df)
transitions_meth_df = pd.DataFrame(transitions_meth_df)

METH_LEVELS  = ["Hypomethylated", "Methylated", "Hypermethylated"]
COV_FIELDS   = ["total_hits", "total_bases", "all_bases", "coverage"]
FIELDS_G4    = ["seqID", "start", "end"]

REGIONS_METH = [
    ("G4",         G4_meth_df),
    ("G-run",      gruns_meth_df),
    ("Loop",       loops_meth_df),
    ("Transition", transitions_meth_df),
]


In [ ]:
g4_df = pd.read_table(G4HUNTER)
regex_df = pd.read_table(REGEX)

In [ ]:
import re
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

def extract_subregions(df, dataset_name):
    """Extract G-runs, loops, transitions from a G4 dataframe with a 'sequence' column."""
    gruns, loops, transitions = [], [], []

    for row in df.itertuples(index=False):
        seqID    = row.seqID
        start    = row.start
        end      = row.end
        sequence = row.sequence.upper()
        grun_char = sequence[0]

        found = list(re.finditer(r"%s{3,}" % re.escape(grun_char), sequence))
        if len(found) < 2:
            continue

        # first G-run
        gruns.append({"seqID": seqID, "start": start,
                      "end": start + found[0].span()[1],
                      "dataset": dataset_name})

        for prev, cur in zip(found, found[1:]):
            p_s, p_e = prev.span()
            c_s, c_e = cur.span()

            # loop between consecutive G-runs
            loops.append({"seqID": seqID,
                          "start": start + p_e,
                          "end":   start + c_s,
                          "dataset": dataset_name})

            # subsequent G-run
            gruns.append({"seqID": seqID,
                          "start": start + c_s,
                          "end":   start + c_e,
                          "dataset": dataset_name})

            # transitions (G-run boundaries, ±1 bp)
            for offset in [p_e, c_s]:
                transitions.append({"seqID": seqID,
                                    "start": start + offset - 1,
                                    "end":   start + offset + 1,
                                    "dataset": dataset_name})

    return (pd.DataFrame(gruns),
            pd.DataFrame(loops),
            pd.DataFrame(transitions))


def snv_rate_per_subregion(region_df, rare_snv_df, common_snv_df):
    """Compute rare and common SNV density (per kb) + Fisher rare vs common."""
    cov_cols = ["seqID", "start", "end", "total_hits", "overlapping_bases", "all_bases", "coverage"]

    region_bed = BedTool.from_dataframe(region_df[["seqID", "start", "end"]]).sort().merge()

    def _coverage(mut_df):
        mut_b = BedTool.from_dataframe(
            mut_df.drop_duplicates(subset=["#CHROM", "start"])[["#CHROM", "start", "end"]]
        ).sort().merge()
        return pd.read_table(region_bed.coverage(mut_b).fn, header=None, names=cov_cols)

    rare_cov   = _coverage(rare_snv_df)
    common_cov = _coverage(common_snv_df)

    total_bases = rare_cov["all_bases"].sum()   # same for both (same region bed)

    rare_density   = rare_cov["overlapping_bases"].sum()   * 1e3 / total_bases
    common_density = common_cov["overlapping_bases"].sum() * 1e3 / total_bases

    # Fisher: rare vs common hit/no-hit
    rare_yes   = int((rare_cov["total_hits"]   > 0).sum()); rare_no   = len(rare_cov)   - rare_yes
    common_yes = int((common_cov["total_hits"] > 0).sum()); common_no = len(common_cov) - common_yes
    odds, pval = fisher_exact([[rare_yes, rare_no], [common_yes, common_no]])

    return {
        "rare_density":   rare_density,
        "common_density": common_density,
        "log2_rare_common": np.log2(rare_density / common_density) if common_density > 0 and rare_density > 0 else np.nan,
        "odds":  odds,
        "pvalue": pval,
        "total_bases": total_bases,
    }


# --- SNV subsets ---
rare_snv   = rare_df_filtered[rare_df_filtered["mut_type"]   == "snp"]
common_snv = common_df_filtered[common_df_filtered["mut_type"] == "snp"]

# --- extract subregions for each dataset ---
results = []
for ds_name, df in [("G4Hunter", g4_df), ("Quadparser", regex_df)]:
    gruns_df, loops_df, transitions_df = extract_subregions(df, ds_name)

    for region_name, region_df in [
        ("Full G4",    df),
        ("G-run",      gruns_df),
        ("Loop",       loops_df),
        ("Transition", transitions_df),
    ]:
        if region_df.empty:
            continue
        row = snv_rate_per_subregion(region_df, rare_snv, common_snv)
        row["dataset"]  = ds_name
        row["subregion"] = region_name
        results.append(row)

subregion_results = pd.DataFrame(results)
_, subregion_results["qvalue"], _, _ = multipletests(subregion_results["pvalue"], method="fdr_bh")
subregion_results


In [ ]:
subregion_results["star"] = subregion_results["qvalue"].apply(p_to_star)
subregion_results["rare_common"] = np.exp(subregion_results["log2_rare_common"])
subregion_results

In [ ]:
grun_res = subregion_results[subregion_results["subregion"] == "Full G4"][["rare_common", "dataset"]]
subregion_results.drop(columns=["rare_common_g4"], inplace=True)
subregion_results = subregion_results.merge(
                    grun_res,
                    on=["dataset"],
                    how="left",
                    suffixes=("", "_g4")
            ).assign(rel_enrichment=lambda ds: ds["rare_common"] / ds["rare_common_g4"])
subregion_results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib

# Optional: higher-quality vector text in PDFs
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# ------------------------------------------------------------------
# ORDER subregions by rel_enrichment (mean across datasets)
# ------------------------------------------------------------------
plot_df = subregion_results[subregion_results["subregion"] != "Full G4"].copy()

order = (
    plot_df.groupby("subregion")["rel_enrichment"]
    .mean()
    .sort_values(ascending=False)
    .index
)

# ------------------------------------------------------------------
# STYLE
# ------------------------------------------------------------------
sns.set_style("whitegrid")

fig, ax = plt.subplots(figsize=(9, 6.5), dpi=300)

bars = sns.barplot(
    data=plot_df,
    x="subregion",
    y="odds",
    hue="dataset",
    order=order,
    palette="Set2",
    edgecolor="black",
    linewidth=1.8,
    ax=ax
)

# ------------------------------------------------------------------
# Make borders dotted / dashed
# ------------------------------------------------------------------
for patch in ax.patches:
    patch.set_linestyle((0, (2, 2)))  # dotted-like border
    patch.set_linewidth(1.8)

# ------------------------------------------------------------------
# Horizontal reference line
# ------------------------------------------------------------------
ax.axhline(
    0,
    color="black",
    lw=1.2,
    ls="--",
    alpha=0.8
)

# ------------------------------------------------------------------
# Labels / ticks
# ------------------------------------------------------------------
ax.set_ylabel("Low Frequency / Common SNVs", fontsize=16, fontweight="bold")
ax.set_xlabel("")

ax.tick_params(axis='x', labelsize=15)
ax.tick_params(axis='y', labelsize=13)

# Rotate x labels slightly for elegance
plt.setp(
    ax.get_xticklabels(),
    rotation=12,
    ha="right",
    fontweight="bold"
)

# ------------------------------------------------------------------
# Fancy spines
# ------------------------------------------------------------------
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_linewidth(1.5)
ax.spines["bottom"].set_linewidth(1.5)

# Softer grid
ax.grid(
    axis="y",
    linestyle=":",
    linewidth=0.8,
    alpha=0.5
)

# ------------------------------------------------------------------
# Legend
# ------------------------------------------------------------------
leg = ax.legend(
    title="",
    fontsize=12,
    frameon=True,
    fancybox=True,
    edgecolor="black"
)

# ------------------------------------------------------------------
# Add significance stars above bars
# ------------------------------------------------------------------
# IMPORTANT:
# assumes plot_df has a "star" column
# and rows are in the same order seaborn draws them
# ------------------------------------------------------------------

for patch, (_, row) in zip(ax.patches, plot_df.iterrows()):

    height = patch.get_height()

    x = patch.get_x() + patch.get_width() / 2

    # Position stars slightly above bar
    if height >= 0:
        y = height + 0.02 * abs(ax.get_ylim()[1])
        va = "bottom"
    else:
        y = height - 0.02 * abs(ax.get_ylim()[1])
        va = "top"

    ax.text(
        x,
        y,
        str(row["star"]),
        ha="center",
        va=va,
        fontsize=13,
        fontweight="bold"
    )

# ------------------------------------------------------------------
# Tight layout + export
# ------------------------------------------------------------------
plt.tight_layout()

# Optional ultra-high-quality save
plt.savefig("fancy_subregion_barplot.pdf", bbox_inches="tight")
plt.savefig("fancy_subregion_barplot.png", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import re
import itertools

mut_df["AF_samples"] = mut_df["AC_samples"] / mut_df["AN_samples"]
RARE_THRESHOLD = 0.05
rare_df_filtered = mut_df[(mut_df["AF_samples"] < RARE_THRESHOLD) & (mut_df["AN_samples"] >= 73)].drop_duplicates(subset=["#CHROM", "start"]).reset_index(drop=True)
common_df_filtered = mut_df[(mut_df["AF_samples"] >= RARE_THRESHOLD) & (mut_df["AN_samples"] >= 73)].drop_duplicates(subset=["#CHROM", "start"]).reset_index(drop=True)
RARITY_COLLECTIONS = {"all": mut_df, "rare": rare_df_filtered, "common": common_df_filtered}
METH_LEVELS = ["Hypomethylated", "Methylated", "Hypermethylated"]
all_results_meth = []

datasets_meth = []
for tissue in meth_df:
    for LEVEL in METH_LEVELS:
        base_df = meth_df[tissue][meth_df[tissue]["methylation_level"] == LEVEL]
        g4_bed = BedTool.from_dataframe(base_df[["seqID", "start", "end"]]).sort()
        ctrl_bed = BedTool.from_dataframe(
            base_df[["control_seqID", "control_start", "control_end"]].rename(columns={
                "control_seqID": "seqID",
                "control_start": "start",
            "control_end":   "end",
        })
        ).sort()
        datasets_meth.append((LEVEL, tissue, g4_bed, ctrl_bed, base_df))

mutations = mut_df["mut_type"].unique().tolist()
for (LEVEL, tissue, g4_bed, ctrl_bed, df_temp) in datasets_meth:
    for rarity, mut_dict in RARITY_COLLECTIONS.items():
        df = compute_enrichment(mut_dict, 
                                label=rarity, 
                                g4_bed=g4_bed, 
                                ctrl_bed=ctrl_bed, 
                                ds_name=ds_name, 
                                matched_df=df_temp)
        df["dataset"] = LEVEL
        df["tissue"] = tissue
        all_results_meth.append(df)

all_results_meth = pd.concat(all_results_meth, ignore_index=True)
all_results_meth


In [ ]:

rows = []
for rarity, snp_bed in SNV_df.items():
    for meth_level in METH_LEVELS:
        pair = {}
        for region_name, region_df in REGIONS_METH:
            sub = region_df[region_df["methylation_level"] == meth_level]
            if sub.empty:
                continue

            bed = BedTool.from_dataframe(
                sub[FIELDS_G4].drop_duplicates()
            ).sort().merge().sort()

            cov = pd.read_table(
                bed.coverage(snp_bed).fn,
                header=None,
                names=["seqID", "start", "end"] + COV_FIELDS
            )

            observed        = cov["total_hits"].sum()
            observed_bp     = cov["total_bases"].sum()
            total_bp        = int((cov["end"] - cov["start"]).sum())
            expected_gw     = gw_density[rarity] * total_bp
            variant_density = observed_bp / total_bp if total_bp > 0 else np.nan
            total_yes       = int((cov["total_hits"] > 0).sum())
            total_no        = int((cov["total_hits"] == 0).sum())

            pair[region_name] = {"total_yes": total_yes, "total_no": total_no}

            rows.append({
                "rarity":          rarity,
                "meth_level":      meth_level,
                "region":          region_name,
                "observed":        observed,
                "observed_bp":     observed_bp,
                "expected_gw":     expected_gw,
                "total_bp":        total_bp,
                "total_yes":       total_yes,
                "total_no":        total_no,
                "variant_density": variant_density,
                "norm_density_gw": observed / expected_gw if expected_gw > 0 else np.nan,
            })

        # pairwise Fisher within (rarity, meth_level)
        available = [r for r, _ in REGIONS_METH if r in pair]
        raw_p, meta = [], []
        for a, b in itertools.combinations(available, 2):
            table = [
                [pair[a]["total_yes"], pair[a]["total_no"]],
                [pair[b]["total_yes"], pair[b]["total_no"]],
            ]
            oddsratio, pval = fisher_exact(table, alternative="two-sided")
            raw_p.append(pval)
            meta.append((a, b, oddsratio))

        if raw_p:
            _, fdr_p, _, _ = multipletests(raw_p, method="fdr_bh")
            for (a, b, oddsratio), p_fdr in zip(meta, fdr_p):
                if a == "G4":
                    for r in rows:
                        if r["rarity"] == rarity and r["meth_level"] == meth_level and r["region"] == b:
                            r["pval"]       = p_fdr
                            r["odds_ratio"] = oddsratio
                            r["stars"]      = p_to_star(p_fdr)

burden_meth_comp_df = pd.DataFrame(rows)

# global FDR across all G4-vs-subregion tests
g4_mask = burden_meth_comp_df["region"] != "G4"
_, fdr_global, _, _ = multipletests(
    burden_meth_comp_df.loc[g4_mask, "pval"].fillna(1), method="fdr_bh"
)
burden_meth_comp_df.loc[g4_mask, "pval_fdr"]  = fdr_global
burden_meth_comp_df.loc[g4_mask, "stars_fdr"] = [p_to_star(p) for p in fdr_global]

burden_meth_comp_df["rate"] = burden_meth_comp_df["observed_bp"] / burden_meth_comp_df["expected_gw"]
burden_meth_comp_df

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

REGION_ORDER = ["G4", "G-run", "Loop", "Transition"]
METH_ORDER   = ["Hypomethylated", "Hypermethylated"]
METH_PALETTE = {"Hypomethylated": "#667BB2", "Hypermethylated": "#DA3E89"}

plot_df      = burden_meth_comp_df[burden_meth_comp_df["meth_level"].isin(METH_ORDER)].copy()
plot_df["variant_density"] = plot_df["variant_density"] * 1e3

global_ymax  = plot_df["variant_density"].max()
bar_width    = 0.8 / 2

# Fisher Hypo vs Hyper per (rarity, region) with FDR
hypo_hyper_stars = {}
for rarity in ["Rare", "Common"]:
    sub = plot_df[plot_df["rarity"] == rarity]
    raw_p, regions_tested = [], []
    for region in REGION_ORDER:
        hypo  = sub[(sub["region"] == region) & (sub["meth_level"] == "Hypomethylated")]
        hyper = sub[(sub["region"] == region) & (sub["meth_level"] == "Hypermethylated")]
        if hypo.empty or hyper.empty:
            continue
        table = [
            [int(hypo["observed"].values[0]),  max(0, int(hypo["total_bp"].values[0])  - int(hypo["observed"].values[0]))],
            [int(hyper["observed"].values[0]), max(0, int(hyper["total_bp"].values[0]) - int(hyper["observed"].values[0]))],
        ]
        _, pval = fisher_exact(table, alternative="two-sided")
        raw_p.append(pval)
        regions_tested.append(region)
    _, fdr_p, _, _ = multipletests(raw_p, method="fdr_bh")
    for region, p_fdr in zip(regions_tested, fdr_p):
        hypo_hyper_stars[(rarity, region)] = p_to_star(p_fdr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.subplots_adjust(wspace=0.05)
for ax, rarity in zip(axes, ["Rare", "Common"]):
    sub = plot_df[plot_df["rarity"] == rarity].copy()

    sns.barplot(
        data=sub,
        x="region", 
        y="variant_density",
        hue="meth_level",
        order=REGION_ORDER, hue_order=METH_ORDER,
        palette=METH_PALETTE,
        ax=ax, errorbar=None,
        edgecolor='black',
        lw=1.2,
    )

    ax.set_xlabel("", fontsize=16)
    ax.set_ylabel("Variant Density (SNPs / kb)" if rarity == "Rare" else "", fontsize=18)
    ax.tick_params(labelsize=18)
    ax.legend(frameon=False, handles=[])
    # ax.legend(title="", fontsize=16, bbox_to_anchor=(0.5, -0.4), title_fontsize=13)
    ax.grid(axis="y", lw=0.4, alpha=0.6)
    ax.set_axisbelow(True)
    ax.set_ylim(0, global_ymax * 1.22)

    ax.text(0.5, 1.03, rarity, transform=ax.transAxes,
            ha="center", va="bottom", fontsize=16, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", lw=1.2))

    for i, region in enumerate(REGION_ORDER):
        stars = hypo_hyper_stars.get((rarity, region), "")
        if not stars:
            continue
        x_hypo  = i - bar_width / 2
        x_hyper = i + bar_width / 2
        y_top   = global_ymax * 1.10
        ax.plot([x_hypo, x_hypo, x_hyper, x_hyper],
                [y_top - global_ymax * 0.02, y_top, y_top, y_top - global_ymax * 0.02],
                color="black", lw=1)
        ax.text((x_hypo + x_hyper) / 2, y_top + global_ymax * 0.01, stars,
                ha="center", va="bottom", fontsize=13, fontweight="bold")

plt.savefig(os.path.join(target_fig, "burden_hypo_hyper_subregions.pdf"),
            transparent=True, 
            bbox_inches="tight")
plt.savefig(os.path.join(target_fig, "burden_hypo_hyper_subregions.png"),
            transparent=True, 
            dpi=300,
            bbox_inches="tight")
plt.show()

In [ ]:
from matplotlib.colors import TwoSlopeNorm

import pyranges as pr

genome_size = sum(seq_sizes.values())
gw_densities_meth = {}
gw_densities_controls_meth = {}
matched_meth_pl = pl.from_pandas(meth_matched)
METHYLATION_LEVELS = ["Hypomethylated", "Methylated", "Hypermethylated"]

for meth_level in METHYLATION_LEVELS:
    subset = (
        matched_meth_pl
        .filter(pl.col("methylation_level") == meth_level)
        .select(["seqID", "start", "end"])
        .rename({"seqID": "Chromosome", "start": "Start", "end": "End"})
        .to_pandas()
    )
    merged_length = (
        pr.PyRanges(subset)
        .merge()
        .df
        .assign(length=lambda d: d["End"] - d["Start"])
        ["length"]
        .sum()
    )
    
    gw_densities_meth[meth_level] = 1e6 * merged_length / genome_size
    
    # Controls
    control_subset = (
        matched_meth_pl
        .filter(pl.col("methylation_level") == meth_level)
        .select(["seqID", "control_start", "control_end"])
        .rename({"seqID": "Chromosome", "control_start": "Start", "control_end": "End"})
        .to_pandas()
    )
    control_merged_length = (
        pr.PyRanges(control_subset)
        .merge()
        .df
        .assign(length=lambda d: d["End"] - d["Start"])
        ["length"]
        .sum()
    )
    gw_densities_controls_meth[meth_level] = 1e6 * control_merged_length / genome_size

print(gw_densities_meth)
print(gw_densities_controls_meth)

In [ ]:

MUTATIONS   = ["snp", "mnp", "ins", "smallins", "smalldel", "del"]
RARITIES    = ["rare", "common"]
METH_LEVELS = ["Hypomethylated", "Methylated", "Hypermethylated"]

df = {
    "rare":   rare_df_filtered,
    "common": common_df_filtered,
}
g4_meth_bed = {}
ctrl_meth_bed = {}
matched_meth_pl = pl.from_pandas(meth_matched)

for meth_level in METHYLATION_LEVELS:
    g4_meth_bed[meth_level] = BedTool.from_dataframe(
        matched_meth_pl
        .filter(pl.col("methylation_level") == meth_level)
        .select(["seqID", "start", "end"])
        .to_pandas()
    ).sort()

    ctrl_meth_bed[meth_level] = BedTool.from_dataframe(
        matched_meth_pl
        .filter(pl.col("methylation_level") == meth_level)
        .select(["control_seqID", "control_start", "control_end"])
        .rename({"control_seqID": "seqID", "control_start": "start", "control_end": "end"})
        .to_pandas()
    ).sort()


In [ ]:
from scipy.stats import wilcoxon
FIELDS = ["seqID", "start", "end"]
for meth_level in tqdm(METH_LEVELS):
    # merged for Fisher/density
    g4_merged   = g4_meth_bed[meth_level].sort().merge().sort()
    ctrl_merged = ctrl_meth_bed[meth_level].sort().merge().sort()

    # unmerged for Wilcoxon pairing
    g4_unmerged   = g4_meth_bed[meth_level].sort()
    ctrl_unmerged = ctrl_meth_bed[meth_level].sort()

    for rarity in RARITIES:
        for mutation in MUTATIONS:
            df_mut = df[rarity]
            df_mut = df_mut[df_mut["mut_type"] == mutation].reset_index(drop=True)
            df_mut = df_mut[["#CHROM", "start", "end"]].drop_duplicates(subset=["#CHROM", "start", "end"])
            mut_b  = BedTool.from_dataframe(df_mut).sort()

            cov_cols = ["seqID", "start", "end", "total_hits", "overlapping_bases", "all_bases", "coverage"]

            # merged coverage for Fisher + density
            g4_cov_m   = pd.read_table(g4_merged.coverage(mut_b).fn,   header=None, names=cov_cols)
            ctrl_cov_m = pd.read_table(ctrl_merged.coverage(mut_b).fn, header=None, names=cov_cols)

            # unmerged coverage for Wilcoxon
            g4_cov   = pd.read_table(g4_unmerged.coverage(mut_b).fn,   header=None, names=cov_cols)
            ctrl_cov = pd.read_table(ctrl_unmerged.coverage(mut_b).fn, header=None, names=cov_cols)

            matched_meth_df_merged = (
                meth_matched[meth_matched["methylation_level"] == meth_level]
                .merge(g4_cov, on=["seqID", "start", "end"])
                .merge(ctrl_cov,
                       left_on=["control_seqID", "control_start", "control_end"],
                       right_on=["seqID", "start", "end"],
                       suffixes=("_g4", "_ctrl"))
            )
            assert matched_meth_df_merged.shape[0] == meth_matched[meth_matched["methylation_level"] == meth_level].shape[0], \
                f"Shape mismatch for {meth_level} {rarity} {mutation}"

            stat, pval_wx = wilcoxon(matched_meth_df_merged["coverage_g4"], matched_meth_df_merged["coverage_ctrl"])

            g4_density   = g4_cov_m["overlapping_bases"].sum() * 1e3 / g4_cov_m["all_bases"].sum()
            ctrl_density = ctrl_cov_m["overlapping_bases"].sum() * 1e3 / ctrl_cov_m["all_bases"].sum()

            g4_yes   = (g4_cov_m["total_hits"] > 0).sum();   g4_no   = len(g4_cov_m)   - g4_yes
            ctrl_yes = (ctrl_cov_m["total_hits"] > 0).sum(); ctrl_no = len(ctrl_cov_m) - ctrl_yes
            oddsratio, pval  = fisher_exact([[g4_yes, g4_no], [ctrl_yes, ctrl_no]])

            records.append({
                "methylation_level": meth_level,
                "rarity":            rarity,
                "mutation":          mutation,
                "g4_density":        g4_density,
                "ctrl_density":      ctrl_density,
                "oddsratio":         oddsratio,
                "fe":                g4_density / ctrl_density if ctrl_density > 0 else np.nan,
                "log2_fe":           np.log2(g4_density / ctrl_density) if ctrl_density > 0 and g4_density > 0 else np.nan,
                "pvalue":            pval,
                "pval_wx":           pval_wx,
            })

results_mut_df = pd.DataFrame(records)
_, qvals, _, _ = multipletests(results_mut_df["pvalue"], method="fdr_bh")
results_mut_df["qvalue"] = qvals
results_mut_df["stars"]  = results_mut_df["qvalue"].apply(
    lambda q: "***" if q < 0.001 else ("**" if q < 0.01 else ("*" if q < 0.05 else "ns"))
)
_, qvals, _, _ = multipletests(results_mut_df["pval_wx"], method="fdr_bh")
results_mut_df["qvalue_wx"] = qvals
results_mut_df["stars_wx"]  = results_mut_df["qvalue_wx"].apply(
    lambda q: "***" if q < 0.001 else ("**" if q < 0.01 else ("*" if q < 0.05 else "ns"))
)
results_mut_df

## The end?

In [ ]:
# Load Controls
target = os.getenv("SCRATCH")
target = Path(target).joinpath("g4_t2t_revisions_data")
group = "G4Hunter"
controls = dict()
controls_bed = dict()
g4_bed = dict()
matched = dict()
for group in tqdm(["G4Hunter", "Quadparser", "eG4"]):
    matched_df = pd.read_table(f"{target}/matched_controls_{group}_global.tsv.gz")
    controls[group] = matched_df
    matched[group] = matched_df

    controls_bed[group] = BedTool.from_dataframe(matched_df[["seqID", "control_start", "control_end"]]).sort()
    g4_bed[group] = BedTool.from_dataframe(
                        matched_df[["seqID", "start", "end"]]   
                    ).sort()

for group in tqdm(["G4Hunter", "Quadparser", "eG4"]):
    for groupA in tqdm(["G4Hunter", "Quadparser", "eG4"]):
        assert g4_bed[group].intersect(controls_bed[groupA]).count() == 0, g4_bed[group].intersect(controls_bed[groupA]).count() 

In [ ]:
regions_df = pd.read_table(Path(os.getenv('WORK')).joinpath("compartments_coords.tsv.gz"))
regions_df

In [ ]:
import pybedtools
from pybedtools import BedTool
from pathlib import Path
from scipy.stats import expon
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import os

target = Path(os.getenv("SCRATCH")).joinpath("g4_t2t_revisions_data")
groups = ["G4Hunter", "Quadparser"]

KS_THRESH  = 0.05
RHO_THRESH = 0.90
GC_ERR_MAX = 0.01
DUP_MAX    = 0.01
def verdict(ok): return "✓ PASS" if ok else "✗ FAIL"

for group in groups:
    fname = "/scratch/10904/nikolchanchan/g4_t2t_revisions_data/matched_rank_HG002_G4Hunter_filtered_with_meth.cpg_strict.no_rank.filtered.tsv.gz"
    matched_df = pd.read_table(fname)

    print(f"\n{'='*62}")
    print(f"  {group}  —  Control Validation  (n={len(matched_df):,} pairs)")
    print(f"{'='*62}")

    results = {}

    # ── 1. Duplicate controls ────────────────────────────────────────
    n_dups   = matched_df[["seqID", "control_start", "control_end"]].duplicated().sum()
    dup_frac = n_dups / len(matched_df)
    ok = dup_frac <= DUP_MAX
    results["uniqueness"] = ok
    print(f"\n[1] Control uniqueness          {verdict(ok)}")
    print(f"    Duplicate controls: {n_dups:,} ({dup_frac*100:.3f}%)")

    # ── 2. Chromosome matching ───────────────────────────────────────
    if "control_seqID" in matched_df.columns:
        chrom_match = (matched_df["seqID"] == matched_df["control_seqID"]).mean()
        ok = chrom_match == 1.0
        results["chrom_match"] = ok
        print(f"\n[2] Same-chromosome pairing     {verdict(ok)}")
        print(f"    Fraction same chrom: {chrom_match:.6f}")
    else:
        print(f"\n[2] Same-chromosome pairing     — control_seqID column absent")

    # ── 3. Self-overlap (G4 ↔ its control) ──────────────────────────
    g4_bed   = BedTool.from_dataframe(matched_df[["seqID", "start", "end"]]).sort()
    ctrl_bed = BedTool.from_dataframe(matched_df[["seqID", "control_start", "control_end"]]).sort()
    n_self   = ctrl_bed.intersect(g4_bed, u=True).count()
    ok = n_self == 0
    results["no_self_overlap"] = ok
    print(f"\n[3] No G4 ↔ control self-overlap {verdict(ok)}")
    print(f"    Overlapping pairs: {n_self:,} ({n_self/len(matched_df)*100:.3f}%)")

    # ── 4. Length matching ───────────────────────────────────────────
    g4_len   = matched_df["end"] - matched_df["start"]
    ctrl_len = matched_df["control_end"] - matched_df["control_start"]
    ks_len   = stats.ks_2samp(g4_len, ctrl_len)
    ok = ks_len.pvalue > KS_THRESH
    results["length_match"] = ok
    print(f"\n[4] Length distribution match   {verdict(ok)}")
    print(f"    G4 mean={g4_len.mean():.1f} bp  Control mean={ctrl_len.mean():.1f} bp")
    print(f"    KS stat={ks_len.statistic:.4f}  p={ks_len.pvalue:.4f}  (want p>{KS_THRESH})")

    # ── 5. GC content matching ───────────────────────────────────────
    ks_gc      = stats.ks_2samp(matched_df["gc_content"], matched_df["control_gc_content"])
    rho, p_rho = stats.spearmanr(matched_df["gc_content"], matched_df["control_gc_content"])
    ok = (ks_gc.pvalue > KS_THRESH) and (rho >= RHO_THRESH)
    results["gc_match"] = ok
    print(f"\n[5] GC content matching         {verdict(ok)}")
    print(f"    G4 mean={matched_df['gc_content'].mean():.4f}  Control mean={matched_df['control_gc_content'].mean():.4f}")
    print(f"    KS stat={ks_gc.statistic:.4f}  p={ks_gc.pvalue:.4f}  (want p>{KS_THRESH})")
    print(f"    Spearman ρ={rho:.4f}  p={p_rho:.2e}  (want ρ>{RHO_THRESH})")

    # ── 6. Sequence-derived GC accuracy ─────────────────────────────
    sample = matched_df[matched_df["seqID"] != "chrY"].sample(min(2000, len(matched_df)), random_state=42)

    def gc_from_seq(row, s_col, e_col):
        seq = chrom_sequences[row["seqID"]][row[s_col]:row[e_col]].upper()
        return seq.count("G") + seq.count("C")

    def cpg_from_seq(row, s_col, e_col):
        seq = chrom_sequences[row["seqID"]][row[s_col]:row[e_col]].upper()
        return seq.count("CG")

    seq_gc_g4   = sample.apply(lambda r: gc_from_seq(r, "start", "end"), axis=1)
    seq_gc_ctrl = sample.apply(lambda r: gc_from_seq(r, "control_start", "control_end"), axis=1)
    delta_g4    = (seq_gc_g4   - sample["gc_content"].values).abs().mean()
    delta_ctrl  = (seq_gc_ctrl - sample["control_gc_content"].values).abs().mean()
    corr_g4,   _ = stats.pearsonr(seq_gc_g4,   sample["gc_content"])
    corr_ctrl, _ = stats.pearsonr(seq_gc_ctrl, sample["control_gc_content"])
    ok = (delta_g4 <= GC_ERR_MAX) and (delta_ctrl <= GC_ERR_MAX)
    results["gc_accuracy"] = ok
    print(f"\n[6] Stored GC vs sequence truth {verdict(ok)}  (sample n={len(sample):,}, no chrY)")
    print(f"    G4     r={corr_g4:.4f}   mean|Δ|={delta_g4:.6f}  (want ≤{GC_ERR_MAX})")
    print(f"    Control r={corr_ctrl:.4f}  mean|Δ|={delta_ctrl:.6f}  (want ≤{GC_ERR_MAX})")

    # ── 7. Compartment distribution (informational) ──────────────────
    reg_bed = BedTool.from_dataframe(regions_df[["seqID", "start", "end", "group"]]).sort()

    def compartment_counts(query_bed):
        hit = query_bed.intersect(reg_bed, 
                                  wa=True, 
                                  wb=True, 
                                  loj=True)
        df  = pd.read_table(hit.fn, 
                            header=None,
                            names=["q_chr","q_s","q_e","r_chr","r_s","r_e","group"])
        df["group"] = df["group"].replace(".", "intergenic")
        return df.groupby("group").size()

    g4_comp   = compartment_counts(g4_bed)
    ctrl_comp = compartment_counts(ctrl_bed)
    comp_df   = pd.DataFrame({"G4": g4_comp, 
                            "Control": ctrl_comp}).fillna(0).astype(int)
    chi2, p_chi2, _, _ = stats.chi2_contingency(comp_df.T)
    results["compartment_informational"] = True
    print(f"\n[7] Compartment distribution    (informational — G4 enrichment expected)")
    print(f"    Chi² = {chi2:.2f}  p = {p_chi2:.4f}")
    print(comp_df.to_string())

    # ── 8. Controls don't overlap any G4 dataset ─────────────────────
    all_g4_bed = BedTool.from_dataframe(
        pd.concat([
            df.iloc[:, :3].set_axis(["seqID", "start", "end"], axis=1)
            for _, df in datasets
        ], ignore_index=True)
    ).sort().merge()
    n_ctrl_in_g4 = ctrl_bed.intersect(all_g4_bed, u=True).count()
    ok = n_ctrl_in_g4 == 0
    results["no_g4_overlap"] = ok
    print(f"\n[8] Controls free of all G4 datasets  {verdict(ok)}")
    print(f"    Controls overlapping any G4 dataset: {n_ctrl_in_g4:,} ({n_ctrl_in_g4/len(matched_df)*100:.3f}%)")
    for ds_name, ds_df in datasets:
        ds_bed = BedTool.from_dataframe(ds_df.iloc[:, :3].set_axis(["seqID","start","end"], axis=1)).sort()
        print(f"      vs {ds_name:12s}: {ctrl_bed.intersect(ds_bed, u=True).count():,}")

    # ── 9. CpG content matching ──────────────────────────────────────
    cpg_g4   = sample.apply(lambda r: cpg_from_seq(r, "start", "end"), axis=1)
    cpg_ctrl = sample.apply(lambda r: cpg_from_seq(r, "control_start", "control_end"), axis=1)
    ks_cpg      = stats.ks_2samp(cpg_g4, cpg_ctrl)
    rho_cpg, _  = stats.spearmanr(cpg_g4, cpg_ctrl)
    ok = rho_cpg >= RHO_THRESH
    results["cpg_match"] = ok
    print(f"\n[9] CpG content matching        {verdict(ok)}  (sample n={len(sample):,}, no chrY)")
    print(f"    G4 mean CpG={cpg_g4.mean():.4f}  Control mean CpG={cpg_ctrl.mean():.4f}")
    print(f"    KS stat={ks_cpg.statistic:.4f}  p={ks_cpg.pvalue:.4f}")
    print(f"    Spearman ρ={rho_cpg:.4f}  (pair-level, want ρ>{RHO_THRESH})")

    # ── 10. Positional distribution within chromosomes ───────────────
    print(f"\n[10] Positional distribution within chromosomes")
    chrom_sizes = {ch: len(seq) for ch, seq in chrom_sequences.items()}
    pos_results = {}
    cv_results  = {}

    for chrom, grp in matched_df[matched_df["seqID"] != "chrY"].groupby("seqID"):
        if chrom not in chrom_sizes or len(grp) < 50:
            continue
        clen      = chrom_sizes[chrom]
        g4_norm   = grp["start"].values / clen
        ctrl_norm = np.sort(grp["control_start"].values / clen)

        ks = stats.ks_2samp(g4_norm, ctrl_norm)
        pos_results[chrom] = {"ks_stat": ks.statistic, "pvalue": ks.pvalue, "n": len(grp)}

        g4_gaps   = np.diff(np.sort(g4_norm))
        ctrl_gaps = np.diff(ctrl_norm)
        cv_g4   = g4_gaps.std()   / g4_gaps.mean()   if g4_gaps.mean()   > 0 else np.nan
        cv_ctrl = ctrl_gaps.std() / ctrl_gaps.mean() if ctrl_gaps.mean() > 0 else np.nan
        cv_results[chrom] = {"cv_g4": cv_g4, "cv_ctrl": cv_ctrl,
                             "ratio": cv_ctrl / cv_g4 if cv_g4 > 0 else np.nan}

    pos_df = pd.DataFrame(pos_results).T.sort_index()
    cv_df  = pd.DataFrame(cv_results).T.sort_index()

    n_differ   = (pos_df["pvalue"] < KS_THRESH).sum()
    ok_10a     = n_differ >= len(pos_df) * 0.8
    mean_ratio = cv_df["ratio"].mean()
    ok_10b     = 0.5 <= mean_ratio <= 2.0
    results["ctrl_differs_from_g4"]  = ok_10a
    results["ctrl_spacing_like_g4s"] = ok_10b

    print(f"\n  [10a] Controls differ positionally from G4s  {verdict(ok_10a)}")
    print(f"        Chroms where ctrl ≠ G4 (p<0.05): {n_differ}/{len(pos_df)}  (want ≥80%)")
    for ch, r in pos_df.nsmallest(3, "ks_stat").iterrows():
        print(f"        {ch:12s}  KS={r['ks_stat']:.4f}  p={r['pvalue']:.2e}")

    print(f"\n  [10b] Control gap CV ≈ G4 gap CV (same genome structure)  {verdict(ok_10b)}")
    print(f"        Mean CV ratio ctrl/G4: {mean_ratio:.3f}  (want 0.5–2.0)")
    print(f"        Mean CV G4={cv_df['cv_g4'].mean():.2f}  Mean CV ctrl={cv_df['cv_ctrl'].mean():.2f}")

    # ── 11. Inter-control gap randomness ────────────────────────────
    print(f"\n[11] Inter-control gap randomness")
    ks_exp_per_chrom = {}
    ks_g4_per_chrom  = {}

    for chrom, grp in matched_df[matched_df["seqID"] != "chrY"].groupby("seqID"):
        if len(grp) < 50:
            continue
        ctrl_gaps = np.diff(np.sort(grp["control_start"].values)).astype(float)
        g4_gaps   = np.diff(np.sort(grp["start"].values)).astype(float)
        if ctrl_gaps.mean() == 0:
            continue
        ks_exp = stats.kstest(ctrl_gaps, "expon", args=(0, ctrl_gaps.mean()))
        ks_g4  = stats.ks_2samp(ctrl_gaps, g4_gaps)
        ks_exp_per_chrom[chrom] = {"ks_stat": ks_exp.statistic, "pvalue": ks_exp.pvalue,
                                   "cv": ctrl_gaps.std() / ctrl_gaps.mean()}
        ks_g4_per_chrom[chrom]  = {"ks_stat": ks_g4.statistic,  "pvalue": ks_g4.pvalue}

    exp_df     = pd.DataFrame(ks_exp_per_chrom).T.sort_index()
    gap_g4_df  = pd.DataFrame(ks_g4_per_chrom).T.sort_index()
    n_exp_pass = (exp_df["pvalue"] > KS_THRESH).sum()
    n_g4_sim   = (gap_g4_df["pvalue"] > KS_THRESH).sum()
    ok = n_g4_sim >= len(gap_g4_df) * 0.5
    results["ctrl_gaps_like_g4s"] = ok

    print(f"    Ctrl gaps ≈ exponential (theoretical random): {n_exp_pass}/{len(exp_df)} chroms")
    print(f"    Ctrl gaps ≈ G4 gaps     (same GC landscape):  {n_g4_sim}/{len(gap_g4_df)} chroms  {verdict(ok)}")
    print(f"    Mean CV of ctrl gaps: {exp_df['cv'].mean():.3f}  (exponential=1.0)")
    for ch, r in exp_df.nsmallest(3, "pvalue").iterrows():
        print(f"      {ch:12s}  KS={r['ks_stat']:.4f}  p={r['pvalue']:.2e}  CV={r['cv']:.2f}")

    # representative chrom for gap plot (most intervals)
    best_chrom  = matched_df[matched_df["seqID"] != "chrY"].groupby("seqID").size().idxmax()
    grp_best    = matched_df[matched_df["seqID"] == best_chrom]
    ctrl_gaps_plot = np.diff(np.sort(grp_best["control_start"].values)).astype(float)
    g4_gaps_plot   = np.diff(np.sort(grp_best["start"].values)).astype(float)
    clip_val       = np.percentile(ctrl_gaps_plot, 99)

    # ── 12. Paired G4 ↔ control distance ────────────────────────────
    print(f"\n[12] Paired G4 ↔ control distance")
    signed_dist  = matched_df["control_start"] - matched_df["start"]
    abs_dist     = signed_dist.abs()
    n_downstream = (signed_dist > 0).sum()
    n_upstream   = (signed_dist < 0).sum()
    binom_p      = stats.binomtest(n_downstream, n_downstream + n_upstream, p=0.5).pvalue
    ok_12a       = binom_p > KS_THRESH
    ok_12b       = abs_dist.median() >= 1000
    results["ctrl_no_direction_bias"] = ok_12a
    results["ctrl_not_too_close"]     = ok_12b

    print(f"\n  [12a] No directional bias  {verdict(ok_12a)}")
    print(f"        Downstream: {n_downstream:,} ({n_downstream/len(matched_df)*100:.1f}%)"
          f"  Upstream: {n_upstream:,} ({n_upstream/len(matched_df)*100:.1f}%)")
    print(f"        Binomial p={binom_p:.4f}  (want p>0.05)")

    print(f"\n  [12b] Controls not hugging G4s (median ≥1 kb)  {verdict(ok_12b)}")
    print(f"        Min={abs_dist.min():,}  Median={abs_dist.median():,.0f}"
          f"  Mean={abs_dist.mean():,.0f}  Max={abs_dist.max():,} bp")

    chrom_bias = {}
    for chrom, grp in matched_df[matched_df["seqID"] != "chrY"].groupby("seqID"):
        d    = grp["control_start"] - grp["start"]
        n_dn = (d > 0).sum(); n_up = (d < 0).sum()
        p    = stats.binomtest(n_dn, n_dn + n_up, p=0.5).pvalue
        chrom_bias[chrom] = {"pct_downstream": n_dn / len(grp) * 100, "binom_p": p}
    bias_df  = pd.DataFrame(chrom_bias).T
    n_biased = (bias_df["binom_p"] < KS_THRESH).sum()
    print(f"\n  [12c] Per-chromosome direction bias")
    print(f"        Chroms with significant bias: {n_biased}/{len(bias_df)}")
    for ch, r in bias_df.nsmallest(3, "binom_p").iterrows():
        print(f"        {ch:12s}  {r['pct_downstream']:.1f}% downstream  p={r['binom_p']:.2e}")

    # ── Summary ──────────────────────────────────────────────────────
    n_pass  = sum(results.values())
    n_total = len(results)
    print(f"\n{'─'*62}")
    print(f"  SUMMARY: {n_pass}/{n_total} checks passed",
          "✓ Controls look good!" if n_pass == n_total else "✗ Issues found — inspect above")
    print(f"{'─'*62}")

    # ── Plots ────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 4, figsize=(20, 9))

    # [0,0] GC scatter
    axes[0,0].scatter(matched_df["gc_content"], matched_df["control_gc_content"],
                      alpha=0.05, s=1, color="steelblue", rasterized=True)
    lims = [min(matched_df["gc_content"].min(), matched_df["control_gc_content"].min()),
            max(matched_df["gc_content"].max(), matched_df["control_gc_content"].max())]
    axes[0,0].plot(lims, lims, "r--", lw=1)
    axes[0,0].set_xlabel("G4 GC content", fontsize=12)
    axes[0,0].set_ylabel("Control GC content", fontsize=12)
    axes[0,0].set_title(f"GC scatter  ρ={rho:.3f}")
    axes[0,0].grid(lw=0.4, alpha=0.6)

    # [0,1] GC distribution
    axes[0,1].hist(matched_df["gc_content"], bins=50, alpha=0.6, label="G4", color="steelblue", density=True)
    axes[0,1].hist(matched_df["control_gc_content"], bins=50, alpha=0.6, label="Control", color="tomato", density=True)
    axes[0,1].set_xlabel("GC content", fontsize=12)
    axes[0,1].set_ylabel("Density", fontsize=12)
    axes[0,1].legend(fontsize=11)
    axes[0,1].grid(lw=0.4, alpha=0.6)
    axes[0,1].set_title(f"GC distribution  KS p={ks_gc.pvalue:.3f}")

    # [0,2] Length distribution
    axes[0,2].hist(g4_len, bins=50, alpha=0.6, label="G4", color="steelblue", density=True)
    axes[0,2].hist(ctrl_len, bins=50, alpha=0.6, label="Control", color="tomato", density=True)
    axes[0,2].set_xlabel("Region length (bp)", fontsize=12)
    axes[0,2].set_ylabel("Density", fontsize=12)
    axes[0,2].legend(fontsize=11)
    axes[0,2].grid(lw=0.4, alpha=0.6)
    axes[0,2].set_title(f"Length distribution  KS p={ks_len.pvalue:.3f}")

    # [0,3] Compartments
    comp_pct = comp_df.div(comp_df.sum()).mul(100)
    x = np.arange(len(comp_pct)); w = 0.35
    axes[0,3].bar(x - w/2, comp_pct["G4"],      w, label="G4",      color="steelblue", alpha=0.8)
    axes[0,3].bar(x + w/2, comp_pct["Control"], w, label="Control", color="tomato",    alpha=0.8)
    axes[0,3].set_xticks(x)
    axes[0,3].set_xticklabels(comp_pct.index, rotation=35, ha="right", fontsize=8)
    axes[0,3].set_ylabel("% of regions", fontsize=12)
    axes[0,3].set_title("Compartments (informational)")
    axes[0,3].legend(fontsize=10)
    axes[0,3].grid(lw=0.4, alpha=0.6)

    # [1,0] CpG scatter
    axes[1,0].scatter(cpg_g4, cpg_ctrl, alpha=0.2, s=3, color="steelblue", rasterized=True)
    lims_cpg = [min(cpg_g4.min(), cpg_ctrl.min()), max(cpg_g4.max(), cpg_ctrl.max())]
    axes[1,0].plot(lims_cpg, lims_cpg, "r--", lw=1)
    axes[1,0].set_xlabel("G4 CpG count", fontsize=12)
    axes[1,0].set_ylabel("Control CpG count", fontsize=12)
    axes[1,0].set_title(f"CpG scatter  ρ={rho_cpg:.3f}")
    axes[1,0].grid(lw=0.4, alpha=0.6)

    # [1,1] CpG distribution
    axes[1,1].hist(cpg_g4,   bins=30, alpha=0.6, label="G4",      color="steelblue", density=True)
    axes[1,1].hist(cpg_ctrl, bins=30, alpha=0.6, label="Control", color="tomato",    density=True)
    axes[1,1].set_xlabel("CpG count", fontsize=12)
    axes[1,1].set_ylabel("Density", fontsize=12)
    axes[1,1].legend(fontsize=11)
    axes[1,1].grid(lw=0.4, alpha=0.6)
    axes[1,1].set_title(f"CpG distribution  KS p={ks_cpg.pvalue:.3f}")

    # [1,2] Inter-control gap distribution (best chrom)
    axes[1,2].hist(np.clip(ctrl_gaps_plot, 0, clip_val), bins=80, alpha=0.6,
                   color="tomato", density=True, label="Control gaps")
    axes[1,2].hist(np.clip(g4_gaps_plot,  0, clip_val), bins=80, alpha=0.6,
                   color="steelblue", density=True, label="G4 gaps")
    x_exp = np.linspace(0, clip_val, 300)
    axes[1,2].plot(x_exp, expon.pdf(x_exp, scale=ctrl_gaps_plot.mean()),
                   "k--", lw=1.5, label="Exponential(mean)")
    axes[1,2].set_xlabel("Gap between consecutive controls (bp)", fontsize=12)
    axes[1,2].set_ylabel("Density", fontsize=12)
    axes[1,2].set_title(f"Inter-control gaps ({best_chrom})\nCV={exp_df.loc[best_chrom,'cv']:.2f}  ctrl≈G4: {n_g4_sim}/{len(gap_g4_df)}")
    axes[1,2].legend(fontsize=10)
    axes[1,2].grid(lw=0.4, alpha=0.6)

    # [1,3] Paired G4↔control signed distance
    clip_dist = np.percentile(abs_dist, 99)
    axes[1,3].hist(np.clip(signed_dist, -clip_dist, clip_dist), bins=100,
                   color="steelblue", alpha=0.8, density=True)
    axes[1,3].axvline(0, color="r", lw=1.2, ls="--", label="distance=0")
    axes[1,3].set_xlabel("Control start − G4 start (bp)", fontsize=12)
    axes[1,3].set_ylabel("Density", fontsize=12)
    axes[1,3].set_title(f"Paired distance\n{n_downstream/len(matched_df)*100:.1f}% downstream  binom p={binom_p:.3f}")
    axes[1,3].legend(fontsize=10)
    axes[1,3].grid(lw=0.4, alpha=0.6)

    plt.suptitle(f"{group} — Control Validation", fontsize=14, y=1.01)
    plt.tight_layout()
    out_dir = Path(os.getenv("SCRATCH")) / "figures_g4_t2t"
    out_dir.mkdir(exist_ok=True)
    plt.savefig(out_dir / f"control_validation_{group}.pdf", bbox_inches="tight", transparent=True)
    plt.show()


### Trinucleotide Model

In [ ]:
from pathlib import Path
import pickle 
import os

with open("../pG4utils/models/trinucleotide_model_event_rate_1.pkl", "rb") as f:
    model = pickle.load(f)
gw_SNV_density = model["baseline"]

In [ ]:
nuc = {'A': 'T' ,'T': 'A', 'G': 'C', 'C': 'G'}

def map_canonical(x):
    rev = ''.join(nuc[c] for c in x)[::-1]
    return min(rev, x)
    
def prepare_snp_df(df):
    df = df[df["mut_type"] == "snp"].drop_duplicates(subset=["#CHROM", "start"]).reset_index(drop=True)
    df = df.copy()
    df = df[~df["ALT"].isna()].reset_index(drop=True)
    df["size"]           = df["#CHROM"].map(seq_sizes)
    df["expanded_start"] = np.maximum(0, df["start"] - 1)
    df["expanded_end"]   = np.minimum(df["end"] + 1, df["size"])
    df["context_length"] = df["expanded_end"] - df["expanded_start"]
    df = df[df["context_length"] == 3].reset_index(drop=True)
    df["context"] = df.apply(
        lambda row: chrom_sequences[row["#CHROM"]][row["expanded_start"]: row["expanded_end"]], axis=1
    )
    df["canonical"] = df["context"].apply(map_canonical)
    return df


snp_rare_df = prepare_snp_df(rare_df_filtered)
snp_common_df = prepare_snp_df(common_df_filtered)

In [ ]:
OUT_DIR = Path(os.getenv("SCRATCH")) / "g4_t2t_revisions_data"
def load_matched_controls(name: str, out_dir: Path = OUT_DIR) -> pd.DataFrame:
    return pd.read_table(out_dir / f"matched_controls_{name}_global.tsv.gz")

datasets_to_match = {
    "G4Hunter":   g4_df,
    "Quadparser": regex_df,
}

matched = {}
for name in datasets_to_match.keys():
    matched[name] = load_matched_controls(name)
    print(f"{name}: {matched[name].shape[0]} matched pairs")

In [ ]:
target     = Path("/scratch/10904/nikolchanchan/g4_t2t_revisions_data")
meth_annotated = {}
for sample_name in ["HG002", "CHM13"]:
    for ds_name in ['G4Hunter', 'Quadparser', 'eG4', 'Control']:
        out = target / f"{ds_name}_motif_methylation_annot_{sample_name}.tsv.gz"
        result = pd.read_table(str(out))
        meth_annotated[(ds_name, sample_name)] = result
        print(f"{out.name}  ({len(result):,} rows)", flush=True)

In [ ]:
matched_all_df = matched["G4Hunter"].copy()
matched_all_df.head()

In [ ]:
import pyranges as pr
def to_pr(df):
    return pr.PyRanges(df.rename(columns={"#CHROM": "Chromosome", "start": "Start", "end": "End"}))

def rename(df):
    return df.rename(columns={"seqID": "Chromosome", 
                              "start": "Start", 
                              "end": "End",
                              "control_seqID": "Chromosome",
                              "control_start": "Start",
                              "control_end": "End"
                              })

g4_df = pd.read_table(G4HUNTER)
regex_df = pd.read_table(REGEX)

if "NBR" in g4_df.columns:
    g4_df.drop(columns=["NBR"], inplace=True)
    
g4_pr                  = pr.PyRanges(rename(g4_df)).merge()
regex_pr               = pr.PyRanges(rename(regex_df)).merge()
# matched_all_df_g4_pr   = pr.PyRanges(rename(matched_all_df[["seqID", "start", "end"]])).merge()
# matched_all_df_ctrl_pr = pr.PyRanges(
#     matched_all_df[["control_seqID", "control_start", "control_end"]]
#     .rename(columns={"control_seqID": "Chromosome", "control_start": "Start", "control_end": "End"})
# ).merge()

# matched_all_df_g4_pr = pr.PyRanges(rename(matched_all_df[["seqID", "start", "end"]]))
# matched_all_df_ctrl_pr = pr.PyRanges(rename(matched_all_df[["control_seqID", "control_start", "control_end"]]))


def snvs_in_regions(snp_df, region_pr):
    snp_pr = to_pr(snp_df)
    joined = snp_pr.join(region_pr, suffix="_g4")
    df = joined.df
    df = df[(df["Start"] >= df["Start_g4"]) & (df["End"] <= df["End_g4"])]
    return df.rename(columns={"Chromosome": "seqID", "Start": "start", "End": "end"}).reset_index(drop=True)


common_in_g4hunter    = snvs_in_regions(snp_common_df, g4_pr)
common_in_regex    = snvs_in_regions(snp_common_df, regex_pr)
# common_in_matched_g4   = snvs_in_regions(snp_common_df,   matched_all_df_g4_pr)
# common_in_matched_ctrl      = snvs_in_regions(snp_common_df,   matched_all_df_ctrl_pr)
rare_in_g4hunter   = snvs_in_regions(snp_rare_df,   g4_pr)
rare_in_regex      = snvs_in_regions(snp_rare_df,   regex_pr)
# rare_in_matched_g4   = snvs_in_regions(snp_rare_df,   matched_all_df_g4_pr)
# rare_in_matched_ctrl      = snvs_in_regions(snp_rare_df,   matched_all_df_ctrl_pr)

print(f"Common SNVs in G4Hunter: {len(common_in_g4hunter):,}")
print(f"Common SNVs in Regex:    {len(common_in_regex):,}")
print(f"Rare SNVs in G4Hunter:   {len(rare_in_g4hunter):,}")
print(f"Rare SNVs in Regex:      {len(rare_in_regex):,}")

# MATCHED
# print(f"Common SNVs in G4Hunter (matched): {len(common_in_matched_g4):,}")
# print(f"Common SNVs in Controls (matched):    {len(common_in_matched_ctrl):,}")
# print(f"Rare SNVs in G4Hunter (matched):   {len(rare_in_matched_g4):,}")
# print(f"Rare SNVs in Controls (matched):      {len(rare_in_matched_ctrl):,}")

In [ ]:
rare_in_g4hunter

In [ ]:
rare_in_g4hunter_mutated = rare_in_g4hunter.groupby("canonical").agg(total_mut=("start", "count"))
common_in_regex_mutated = common_in_regex.groupby("canonical").agg(total_mut=("start", "count"))
common_in_g4hunter_mutated = common_in_g4hunter.groupby("canonical").agg(total_mut=("start", "count"))
rare_in_regex_mutated = rare_in_regex.groupby("canonical").agg(total_mut=("start", "count"))

# MATCHED
# rare_in_g4_matched_mutated = rare_in_matched_g4.groupby("canonical").agg(total_mut=("start", "count"))
# common_in_g4_matched_mutated = common_in_matched_g4.groupby("canonical").agg(total_mut=("start", "count"))
# common_in_ctrl_matched_mutated = common_in_matched_ctrl.groupby("canonical").agg(total_mut=("start", "count"))
# rare_in_ctrl_matched_mutated = rare_in_matched_ctrl.groupby("canonical").agg(total_mut=("start", "count"))

In [ ]:
gw_rare_mutated = snp_rare_df.groupby("canonical", as_index=False).agg(total_counts=("start", "count"))
gw_common_mutated = snp_common_df.groupby("canonical", as_index=False).agg(total_counts=("start", "count"))

In [ ]:
gw_3mers = pd.read_csv("/work/10904/nikolchanchan/vista/g4_t2t_identification/scripts/tri_model_data/chm13v2.fasta_3mers.txt",
                            delimiter=" ", 
                            names=["context", "total_count_gw"], 
                            header=None)
gw_rare_mutated = snp_rare_df.groupby("canonical", as_index=False).agg(total_mut_gw=("start", "count"))
gw_common_mutated = snp_common_df.groupby("canonical", as_index=False).agg(total_mut_gw=("start", "count"))
gw_rare_mutated = gw_rare_mutated.merge(gw_3mers, left_on=["canonical"], right_on=["context"])
gw_common_mutated = gw_common_mutated.merge(gw_3mers, left_on=["canonical"], right_on=["context"])
gw_rare_mutated.loc[:, "freq"] = gw_rare_mutated["total_mut_gw"] / gw_rare_mutated["total_count_gw"]
gw_common_mutated.loc[:, "freq"] = gw_common_mutated["total_mut_gw"] / gw_common_mutated["total_count_gw"]
gw_rare_mutated

In [ ]:
gw_rare_mutated_dict = gw_rare_mutated[["canonical", "freq"]].set_index("canonical")["freq"].to_dict()
gw_common_mutated_dict = gw_common_mutated[["canonical", "freq"]].set_index("canonical")["freq"].to_dict()

In [ ]:
g4_bed = BedTool.from_dataframe(g4_df).sort()
regex_bed = BedTool.from_dataframe(regex_df).sort()

In [ ]:
nuc = {'A': 'T',
       'T': 'A', 
       'G': 'C', 
       'C': 'G'}

def map_canonical(x):
    rev = ''.join(nuc[c] for c in x)[::-1]
    return min(rev, x)
    
g4_df_merged = pd.read_table(g4_bed.merge().sort().fn, 
                        header=None, 
                        names=["seqID", "start", "end"]).query("seqID != 'chrY'")
quadparser_df_merged = pd.read_table(regex_bed.merge().sort().fn, 
                                     header=None, 
                                     names=["seqID", "start", "end"]
                                     ).query("seqID != 'chrY'")

g4_df_merged["size"] = g4_df_merged["seqID"].map(seq_sizes)
g4_df_merged.loc[:, "expanded_start"] = np.maximum(0, g4_df_merged["start"] - 1)
g4_df_merged.loc[:, "expanded_end"] = np.minimum(g4_df_merged["end"] + 1, g4_df_merged["size"])
g4_df_merged.loc[:, "sequence"] = g4_df_merged.apply(lambda row: chrom_sequences[row["seqID"]][row["expanded_start"]: row["expanded_end"]], axis=1)

quadparser_df_merged["size"] = quadparser_df_merged["seqID"].map(seq_sizes)
quadparser_df_merged.loc[:, "expanded_start"] = np.maximum(0, quadparser_df_merged["start"] - 1)
quadparser_df_merged.loc[:, "expanded_end"] = np.minimum(quadparser_df_merged["end"] + 1, quadparser_df_merged["size"])
quadparser_df_merged.loc[:, "sequence"] = quadparser_df_merged.apply(lambda row: chrom_sequences[row["seqID"]][row["expanded_start"]: row["expanded_end"]], axis=1)

# G4
from collections import defaultdict
g4hunter_contexts = defaultdict(int)
regex_contexts = defaultdict(int)
for _, row in tqdm(g4_df_merged.iterrows(), total=g4_df_merged.shape[0]):
    sequence = row["sequence"]
    for i in range(len(sequence)-2):
        tri = map_canonical(sequence[i:i+3])
        g4hunter_contexts[tri] += 1

# Quadparser
for _, row in tqdm(quadparser_df_merged.iterrows(), total=quadparser_df_merged.shape[0]):
    sequence = row["sequence"]
    for i in range(len(sequence)-2):
        tri = map_canonical(sequence[i:i+3])
        regex_contexts[tri] += 1

g4hunter_contexts = pd.Series(g4hunter_contexts).to_frame(name="total_count_g4").reset_index().rename(columns={"index": "canonical"})
regex_contexts = pd.Series(regex_contexts).to_frame(name="total_count_g4").reset_index().rename(columns={"index": "canonical"})
g4hunter_contexts

In [ ]:
# Matched G4 regions
matched_all_df = matched_all_df[matched_all_df["seqID"] != 'chrY'].reset_index(drop=True)
matched_g4_df = matched_all_df[["seqID", "start", "end"]].copy()
matched_g4_df["size"] = matched_g4_df["seqID"].map(seq_sizes)
matched_g4_df.loc[:, "expanded_start"] = np.maximum(0, matched_g4_df["start"] - 1)
matched_g4_df.loc[:, "expanded_end"] = np.minimum(matched_g4_df["end"] + 1, matched_g4_df["size"])
matched_g4_df.loc[:, "sequence"] = matched_g4_df.apply(
    lambda row: chrom_sequences[row["seqID"]][row["expanded_start"]: row["expanded_end"]], axis=1)

# Matched control regions
matched_ctrl_df = matched_all_df[["control_seqID", "control_start", "control_end"]].copy()
matched_ctrl_df = matched_ctrl_df.rename(columns={"control_seqID": "seqID", "control_start": "start", "control_end": "end"})
matched_ctrl_df["size"] = matched_ctrl_df["seqID"].map(seq_sizes)
matched_ctrl_df.loc[:, "expanded_start"] = np.maximum(0, matched_ctrl_df["start"] - 1)
matched_ctrl_df.loc[:, "expanded_end"] = np.minimum(matched_ctrl_df["end"] + 1, matched_ctrl_df["size"])
matched_ctrl_df.loc[:, "sequence"] = matched_ctrl_df.apply(
    lambda row: chrom_sequences[row["seqID"]][row["expanded_start"]: row["expanded_end"]], axis=1)

# Count trinucleotide contexts
matched_g4_contexts = defaultdict(int)
matched_ctrl_contexts = defaultdict(int)

for _, row in tqdm(matched_g4_df.iterrows(), total=matched_g4_df.shape[0]):
    sequence = row["sequence"]
    for i in range(len(sequence) - 2):
        matched_g4_contexts[map_canonical(sequence[i:i+3])] += 1

for _, row in tqdm(matched_ctrl_df.iterrows(), total=matched_ctrl_df.shape[0]):
    sequence = row["sequence"]
    for i in range(len(sequence) - 2):
        matched_ctrl_contexts[map_canonical(sequence[i:i+3])] += 1

matched_g4_contexts = pd.Series(matched_g4_contexts).to_frame(name="total_count_g4").reset_index().rename(columns={"index": "canonical"})
matched_ctrl_contexts = pd.Series(matched_ctrl_contexts).to_frame(name="total_count_g4").reset_index().rename(columns={"index": "canonical"})

In [ ]:
rare_in_g4hunter_mutated = rare_in_g4hunter.groupby("canonical", as_index=False).agg(total_mut=("start", "count"))
common_in_regex_mutated = common_in_regex.groupby("canonical", as_index=False).agg(total_mut=("start", "count"))
common_in_g4hunter_mutated = common_in_g4hunter.groupby("canonical", as_index=False).agg(total_mut=("start", "count"))
rare_in_regex_mutated = rare_in_regex.groupby("canonical", as_index=False).agg(total_mut=("start", "count"))

In [ ]:
rare_in_g4hunter_mutated = rare_in_g4hunter.groupby("canonical", as_index=False).agg(total_mut=("start", "count"))
common_in_regex_mutated = common_in_regex.groupby("canonical", as_index=False).agg(total_mut=("start", "count"))
common_in_g4hunter_mutated = common_in_g4hunter.groupby("canonical", as_index=False).agg(total_mut=("start", "count"))
rare_in_regex_mutated = rare_in_regex.groupby("canonical", as_index=False).agg(total_mut=("start", "count"))

# MATCHED
rare_in_matched_g4_mutated = rare_in_matched_g4.groupby("canonical", as_index=False).agg(total_mut=("start", "count"))
common_in_matched_g4_mutated = common_in_matched_g4.groupby("canonical", as_index=False).agg(total_mut=("start", "count"))
rare_in_matched_ctrl_mutated = rare_in_matched_ctrl.groupby("canonical", as_index=False).agg(total_mut=("start", "count"))
common_in_matched_ctrl_mutated = common_in_matched_ctrl.groupby("canonical", as_index=False).agg(total_mut=("start", "count"))

In [ ]:
datasets_matched = [
    ("Rare",   "G4Hunter_matched", rare_in_matched_g4_mutated),
    ("Common", "G4Hunter_matched", common_in_matched_g4_mutated),
    ("Rare",   "Control_matched",  rare_in_matched_ctrl_mutated),
    ("Common", "Control_matched",  common_in_matched_ctrl_mutated),
]

gw_freq = {"Rare": gw_rare_mutated,
           "Common": gw_common_mutated 
           }

contexts_matched = {
    "G4Hunter_matched": matched_g4_contexts,
    "Control_matched":  matched_ctrl_contexts,
}

matched_contexts_df = []
for (rarity, method, df) in datasets_matched:
    freq_map = gw_freq[rarity].set_index("canonical")["freq"]
    ctx_ = df.merge(contexts_matched[method], on="canonical").merge(
                                gw_freq[rarity],
                                on=["canonical"],
                                how="left",
                                suffixes=("", "_gw")
                            )
    ctx_.loc[:, "freq"]       = ctx_["canonical"].map(freq_map)
    ctx_.loc[:, "g4_freq"]    = ctx_["total_mut"] / ctx_["total_count_g4"]
    ctx_.loc[:, "method"]     = method
    ctx_.loc[:, "rarity"]     = rarity
    ctx_.loc[:, "enrichment"] = ctx_["g4_freq"] / ctx_["freq"]
    ctx_.sort_values(by="enrichment", ascending=False, inplace=True)
    ctx_.reset_index(drop=True, inplace=True)
    matched_contexts_df.append(ctx_)

matched_contexts_df = pd.concat(matched_contexts_df, ignore_index=True)
matched_contexts_df

In [ ]:
datasets = [
        ("Rare", "G4Hunter", rare_in_g4hunter_mutated),
        ("Rare", "Quadparser", rare_in_regex_mutated),
        ("Common", "G4Hunter", common_in_g4hunter_mutated),
        ("Common", "Quadparser", common_in_regex_mutated),
]

contexts = {"G4Hunter": g4hunter_contexts,
            "Quadparser": regex_contexts 
            }

gw_freq = {"Rare": gw_rare_mutated,
           "Common": gw_common_mutated 
           }
g4hunter_contexts_df = []
for (rarity, method, df) in datasets:
    g4hunter_contexts_ = df.merge(contexts[method], 
                                  on="canonical")\
                            .merge(
                                gw_freq[rarity],
                                on=["canonical"],
                                how="left",
                                suffixes=("", "_gw")
                            )
    g4hunter_contexts_.loc[:, "g4_freq"] = g4hunter_contexts_["total_mut"] / g4hunter_contexts_["total_count_g4"]
    g4hunter_contexts_.loc[:, "method"] = method
    g4hunter_contexts_.loc[:, "rarity"] = rarity 
    g4hunter_contexts_.loc[:, "enrichment"] = g4hunter_contexts_["g4_freq"] / g4hunter_contexts_["freq"]
    g4hunter_contexts_.sort_values(by=["enrichment"], ascending=False, inplace=True)
    g4hunter_contexts_.reset_index(drop=True, inplace=True)
    g4hunter_contexts_df.append(g4hunter_contexts_)

g4hunter_contexts_df = pd.concat(g4hunter_contexts_df, ignore_index=True)
g4hunter_contexts_df

In [ ]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from pathlib import Path
import os

MIDDLE_PALETTE = {"GC": "#E74C3C", "AT": "#3498DB"}

def middle_group(ctx):
    return "GC" if ctx[1] in ("G", "C") else "AT"

def p_to_star(p):
    if p < 1e-4:  return "****"
    if p < 1e-3:  return "***"
    if p < 1e-2:  return "**"
    if p < 0.05:  return "*"
    return "ns"

g4hunter_contexts_df = []
for (rarity, method, df) in datasets:
    g4_ctx = (
        df.merge(contexts[method], on="canonical")
          .merge(gw_freq[rarity], on="canonical", how="left", suffixes=("", "_gw"))  # fixed: use gw_freq[rarity]
    )
    g4_ctx["g4_freq"]    = g4_ctx["total_mut"]    / g4_ctx["total_count_g4"]
    g4_ctx["method"]     = method
    g4_ctx["rarity"]     = rarity
    g4_ctx["enrichment"] = g4_ctx["g4_freq"] / g4_ctx["freq"]

    pvals = []
    for _, row in g4_ctx.iterrows():
        mut_g4     = int(row["total_mut"])
        not_mut_g4 = int(row["total_count_g4"]    - mut_g4)
        mut_gw     = int(row["total_mut_gw"])
        not_mut_gw = int(row["total_count_gw"] - mut_gw)
        _, p = fisher_exact([[mut_g4, not_mut_g4], [mut_gw, not_mut_gw]], alternative="greater")
        pvals.append(p)

    g4_ctx["pval"]  = pvals
    g4_ctx["qval"]  = multipletests(pvals, method="fdr_bh")[1]
    g4_ctx["stars"] = g4_ctx["qval"].apply(p_to_star)
    g4_ctx.sort_values("enrichment", ascending=False, inplace=True)
    g4_ctx.reset_index(drop=True, inplace=True)
    g4hunter_contexts_df.append(g4_ctx)

g4hunter_contexts_df = pd.concat(g4hunter_contexts_df, ignore_index=True)
g4hunter_contexts_df

In [ ]:
matched_contexts_df = []
for rarity in ["Rare", "Common"]:
    g4_mut_df   = rare_in_matched_g4_mutated   if rarity == "Rare" else common_in_matched_g4_mutated
    ctrl_mut_df = rare_in_matched_ctrl_mutated  if rarity == "Rare" else common_in_matched_ctrl_mutated

    g4   = g4_mut_df.merge(matched_g4_contexts, on="canonical")
    ctrl = ctrl_mut_df.merge(matched_ctrl_contexts, on="canonical") \
                      .rename(columns={"total_mut": "total_mut_ctrl", "total_count_g4": "total_count_ctrl"})

    merged = g4.merge(ctrl[["canonical", "total_mut_ctrl", "total_count_ctrl"]], on="canonical")
    merged["g4_freq"]    = merged["total_mut"]      / merged["total_count_g4"]
    merged["ctrl_freq"]  = merged["total_mut_ctrl"] / merged["total_count_ctrl"]
    merged["enrichment"] = merged["g4_freq"] / merged["ctrl_freq"]
    merged["rarity"]     = rarity
    merged["method"]     = "G4Hunter_matched"

    pvals = []
    for _, row in merged.iterrows():
        mut_g4       = int(row["total_mut"])
        not_mut_g4   = int(row["total_count_g4"]   - mut_g4)
        mut_ctrl     = int(row["total_mut_ctrl"])
        not_mut_ctrl = int(row["total_count_ctrl"]  - mut_ctrl)
        _, p = fisher_exact([[mut_g4, not_mut_g4], [mut_ctrl, not_mut_ctrl]], alternative="greater")
        pvals.append(p)

    merged["pval"]  = pvals
    merged["qval"]  = multipletests(pvals, method="fdr_bh")[1]
    merged["stars"] = merged["qval"].apply(p_to_star)
    merged.sort_values("enrichment", ascending=False, inplace=True)
    merged.reset_index(drop=True, inplace=True)
    matched_contexts_df.append(merged)

matched_contexts_df = pd.concat(matched_contexts_df, ignore_index=True)
matched_contexts_df

In [ ]:
canonical_order_matched = (
    matched_contexts_df[matched_contexts_df["rarity"] == "Common"]
    .groupby("canonical")["enrichment"].mean()
    .sort_values(ascending=False)
    .index.tolist()
)

bar_colors_matched = [MIDDLE_PALETTE[middle_group(c)] for c in canonical_order_matched]

fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharey="row", sharex=True)

for r, rarity in enumerate(["Common", "Rare"]):
    ax  = axes[r]
    sub = (
        matched_contexts_df[matched_contexts_df["rarity"] == rarity]
        .set_index("canonical")
        .reindex(canonical_order_matched)
        .reset_index()
    )

    ax.bar(range(len(sub)), sub["enrichment"],
           color=bar_colors_matched, edgecolor="black", linewidth=0.5, zorder=3)
    ax.axhline(1.0, color="black", linewidth=1.2, linestyle="--", zorder=4)

    for i, row in sub.iterrows():
        if row["stars"] != "ns":
            ax.text(i + 0.18, row["enrichment"] + 0.04, row["stars"],
                    ha="center", va="bottom", fontsize=14, rotation=90, fontweight="bold")

    ax.set_xticks(range(len(canonical_order_matched)))
    ax.set_xticklabels(canonical_order_matched, rotation=90, fontsize=18)
    ax.tick_params(axis="y", labelsize=24)
    ax.grid(lw=0.4, alpha=0.6, axis="y", zorder=0)
    ax.set_axisbelow(True)
    sns.despine(ax=ax)
    ax.set_ylabel(f"{rarity}\nEnrichment", fontsize=24)

axes[0].annotate("", xy=(0.5, 1.03), xycoords="axes fraction",
                 ha="center", fontsize=24, fontweight="bold")
axes[1].set_xlabel("Trinucleotide context", fontsize=24)

handles = [mpatches.Patch(color=col, label=label) for label, col in MIDDLE_PALETTE.items()]
fig.legend(handles=handles, fontsize=24, loc="lower center", ncol=2,
           frameon=True, fancybox=True, bbox_to_anchor=(0.5, -0.1))

plt.tight_layout()
fig.savefig(target / "trinucleotide_enrichment_matched_fisher.pdf",
            transparent=True, bbox_inches="tight")
plt.show()

In [ ]:
canonical_order = (
    g4hunter_contexts_df[g4hunter_contexts_df["rarity"] == "Common"]
    .groupby("canonical")["enrichment"].mean()
    .sort_values(ascending=False)
    .index.tolist()
)

MIDDLE_PALETTE = {"AT": "#3B7CD4", 
                  "GC": "#DE4365"}
def middle_group(ctx):
    return "GC" if ctx[1] in ("G", "C") else "AT"

bar_colors = [MIDDLE_PALETTE[middle_group(c)] for c in canonical_order]

fig, axes = plt.subplots(2, 2, figsize=(18, 10), sharey="row", sharex=True)
RARITY_ORDER = ["Common", "Rare"]
METHOD_ORDER = ["G4Hunter", "Quadparser"]
for r, rarity in enumerate(RARITY_ORDER):
    for c, method in enumerate(METHOD_ORDER):
        ax  = axes[r, c]
        sub = (
            g4hunter_contexts_df[
                (g4hunter_contexts_df["rarity"] == rarity) &
                (g4hunter_contexts_df["method"] == method)
            ]
            .set_index("canonical")
            .reindex(canonical_order)
            .reset_index()
        )

        ax.bar(range(len(sub)), sub["enrichment"],
               color=bar_colors, edgecolor="black", linewidth=0.5, zorder=3)
        ax.axhline(1.0, color="black", linewidth=1.2, linestyle="--", zorder=4)

        for i, row in sub.iterrows():
            if row["stars"] != "ns":
                ax.text(i + 0.18, row["enrichment"] + 0.04, row["stars"],
                        ha="center", va="bottom", fontsize=14, rotation=90, fontweight="bold")

        ax.set_xticks(range(len(canonical_order)))
        ax.set_xticklabels(canonical_order, rotation=90, fontsize=18)
        ax.tick_params(axis="y", labelsize=24)
        ax.grid(lw=0.4, alpha=0.6, axis="y", zorder=0)
        ax.set_axisbelow(True)
        sns.despine(ax=ax)
        if r == 0:
            ax.annotate(method, xy=(0.5, 1.03), xycoords="axes fraction",
                        ha="center", fontsize=28, fontweight="bold",
                        bbox=dict(boxstyle="round,pad=0.45", facecolor="#f0f0f0",
                                  edgecolor="#333333", lw=1.5))

        if c == 0:
            ax.set_ylabel(f"{rarity.replace('Rare', 'Low Frequency')}\nEnrichment", fontsize=24)

        if r == 1:
            ax.set_xlabel("Trinucleotide context", fontsize=24,)

handles = [mpatches.Patch(color=col, label=label) for label, col in MIDDLE_PALETTE.items()]
fig.legend(handles=handles, title="", fontsize=24, title_fontsize=12,
           loc="lower center", ncol=2, frameon=True, fancybox=True,
           bbox_to_anchor=(0.5, -0.1))

plt.tight_layout()
target = Path(os.getenv("SCRATCH")) / "figures_g4_t2t"
target.mkdir(exist_ok=True)
fig.savefig(target / "trinucleotide_enrichment_fisher.pdf", 
            transparent=True, 
            bbox_inches="tight")
fig.savefig(target / "trinucleotide_enrichment_fisher.png",
            dpi=300, 
            transparent=True, 
            bbox_inches="tight")
plt.show()

In [ ]:
rare_matched  = matched_contexts_df[matched_contexts_df["rarity"] == "Rare"].set_index("canonical")
common_matched = matched_contexts_df[matched_contexts_df["rarity"] == "Common"].set_index("canonical")

merged_matched = rare_matched[["total_mut", "total_count_g4", "enrichment"]].join(
    common_matched[["total_mut", "total_count_g4", "enrichment"]],
    lsuffix="_rare", rsuffix="_common"
).dropna()

merged_matched["enrichment_ratio"] = merged_matched["enrichment_rare"] / merged_matched["enrichment_common"]

pvals = []
for _, row in merged_matched.iterrows():
    table = [
        [int(row["total_mut_rare"]),   int(row["total_count_g4_rare"]   - row["total_mut_rare"])],
        [int(row["total_mut_common"]), int(row["total_count_g4_common"] - row["total_mut_common"])],
    ]
    _, p = fisher_exact(table, alternative="two-sided")
    pvals.append(p)

merged_matched["pval"]  = pvals
merged_matched["qval"]  = multipletests(pvals, method="fdr_bh")[1]
merged_matched["stars"] = merged_matched["qval"].apply(p_to_star)
merged_matched = merged_matched.reset_index().sort_values("enrichment_ratio", ascending=False)

fig, ax = plt.subplots(1, 1, figsize=(10, 4))

sub    = merged_matched.sort_values("enrichment_ratio", ascending=False)
colors = [MIDDLE_PALETTE[middle_group(c)] for c in sub["canonical"]]

ax.bar(range(len(sub)), sub["enrichment_ratio"], color=colors, edgecolor="black", linewidth=1.0, zorder=3)
ax.axhline(1.0, color="black", linewidth=1.2, linestyle="--", zorder=4)

for i, (_, row) in enumerate(sub.iterrows()):
    if row["stars"] != "ns":
        ax.text(i + 0.2, row["enrichment_ratio"] + 0.02, row["stars"],
                ha="center", va="bottom", fontsize=8, fontweight="bold", rotation=90)

ax.set_xticks(range(len(sub)))
ax.set_xticklabels(sub["canonical"], rotation=90, fontsize=11)
ax.set_ylabel("Enrichment Ratio", fontsize=16)
ax.set_xlabel("Canonical trinucleotide context", fontsize=14)
ax.tick_params(labelsize=14)
ax.set_ylim(ymax=1.2)
ax.grid(lw=0.4, alpha=0.6, axis="y", zorder=0)
ax.set_axisbelow(True)
ax.annotate("", xy=(0.5, 1.03), xycoords="axes fraction",
            ha="center", fontsize=15, fontweight="bold")
sns.despine(ax=ax)

handles = [mpatches.Patch(color=col, label=label) for label, col in MIDDLE_PALETTE.items()]
fig.legend(handles=handles, fontsize=14, loc="lower center", ncol=2,
           frameon=True, fancybox=True, bbox_to_anchor=(0.5, -0.25))

fig.savefig(target / "rare_vs_common_enrichment_ratio_matched.pdf", transparent=True, bbox_inches="tight")
plt.show()

In [ ]:
rare_common = []
for method in METHOD_ORDER:
    rare   = g4hunter_contexts_df[(g4hunter_contexts_df["rarity"] == "Rare")   & (g4hunter_contexts_df["method"] == method)].set_index("canonical")
    common = g4hunter_contexts_df[(g4hunter_contexts_df["rarity"] == "Common") & (g4hunter_contexts_df["method"] == method)].set_index("canonical")

    merged = rare[["total_mut", "total_count_g4", "enrichment"]].join(
        common[["total_mut", "total_count_g4", "enrichment"]], lsuffix="_rare", rsuffix="_common"
    ).dropna()

    merged["enrichment_ratio"] = merged["enrichment_rare"] / merged["enrichment_common"]

    pvals = []
    for _, row in merged.iterrows():
        table = [
            [int(row["total_mut_rare"]),   int(row["total_count_g4_rare"]   - row["total_mut_rare"])],
            [int(row["total_mut_common"]), int(row["total_count_g4_common"] - row["total_mut_common"])],
        ]
        _, p = fisher_exact(table, alternative="two-sided")
        pvals.append(p)

    merged["pval"]   = pvals
    merged["qval"]   = multipletests(pvals, method="fdr_bh")[1]
    merged["stars"]  = merged["qval"].apply(p_to_star)
    merged["method"] = method
    rare_common.append(merged.reset_index())

rare_common_df = pd.concat(rare_common).sort_values("enrichment_ratio", ascending=False)
rare_common_df

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharey=False)
fig.subplots_adjust(hspace=0.5)

for j, (ax, method) in enumerate(zip(axes, METHOD_ORDER)):
    sub = rare_common_df[rare_common_df["method"] == method].sort_values("enrichment_ratio", ascending=False)
    colors = [MIDDLE_PALETTE[middle_group(c)] for c in sub["canonical"]]

    ax.bar(range(len(sub)), sub["enrichment_ratio"], color=colors, edgecolor="black", linewidth=1.0, zorder=3)
    ax.axhline(1.0, color="black", linewidth=1.2, linestyle="--", zorder=4)

    for i, (_, row) in enumerate(sub.iterrows()):
        if row["stars"] != "ns":
            ax.text(i + 0.2, row["enrichment_ratio"] + 0.02, row["stars"],
                    ha="center", va="bottom", fontsize=8, fontweight="bold", rotation=90)

    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels(sub["canonical"], rotation=90, fontsize=11)
    ax.set_ylabel("Enrichment Ratio", fontsize=16)
    if j != 0:
        ax.set_xlabel("Canonical trinucleotide context", fontsize=14)
    else:
        ax.set_xlabel("")
    ax.tick_params(labelsize=14)
    ax.grid(lw=0.4, alpha=0.6, axis="y", zorder=0)
    ax.set_axisbelow(True)
    ax.set_ylim(ymax=1.4)
    ax.annotate(method, xy=(0.5, 1.03), xycoords="axes fraction", ha="center", fontsize=15, fontweight="bold")
    sns.despine(ax=ax)

handles = [mpatches.Patch(color=col, label=label) for label, col in MIDDLE_PALETTE.items()]
fig.legend(handles=handles, title="", fontsize=14, title_fontsize=12,
           loc="lower center", ncol=2, frameon=True, fancybox=True, bbox_to_anchor=(0.5, -0.05))
# plt.tight_layout()
fig.savefig(target / "rare_vs_common_enrichment_ratio.pdf", transparent=True, bbox_inches="tight")
plt.show()


In [ ]:
matched_gc_at = matched_contexts_df.copy()
matched_gc_at["middle_group"] = matched_gc_at["canonical"].apply(middle_group)

agg_g4 = matched_gc_at.groupby(["rarity", "middle_group"]).agg(
    total_mut=("total_mut", "sum"),
    total_count_g4=("total_count_g4", "sum"),
).reset_index()
agg_g4["freq"] = agg_g4["total_mut"] / agg_g4["total_count_g4"]

agg_ctrl = matched_gc_at.groupby(["rarity", "middle_group"]).agg(
    total_mut_ctrl=("total_mut_ctrl", "sum"),
    total_count_ctrl=("total_count_ctrl", "sum"),
).reset_index()
agg_ctrl["freq"] = agg_ctrl["total_mut_ctrl"] / agg_ctrl["total_count_ctrl"]

# Rare / Common ratio per middle_group for G4 and Control
ratio_rows = []
stars_map_matched = {}
for grp in ["GC", "AT"]:
    rare_g4    = agg_g4[(agg_g4["rarity"] == "Rare")   & (agg_g4["middle_group"] == grp)].iloc[0]
    common_g4  = agg_g4[(agg_g4["rarity"] == "Common") & (agg_g4["middle_group"] == grp)].iloc[0]
    rare_ctrl  = agg_ctrl[(agg_ctrl["rarity"] == "Rare")   & (agg_ctrl["middle_group"] == grp)].iloc[0]
    common_ctrl= agg_ctrl[(agg_ctrl["rarity"] == "Common") & (agg_ctrl["middle_group"] == grp)].iloc[0]

    ratio_rows.append({"middle_group": grp, "region": "G4",      "ratio": rare_g4["freq"]   / common_g4["freq"]})
    ratio_rows.append({"middle_group": grp, "region": "Control", "ratio": rare_ctrl["freq"] / common_ctrl["freq"]})

    # Fisher: does rare/common composition differ between G4 and Control?
    table = [
        [int(rare_g4["total_mut"]),    int(rare_ctrl["total_mut_ctrl"])],
        [int(common_g4["total_mut"]),  int(common_ctrl["total_mut_ctrl"])],
    ]
    _, p = fisher_exact(table, alternative="two-sided")
    stars_map_matched[grp] = p_to_star(p)

ratio_df = pd.DataFrame(ratio_rows)

REGION_PALETTE = {"G4": "#2ECC71", "Control": "#95A5A6"}
GROUP_ORDER    = ["GC", "AT"]

fig, ax = plt.subplots(figsize=(6, 5))
sns.barplot(data=ratio_df, x="middle_group", y="ratio", hue="region",
            hue_order=["G4", "Control"], order=GROUP_ORDER,
            palette=REGION_PALETTE, edgecolor="black", linewidth=0.8, ax=ax)
ax.axhline(1.0, color="black", linewidth=1.2, linestyle="--")

for x_pos, grp in enumerate(GROUP_ORDER):
    star = stars_map_matched[grp]
    if star != "ns":
        y_max = ratio_df[ratio_df["middle_group"] == grp]["ratio"].max()
        x1, x2 = x_pos - 0.2, x_pos + 0.2
        ax.plot([x1, x1, x2, x2],
                [y_max + 0.02, y_max + 0.03, y_max + 0.03, y_max + 0.02],
                color="black", linewidth=1.2)
        ax.text(x_pos, y_max + 0.03, star, ha="center", va="bottom",
                fontsize=13, fontweight="bold")

ax.set_xlabel("")
ax.set_ylabel("Rare / Common mutation rate ratio", fontsize=16)
ax.tick_params(labelsize=16)
ax.grid(lw=0.4, alpha=0.6, axis="y", zorder=0)
ax.set_axisbelow(True)
ax.annotate("G4Hunter matched", xy=(0.5, 1.03), xycoords="axes fraction",
            ha="center", fontsize=15, fontweight="bold")
sns.despine(ax=ax)

handles = [mpatches.Patch(color=c, label=l) for l, c in REGION_PALETTE.items()]
ax.legend(handles=handles, fontsize=14, loc="upper right", frameon=True, fancybox=True)

plt.tight_layout()
fig.savefig(target / "gc_at_rare_common_ratio_matched.pdf", transparent=True, bbox_inches="tight")
plt.show()

In [ ]:
from scipy.stats import fisher_exact

gc_at = []
for method in METHOD_ORDER:
    for rarity in RARITY_ORDER:
        sub = g4hunter_contexts_df[
            (g4hunter_contexts_df["method"] == method) &
            (g4hunter_contexts_df["rarity"] == rarity)
        ].copy()
        sub["middle_group"] = sub["canonical"].apply(middle_group)
        agg = sub.groupby("middle_group").agg(
            total_mut=("total_mut", "sum"),
            total_count_g4=("total_count_g4", "sum"),
            total_mut_gw=("total_mut_gw", "sum"),
            total_count_gw=("total_count_gw", "sum"),
        ).reset_index()
        agg["g4_freq"]    = agg["total_mut"] / agg["total_count_g4"]
        agg["gw_freq"]    = agg["total_mut_gw"] / agg["total_count_gw"]
        agg["enrichment"] = agg["g4_freq"] / agg["gw_freq"]
        agg["method"]     = method
        agg["rarity"]     = rarity
        gc_at.append(agg)

gc_at_df = pd.concat(gc_at, ignore_index=True)

# Fisher: Rare vs Common per (method, middle_group)
stars_map = {}

for method in METHOD_ORDER:
    for grp in ["GC", "AT"]:
        rare   = gc_at_df[(gc_at_df["method"] == method) & (gc_at_df["rarity"] == "Rare")   & (gc_at_df["middle_group"] == grp)].iloc[0]
        common = gc_at_df[(gc_at_df["method"] == method) & (gc_at_df["rarity"] == "Common") & (gc_at_df["middle_group"] == grp)].iloc[0]
        table = [
            [int(rare["total_mut"]),   int(rare["total_count_g4"]   - rare["total_mut"])],
            [int(common["total_mut"]), int(common["total_count_g4"] - common["total_mut"])],
        ]
        _, p = fisher_exact(table, alternative="two-sided")
        stars_map[(method, grp)] = p_to_star(p)

RARITY_PALETTE = {"Common": "#E9F02B",
                  "Rare":   "#E827E4"}
GROUP_ORDER    = ["GC", "AT"]

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
for ax, method in zip(axes, METHOD_ORDER):
    sub = gc_at_df[gc_at_df["method"] == method]
    sns.barplot(data=sub, x="middle_group", y="enrichment", hue="rarity",
                hue_order=RARITY_ORDER, order=GROUP_ORDER,
                palette=RARITY_PALETTE, edgecolor="black", linewidth=0.8, ax=ax)
    ax.axhline(1.0, color="black", linewidth=1.2, linestyle="--")

    for grp, x_pos in zip(GROUP_ORDER, range(len(GROUP_ORDER))):
        star = stars_map[(method, grp)]
        if star != "ns":
            y_max = sub[sub["middle_group"] == grp]["enrichment"].max()
            x1, x2 = x_pos - 0.2, x_pos + 0.2
            ax.plot([x1, x1, x2, x2],
                    [y_max + 0.08, y_max + 0.1, y_max + 0.1, y_max + 0.08],
                    color="black", linewidth=1.2)
            ax.text(x_pos, y_max + 0.1, star, ha="center", va="bottom",
                    fontsize=13, fontweight="bold")

    ax.set_xlabel("", fontsize=14)
    ax.set_ylabel("Enrichment" if method == METHOD_ORDER[0] else "", fontsize=16)
    ax.tick_params(labelsize=16)
    ax.grid(lw=0.4, alpha=0.6, axis="y", zorder=0)
    ax.set_axisbelow(True)
    ax.annotate(method, xy=(0.5, 1.03), xycoords="axes fraction",
                ha="center", fontsize=15, fontweight="bold")
    sns.despine(ax=ax)

    handles = [mpatches.Patch(color=c, label=l) for l, c in RARITY_PALETTE.items()]
    ax.legend(handles=handles, fontsize=14, loc="lower center", ncol=2,
              frameon=True, fancybox=True, bbox_to_anchor=(0.5, -0.24), title="")

plt.tight_layout()
fig.savefig(target / "gc_at_enrichment_rare_common.pdf", transparent=True, bbox_inches="tight")
plt.show()

## Loops Vs G-Runs

In [ ]:
from pybedtools import BedTool
import pandas as pd

# You need G4 loop coordinates
# Loop = spacer region between the two arms
# Do you have loop coordinates or just full G4 coordinates?

# If you have full G4 coords + arm length:
# loop_start = g4_start + arm_length
# loop_end = g4_end - arm_length

# Create beds
rare_bed = BedTool.from_dataframe(
    rare_df_filtered[rare_df_filtered["mutation"] == "snp"][["#CHROM", "start", "end"]].drop_duplicates()
).sort()

common_bed = BedTool.from_dataframe(
    common_df_filtered[common_df_filtered["mutation"] == "snp"][["#CHROM", "start", "end"]].drop_duplicates()
).sort()

# Intersect with G4 loops
# What does your G4 dataframe look like?
# Do you have loop/spacer coordinates?

In [ ]:
# import re

# def get_kernels(run_seq):
#     """kernel(n) = run_seq[n : L-n], empty when n >= ceil(L/2)."""
#     L = len(run_seq)
#     kernels = {}
#     for n in range(1, L):
#         k = run_seq[n : L - n]
#         if not k:
#             break
#         kernels[n] = k
#     return kernels

# # ── Validation ────────────────────────────────────────────────────────────
# assert get_kernels("GGGGG") == {1: "GGG", 2: "G"},      "L=5 failed"
# assert get_kernels("GGGG")  == {1: "GG"},                "L=4 failed"
# assert get_kernels("GGG")   == {1: "G"},                 "L=3 failed"
# assert get_kernels("GG")    == {},                        "L=2 failed (should be empty)"
# print("All kernel assertions passed")

# # ── Build kernel dataframe ────────────────────────────────────────────────
# g_kernel_df = []

# if isinstance(regex_df, pd.DataFrame):
#     regex_df_cp = pl.from_pandas(regex_df)
# else:
#     regex_df_cp = regex_df.clone()

# for row in regex_df_cp.iter_rows(named=True):
#     seqID    = row["seqID"]
#     start    = row["start"]
#     end      = row["end"]
#     sequence = row["sequence"].upper()
#     grun_char = sequence[0]

#     gruns = list(re.finditer(r"%s{3,}" % re.escape(grun_char), sequence))
#     total_gruns = len(gruns)
#     if total_gruns <= 2:
#         continue
#     for grun in gruns:
#         g_start, g_end = grun.span()
#         run_seq = sequence[g_start:g_end]
#         L       = len(run_seq)

#         for n, kernel_seq in get_kernels(run_seq).items():
#             g_kernel_df.append({
#                 "seqID":       seqID,
#                 "start":       start + g_start + n,
#                 "end":         start + g_end   - n,
#                 "sequence": sequence,
#                 "motif_start": start,
#                 "motif_end":   end,
#                 "grun_start":  start + g_start,
#                 "grun_end":    start + g_end,
#                 "grun_length": L,
#                 "kernel_n":    n,
#                 "kernel_seq":  kernel_seq,
#                 "kernel_length": len(kernel_seq),
#             })

# g_kernel_df = pd.DataFrame(g_kernel_df)

# # ── Sanity check on output ────────────────────────────────────────────────
# assert (g_kernel_df["kernel_length"] == g_kernel_df["end"] - g_kernel_df["start"]).all(), \
#     "coordinate/length mismatch"
# assert (g_kernel_df["kernel_length"] > 0).all(), \
#     "empty kernels should not appear"
# assert (g_kernel_df["start"] >= g_kernel_df["grun_start"]).all()
# assert (g_kernel_df["end"]   <= g_kernel_df["grun_end"]).all()
# print(f"Kernel df: {len(g_kernel_df):,} rows, kernel_n range: {g_kernel_df['kernel_n'].min()}–{g_kernel_df['kernel_n'].max()}")
# g_kernel_df.head(10)

In [ ]:
# import pyranges as pr
# from scipy.stats import fisher_exact

# MAX_N = 10
# RARITY_PALETTE = {"Common": "#3B7CD4", "Rare": "#DE4365"}

# def build_kernel_pr(kernel_df, n):
#     sub = kernel_df[kernel_df["kernel_n"] == n]
#     return pr.PyRanges(
#         sub[["seqID", "start", "end"]]
#           .rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"})
#     ), int((sub["end"] - sub["start"]).sum())

# def snp_to_pr(df, chrom_col="#CHROM"):
#     return pr.PyRanges(
#         df[[chrom_col, "start", "end"]].drop_duplicates()
#           .rename(columns={chrom_col: "Chromosome", "start": "Start", "end": "End"})
#     )

# # build G4Hunter kernels the same way (reuse get_kernels from previous cell)
# def build_kernels(motif_df):
#     records = []
#     src = motif_df.to_pandas() if isinstance(motif_df, pl.DataFrame) else motif_df
#     for _, row in src.iterrows():
#         seqID, start, end = row["seqID"], row["start"], row["end"]
#         sequence = row["sequence"].upper()
#         grun_char = sequence[0]
#         for grun in re.finditer(r"%s{3,}" % re.escape(grun_char), sequence):
#             g_start, g_end = grun.span()
#             run_seq = sequence[g_start:g_end]
#             for n, kseq in get_kernels(run_seq).items():
#                 records.append({"seqID": seqID,
#                                 "start": start + g_start + n,
#                                 "end":   start + g_end   - n,
#                                 "kernel_n": n})
#     return pd.DataFrame(records)

# kernel_dfs = {
#     "Quadparser": g_kernel_df,
#     "G4Hunter":   build_kernels(g4_df_merged),   # g4_df_merged has sequence col
# }

# rare_pr   = snp_to_pr(snp_rare_df)
# common_pr = snp_to_pr(snp_common_df)

# rows = []
# for method, kdf in kernel_dfs.items():
#     for n in range(1, MAX_N + 1):
#         kern_pr, total_bp = build_kernel_pr(kdf, n)
#         if total_bp == 0:
#             continue
#         for rarity, snp_pr in [("Rare", rare_pr), ("Common", common_pr)]:
#             n_mut = kern_pr.join(snp_pr, suffix="_snp").df.shape[0]
#             rows.append(dict(method=method, kernel_n=n, rarity=rarity,
#                              n_mut=n_mut, total_bp=total_bp,
#                              mut_rate=n_mut / total_bp))

# kernel_mut_df = pd.DataFrame(rows)

# # Fisher: Rare vs Common at each (method, kernel_n)
# fisher_rows = []
# for method in ["G4Hunter", "Quadparser"]:
#     for n in range(1, MAX_N + 1):
#         r = kernel_mut_df[(kernel_mut_df["method"] == method) & (kernel_mut_df["kernel_n"] == n)]
#         if len(r) < 2:
#             continue
#         rare   = r[r["rarity"] == "Rare"].iloc[0]
#         common = r[r["rarity"] == "Common"].iloc[0]
#         table = [
#             [int(rare["n_mut"]),   int(rare["total_bp"]   - rare["n_mut"])],
#             [int(common["n_mut"]), int(common["total_bp"] - common["n_mut"])],
#         ]
#         _, p = fisher_exact(table, alternative="two-sided")
#         fisher_rows.append(dict(method=method, kernel_n=n, pval=p, stars=p_to_star(p)))

# fisher_kernel_df = pd.DataFrame(fisher_rows)

In [ ]:
# # ── Plot ───────────────────────────────────────────────────────────────────
# fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

# for ax, method in zip(axes, ["G4Hunter", "Quadparser"]):
#     sub    = kernel_mut_df[kernel_mut_df["method"] == method]
#     fstars = fisher_kernel_df[fisher_kernel_df["method"] == method].set_index("kernel_n")

#     for rarity, color in RARITY_PALETTE.items():
#         r = sub[sub["rarity"] == rarity].sort_values("kernel_n")
#         ax.plot(r["kernel_n"], r["mut_rate"], marker="o", color=color,
#                 label=rarity, linewidth=2, markersize=7)
#         ax.axhline(gw_density[rarity], color=color, lw=1.2, ls="--", alpha=0.6,
#                    label=f"")

#     # stars above the higher line
#     for n, row in fstars.iterrows():
#         if row["stars"] == "ns":
#             continue
#         y_max = sub[sub["kernel_n"] == n]["mut_rate"].max()
#         ax.text(n, y_max * 1.02, row["stars"], ha="center", va="bottom",
#                 fontsize=13, fontweight="bold")

#     ax.set_xlabel("Kernel depth (n)", fontsize=18)
#     ax.set_ylabel("Mutation rate (mut / bp)" if method == "G4Hunter" else "", fontsize=22)
#     ax.tick_params(labelsize=18)
#     ax.grid(lw=0.4, alpha=0.6, zorder=0)
#     ax.set_axisbelow(True)
#     ax.set_xticks(range(1, MAX_N + 1))
#     ax.annotate(method, xy=(0.5, 1.05), xycoords="axes fraction",
#                 ha="center", fontsize=15, fontweight="bold")
#     ax.legend(fontsize=14, frameon=True, fancybox=True)
#     # ax.legend(handles=[], frameon=False)
#     sns.despine(ax=ax)

# plt.tight_layout()
# target = Path(os.getenv("SCRATCH")) / "figures_g4_t2t"
# target.mkdir(exist_ok=True)
# fig.savefig(target / "kernel_mutation_rate_by_depth.pdf", 
#             transparent=True, 
#             bbox_inches="tight")
# plt.show()

In [ ]:
import re 
# extract loops and grun coordinates
loops_df = []
gruns_df = []
transitions_df = []

if isinstance(regex_df, pd.DataFrame):
    regex_df = pl.from_pandas(regex_df)

# temp_df = regex_df.filter(pl.col("start") >= 2890900).filter(pl.col("end") <= 2890992 + 100)
for row in regex_df.iter_rows(named=True):
    seqID = row["seqID"]
    start = row["start"]
    end = row["end"]
    sequence = row["sequence"].upper()
    grun_char = sequence[0]

    gruns = list(re.finditer(r"%s{3,}" % re.escape(grun_char), sequence))
    total_gruns = len(gruns)

    if total_gruns == 1:
        continue

    gruns_df.append({
        "seqID": seqID,
        "start": start,
        "end": start + gruns[0].span()[1],
        "type": "grun",
        "motif_start": start,
        "motif_end": end,
        "sequence": sequence[:gruns[0].span()[1]]
    })
    for prev_grun, cur_grun in zip(gruns, gruns[1:]):
        g_start_prev, g_end_prev = prev_grun.span()
        g_start_cur, g_end_cur = cur_grun.span()

        loops_df.append({
            "seqID": seqID,
            "start": start + g_end_prev,
            "end": start + g_start_cur,
            "motif_start": start,
            "motif_end": end,
            "type": "loop",
            "sequence": sequence[g_end_prev:g_start_cur]
        })
        # -------------------------
        # G-RUN + KERNEL
        # -------------------------
        run_seq = sequence[g_start_cur:g_end_cur]
        kernel = run_seq[1:-1] if len(run_seq) >= 3 else ""

        gruns_df.append({
            "seqID": seqID,
            "start": start + g_start_cur,
            "end": start + g_end_cur,
            "motif_start": start,
            "motif_end": end,
            "type": "grun",
            "sequence": run_seq
        })
        # -------------------------
        # TRANSITIONS
        # -------------------------
        transitions_df.append({
            "seqID": seqID,
            "start": start + g_end_prev - 1,
            "end": start + g_end_prev + 1,
            "motif_start": start,
            "motif_end": end,
            "type": "transition",
            "sequence": sequence[g_end_prev-1:g_end_prev+1]
        })

        transitions_df.append({
            "seqID": seqID,
            "start": start + g_start_cur - 1,
            "end": start + g_start_cur + 1,
            "motif_start": start,
            "motif_end": end,
            "type": "transition",
            "sequence": sequence[g_start_cur-1:g_start_cur+1]
        })


gruns_df = pd.DataFrame(gruns_df)
loops_df = pd.DataFrame(loops_df)
transitions_df = pd.DataFrame(transitions_df)

In [ ]:
gw_rare_mutated_dict = gw_rare_mutated[["canonical", "freq"]].set_index("canonical")["freq"].to_dict()
gw_common_mutated_dict = gw_common_mutated[["canonical", "freq"]].set_index("canonical")["freq"].to_dict()

In [ ]:
gw_mutated_dict = {"Common": gw_common_mutated_dict,
                   "Rare": gw_rare_mutated_dict}

def predict_burden(df: pd.DataFrame, rarity: str) -> float:
    gw_mut_freq = gw_mutated_dict[rarity]
    if isinstance(df, pl.DataFrame):
        df = df.to_pandas()
    df = df.copy()
    df["size"] = df["seqID"].map(seq_sizes)
    df["expanded_start"] = np.maximum(0, df['start'] - 1)
    df["expanded_end"] = np.minimum(df["size"], df["end"] + 1)
    df = df[df["seqID"] != 'chrY']
    if "expanded_sequence" not in df.columns:
        df["expanded_sequence"] = df.apply(lambda row: chrom_sequences[row["seqID"]][row["expanded_start"]: row["expanded_end"]], 
                                  axis=1)

    tri_counts = defaultdict(int)
    for seq in df["expanded_sequence"]:
        for i in range(len(seq)-2):
            chunk = map_canonical(seq[i:i+3])
            tri_counts[chunk] += 1
    
    burden = sum(tri_counts.get(chunk, 0) * gw_mut_freq[chunk] for chunk in gw_mut_freq.keys())
    return burden

###